# Часть 1. Таблицы: парсинг и обработка

# Разработка системы визуализации медицинской статистики

**Автор: Юрий Кузнецов**

**Дата: 04.04.2026**


### Цель проекта
– создание системы поддержки принятия обоснованных управленческих решений для повышения эффективности противотуберкулезной помощи населению Иркутской области.

### Задачи проекта
Сбор, консолидация и приведение к единому формату данных из нескольких источников.
Обработка и анализ эпидемиологических показателей, расчёт производных метрик (на 100 тыс. населения, динамика, структура).
Разработка интерактивных дашбордов в Data Lens или другие системы для визуализации результатов.
Подготовка итогового отчёта и передача результатов заказчику.

**Целевая аудитория**:
Главный врач Иркутской областной противотуберкулезной больницы.
Заведующий организационно-методическим отделом.

**Источники данных**

- **Показатели заболеваемости, распространённости, смертности, контингенты, лечение (в разрезе субъектов РФ)**
(Форма №33 «Сведения о больных туберкулезом»
Excel (XLSX)
2016–2024)
  

- **Заболеваемость по нозологическим формам, по полу и возрасту**
(Форма №8 «Сведения о заболеваниях активным туберкулезом»
Excel (XLSX)
2016–2024)
  
- **Сводные данные по РФ, федеральным округам, регионам**
(Статистический сборник «Социально значимые заболевания населения России» (ЦНИИОИЗ)
PDF (таблицы)
2016–2024)
  
- **Показатели по округам, сравнение с Иркутской областью**
(«Основные показатели противотуберкулезной деятельности в Сибирском и Дальневосточном федеральных округах»
Excel (XLSX)
2016–2024)
  
- **Основные показатели по области в разрезе муниципальных образований**
(Сводная таблица по Иркутской области (внутренний источник)
Excel (XLSX)
2016–2024)
  
- **Для расчёта показателей на 100 тыс. населения**
(Данные Иркутскстата (численность населения, половозрастная структура)
Сайт (HTML/Excel)
2016–2024)
  
Примечание: все исходные файлы предоставляются заказчиком. Данные в форматах PDF и Excel требуют извлечения, очистки и унификации (задание со звездочкой – обработать исходные данные, либо работать с предобработанными данными)


### Отчетные формы и таблицы

Для расчета всех показателей, потребуются данные из трёх основных источников — двух годовых статистических форм и одной формы, описывающей деятельность организации. Вот полный перечень:

#### 🆕 Форма №8 «Сведения о заболеваниях активным туберкулезом»
Это первичная форма для учета всех **впервые выявленных** случаев заболевания.
*   **Источник:** Годовой отчет.
*   **Таблицы в форме:**
    *   **Таблица 1000:** Содержит сведения о больных с впервые в жизни установленным диагнозом активного туберкулеза, распределенных по полу и возрасту.
    *   **Таблица 1010** (в некоторых версиях): Дополнительные сведения (например, по бактериовыделению).

#### 📊 Форма №33 «Сведения о больных туберкулезом»
Это основная форма для мониторинга всего эпидемического процесса. Она содержит гораздо более детальную информацию, чем форма №8.
*   **Источник:** Годовой отчет.
*   **Таблицы в форме:**
    *   **Таблица 2100:** Данные о **контингентах** (всех состоящих на учете) и **впервые выявленных** больных с распределением по локализации, формам и возрасту.
    *   **Таблица 2200:** Информация о **методах выявления** заболевания (при профилактических осмотрах, по обращаемости) и о случаях с **посмертной диагностикой**.
    *   **Таблица 2300:** Сведения о **рецидивах** и **летальных исходах**.
    *   **Таблица 2310:** Детальные данные об умерших от туберкулеза, включая сочетание с ВИЧ-инфекцией и сроки нахождения на учете.
    *   **Таблица 2400:** Данные о проведении **химиопрофилактики** контактным лицам.
    *   **Таблица 2500:** Ключевая таблица для анализа **лекарственной устойчивости (МЛУ)**, включая данные о бактериовыделителях и результатах тестов на чувствительность к препаратам.
    *   **Таблица 2600:** Сведения о **госпитализации** больных
    *   **2700**
    *   **2800**

#### 🔍 Форма №30 «Сведения о медицинской организации»
Эта форма описывает деятельность самой медицинской организации, включая профилактическую работу.
*   **Источник:** Годовой отчет.
*   **Таблицы в форме:**
    *   **Таблица 2513:** Содержит данные об **охвате населения профилактическими осмотрами** на туберкулез и **результативности этих осмотров**.

---

### 💎 Резюме
Итак, для полного расчета понадобятся:
*   **Две формы по туберкулезу:** №8 (для впервые выявленных) и №33 (для всех больных).
*   **Одна форма по организации:** №30 (для профилактической работы).

Внутри формы №33 используется наибольшее количество таблиц (2100, 2200, 2300, 2310, 2400, 2500, 2600, 2700, 2800), так как она является центральной для эпидемиологического анализа.

## Обработка и подготовка данных

Импорт всех необходимых библиотек

In [1]:
import pandas as pd
import numpy as np
from docx import Document
from docx.text.paragraph import Paragraph
from docx.table import Table
import re
import base64
from IPython.display import HTML
import os
pd.set_option('display.max_colwidth', None)  


### Описание алгоритма извлечения и обработки Exel файлов:

1. **Циклическая загрузка**: в цикле по годам (2016–2024) каждый Excel-файл читался функцией, которая превращала все его листы в словарь DataFrames. Результат сохранялся в переменную вида `exdfs_dict_год`. Так обеспечивалась автоматическая обработка всех файлов без ручного повторения кода.

2. **Функции для каждой формы**: для каждой статистической формы (2800, 2700, 2100, 2200, 2300, 2400, 2500, 2600, 1000) была написана отдельная функция очистки и приведения к длинному формату. Эти функции:
   - удаляли лишние строки/столбцы,
   - объединяли дублирующиеся первые столбцы,
   - заполняли пропуски в заголовках (ffill),
   - склеивали 2–3 верхние строки в один заголовок,
   - выполняли melt для перехода из широкого формата в длинный,
   - добавляли колонку с годом,
   - чистили текст от лишних пробелов.

3. **Цикл по годам и формам**: вложенные циклы (год → форма) вызывали соответствующую функцию обработки и сохраняли результат в переменные `df_{год}_{форма}_long`. Это позволило за несколько строк кода обработать 9 форм × 9 лет = 81 таблицу.

4. **Объединение по формам**: все годовые таблицы одной формы собирались в список и конкатенировались в единый DataFrame `df_all_{форма}`. Так получались полные датасеты за весь период.

5. **Финальная нормализация**: для каждого объединённого датасета:
   - имена столбцов приводились к snake_case,
   - удалялись начальные числовые префиксы (например, `5_`),
   - числовой префикс выносился в отдельную колонку «Графа»,
   - нулевые значения заменялись на NaN,
   - приводились типы (номер строки → int).

**Итог**: Получены чистые длинные таблицы (`df_all_2800`, `df_all_2700`, …) с унифицированной структурой, готовые для анализа, визуализации и построения динамических рядов по всем годам. Благодаря использованию функций и циклов весь процесс оказался масштабируемым, воспроизводимым и не потребовал ручного вмешательства для каждого файла.

### Создание функции excel_to_dataframes_named 
извлекает из Excel‑файла все таблицы, каждая из которых имеет название‑маркер в первом столбце, например (1000), (2200).
Она читает все листы, находит строки с шаблоном (число) и собирает следующие за ними строки (до следующего маркера или конца листа) в отдельный DataFrame.
Из каждой таблицы удаляются строки и столбцы, полностью состоящие из NaN.
Результат — словарь, где ключ – это строка‑маркер (например, (1000)), а значение – соответствующий DataFrame.

In [2]:
def excel_to_dataframes_named(file_path):
    """
    Извлекает все таблицы из Excel, маркеры которых имеют вид (число) в любой ячейке.
    Возвращает словарь {маркер: DataFrame}.
    """
    ext = os.path.splitext(file_path)[1].lower()
    engine = 'openpyxl' if ext == '.xlsx' else 'xlrd'
    sheets = pd.read_excel(file_path, sheet_name=None, header=None, engine=engine)
    
    result = {}
    for df in sheets.values():
        rows, cols = df.shape
        used = set()
        for r in range(rows):
            for c in range(cols):
                if (r, c) in used:
                    continue
                val = df.iloc[r, c]
                if pd.isna(val):
                    continue
                cell_str = str(val).strip()
                m = re.match(r'^\(\d+\)', cell_str)
                if not m:
                    continue
                title = m.group(0)
                # Пропускаем возможные пустые строки после маркера
                start = r + 1
                while start < rows and df.iloc[start, :].isna().all():
                    start += 1
                if start >= rows:
                    continue
                # Ищем конец таблицы: строка, где в том же столбце c появляется новый маркер
                end = start
                while end < rows:
                    cell = df.iloc[end, c]
                    if not pd.isna(cell) and re.match(r'^\(\d+\)', str(cell).strip()):
                        break
                    end += 1
                # Вырезаем блок
                block = df.iloc[start:end, c:]
                block = block.dropna(how='all')
                if not block.empty:
                    block = block.dropna(axis=1, how='all')
                    block = block.reset_index(drop=True)
                    result[title] = block
                used.add((r, c))
    return result

#### Обработка всех форм 8 и 33

In [3]:

BASE_DIR = r"C:\Users\urize\Ирткутск_туберкулез"

# Подпапка, где лежат файлы
DATA_DIR = os.path.join(BASE_DIR, "исходные_данные_сырые", "все формы")

# Генерируем список файлов (2016–2024)
files = []
for year in range(2016, 2025):
    # Для 2016 года расширение .xlsx, для остальных .xls
    ext = ".xlsx" if year == 2016 else ".xls"
    filename = f"f8_f33_{year}{ext}"
    full_path = os.path.join(DATA_DIR, filename)
    files.append(full_path)

# Проверяем, что все файлы существуют
print("Проверка файлов:")
for f in files:
    if os.path.exists(f):
        print(f"  ✅ {os.path.basename(f)}")
    else:
        print(f"  ❌ {os.path.basename(f)} - НЕ НАЙДЕН!")

# ----- Ваш основной цикл обработки (работает с абсолютными путями) -----
for filepath in files:
    # Извлекаем год из имени файла
    basename = os.path.basename(filepath)
    year = int(basename.split('_')[-1].split('.')[0])
    
    var_name = f'exdfs_dict_{year}'
    locals()[var_name] = excel_to_dataframes_named(filepath)
    
    print(f'\n=== {var_name} ===')
    print('Найденные таблицы:')
    print(list(locals()[var_name].keys()))

# Теперь переменные exdfs_dict_2016 ... exdfs_dict_2024 доступны

Проверка файлов:
  ✅ f8_f33_2016.xlsx
  ✅ f8_f33_2017.xls
  ✅ f8_f33_2018.xls
  ✅ f8_f33_2019.xls
  ✅ f8_f33_2020.xls
  ✅ f8_f33_2021.xls
  ✅ f8_f33_2022.xls
  ✅ f8_f33_2023.xls
  ✅ f8_f33_2024.xls

=== exdfs_dict_2016 ===
Найденные таблицы:
['(2100)', '(2110)', '(2120)', '(2130)', '(2200)', '(2300)', '(2310)', '(2330)', '(2400)', '(2500)', '(2510)', '(2520)', '(2610)', '(2620)', '(2600)', '(2700)', '(2710)', '(2800)', '(1000)', '(1002)']

=== exdfs_dict_2017 ===
Найденные таблицы:
['(2100)', '(2110)', '(2120)', '(2130)', '(2200)', '(2300)', '(2310)', '(2330)', '(2400)', '(2500)', '(2510)', '(2520)', '(2600)', '(2610)', '(2620)', '(2700)', '(2800)', '(2710)', '(1000)', '(1001)', '(1002)']

=== exdfs_dict_2018 ===
Найденные таблицы:
['(2100)', '(2110)', '(2120)', '(2130)', '(2200)', '(2300)', '(2310)', '(2330)', '(2400)', '(2500)', '(2510)', '(2520)', '(2600)', '(2610)', '(2620)', '(2700)', '(2800)', '(2710)', '(1000)', '(1001)', '(1002)']

=== exdfs_dict_2019 ===
Найденные таблицы:
['(

Получили набор словарей со всеми таблицами по годам

### Таблица 2800

#### Создание функции для таблицы 2800

In [4]:
def process_2800_to_long(df_2800, year):
    """
    Обрабатывает DataFrame формы 2800 и возвращает длинную таблицу
    
    Parameters:
    -----------
    df_2800 : pandas.DataFrame
        DataFrame с данными формы 2800
    year : int
        Год данных
    
    Returns:
    --------
    pandas.DataFrame
        Длинная таблица с колонками: Показатель, Возраст, Год, Значение
    """
    
    # Объединяем и заменяем исходные столбцы
    df = df_2800.copy()
    df[df.columns[0]] = df.iloc[:, 0].combine_first(df.iloc[:, 1])
    df = df.drop(columns=[df.columns[1]])
    
    # Заполняем пропуски в первой строке
    df.iloc[0] = df.iloc[0].ffill()
    
    # Объединяем две верхние строки в заголовок
    df.columns = df.iloc[2].astype(str) + '_' + df.iloc[0].fillna('') + ' ' + df.iloc[1].fillna('')
    df.columns = df.columns.str.strip()
    
    # Удаляем эти две строки
    df = df.iloc[3:].reset_index(drop=True)
    #df = df.drop(columns=['№ строки'])
    df = df.drop(index=0)
    
    df_clean = df
    
    # Преобразуем из широкого формата в длинный
    df_long = df_clean.melt(
        id_vars=['1_Наименование', '2_№ строки'],
        var_name='Возраст',
        value_name='Значение'
    )
    
    # Добавляем колонку с годом
    df_long['Год'] = year
    
    # Переименовываем для соответствия примеру
    df_long = df_long.rename(columns={'1_Наименование': 'Показатель'})
    
    # Переставляем колонки в нужном порядке
    df_long = df_long[['Показатель', 'Возраст', '2_№ строки', 'Год', 'Значение']]
    
    return df_long


# Пример использования:
# df_2019_2800 = exdfs_dict_2019['(2800)']
# df_2019_2800_long = process_2800_to_long(df_2019_2800, 2019)
#
# df_2018_2800 = exdfs_dict_2018['(2800)']
# df_2018_2800_long = process_2800_to_long(df_2018_2800, 2018)

#### Обработка всех таблиц 2800 за 2017-2024 и приведение к длинному формату

In [5]:
# Список годов
years = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]  

# Список для хранения имен созданных датафреймов
created_dfs = []

# Цикл по всем годам
for year in years:
    dict_name = f'exdfs_dict_{year}'
    df_name = f'df_{year}_2800'
    df_long_name = f'df_{year}_2800_long'
    
    # Достаем таблицу 2800 из словаря
    locals()[df_name] = locals()[dict_name]['(2800)']
    
    # Обрабатываем и приводим к длинному формату
    locals()[df_long_name] = process_2800_to_long(locals()[df_name], year)
    
    # Добавляем в список созданных датафреймов
    created_dfs.append(df_long_name)
    
    print(f'\n=== {year} год ===')
    print(locals()[df_long_name].head())

# Выводим список всех созданных датафреймов
print('\n=== Список созданных датафреймов ===')
for df_name in created_dfs:
    print(f'- {df_name}')


=== 2017 год ===
                    Показатель  Возраст 2_№ строки   Год Значение
0                из них с МБТ+  3_Всего          2  2017        2
1  Наблюдалось в отчетном году  3_Всего          3  2017        7
2        Лечились в стационаре  3_Всего          4  2017        3
3         Лечились амбулаторно  3_Всего          5  2017        4
4  Умерло от туберкулеза всего  3_Всего          6  2017      NaN

=== 2018 год ===
                    Показатель  Возраст 2_№ строки   Год Значение
0                из них с МБТ+  3_Всего          2  2018        2
1  Наблюдалось в отчетном году  3_Всего          3  2018        4
2        Лечились в стационаре  3_Всего          4  2018        2
3         Лечились амбулаторно  3_Всего          5  2018        2
4  Умерло от туберкулеза всего  3_Всего          6  2018      NaN

=== 2019 год ===
                    Показатель  Возраст 2_№ строки   Год Значение
0                из них с МБТ+  3_Всего          2  2019        6
1  Наблюдалось в отчет

#### Обработка таблицы 2800 за 2016 года. Файл имеет другой формат, поэтому обработаем вручную 

In [6]:
df_2016_2800 = exdfs_dict_2016['(2800)']
df_2016_2800.head()

,0,32,39,40,44,49,54,59,78,86,100,113,116
0,наименование,NaN,NaN,NaN,NaN,NaN,№ стр.,из них,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Всего,NaN,детей 0-14 лет,NaN,подростков 15-17 лет,NaN
2,1,NaN,NaN,NaN,NaN,NaN,2,3,NaN,4,NaN,5,NaN
3,Выявлено в текущем году,NaN,NaN,NaN,NaN,NaN,01,16,NaN,NaN,NaN,NaN,NaN
4,из них с МБТ+,NaN,NaN,NaN,NaN,NaN,02,1,NaN,NaN,NaN,NaN,NaN


In [7]:
# Убираем лишние строки и столбцы
df_2016_2800_cl = df_2016_2800.iloc[:10, :12]
df_2016_2800_cl = df_2016_2800_cl.dropna(axis=1, how='all')

df_2016_2800_cl.head()

,0,54,59,86,113
0,наименование,№ стр.,из них,NaN,NaN
1,NaN,NaN,Всего,детей 0-14 лет,подростков 15-17 лет
2,1,2,3,4,5
3,Выявлено в текущем году,01,16,NaN,NaN
4,из них с МБТ+,02,1,NaN,NaN


In [8]:
df_2016_2800_cl = exdfs_dict_2019['(2800)']

# Объединяем и заменяем исходные столбцы
df = df_2019_2800.copy()
df[df.columns[0]] = df.iloc[:, 0].combine_first(df.iloc[:, 1])
df = df.drop(columns=[df.columns[1]])

df.iloc[0] = df.iloc[0].ffill()
# Объединяем две верхние строки в заголовок
df.columns =  df.iloc[2].astype(str) + '_' + df.iloc[0].fillna('') + ' ' + df.iloc[1].fillna('')
df.columns = df.columns.str.strip()

# Удаляем эти две строки
df = df.iloc[3:].reset_index(drop=True)
#df = df.drop(columns=['№ строки'])
df = df.drop(index=0)

df_2016_2800_clean = df

df_2016_2800_clean.head()

,1_Наименование,2_№ строки,3_Всего,4_из них: детей 0-14 лет,5_из них: подростков 15-17 лет
1,из них с МБТ+,2,6,NaN,NaN
2,Наблюдалось в отчетном году,3,7,NaN,NaN
3,Лечились в стационаре,4,4,NaN,NaN
4,Лечились амбулаторно,5,3,NaN,NaN
5,Умерло от туберкулеза всего,6,NaN,NaN,NaN


In [9]:
# Преобразуем из широкого формата в длинный
# Расплавление: каждый столбец возрастной группы становится отдельной строкой
df_long = df_2016_2800_clean.melt(
    id_vars=['1_Наименование', '2_№ строки'],
    var_name='Возраст', 
    value_name='Значение'
)

# Добавляем колонку с годом
df_long['Год'] = 2016

# Переименовываем для соответствия примеру
df_long = df_long.rename(columns={'1_Наименование': 'Показатель'})

# Переставляем колонки в нужном порядке
df_2016_2800_long = df_long[['Показатель', 'Возраст', '2_№ строки', 'Год', 'Значение']]
df_2016_2800_long.head()

,Показатель,Возраст,2_№ строки,Год,Значение
0,из них с МБТ+,3_Всего,2,2016,6
1,Наблюдалось в отчетном году,3_Всего,3,2016,7
2,Лечились в стационаре,3_Всего,4,2016,4
3,Лечились амбулаторно,3_Всего,5,2016,3
4,Умерло от туберкулеза всего,3_Всего,6,2016,NaN


### Таблица 1000

#### Создание функции для таблицы 1000

In [10]:
def process_1000_to_long(df_1000, year):
    """
    Обрабатывает DataFrame формы 1000 и возвращает длинную таблицу
    
    Parameters:
    -----------
    df_1000 : pandas.DataFrame
        DataFrame с данными формы 1000
    year : int
        Год данных
    
    Returns:
    --------
    pandas.DataFrame
        Длинная таблица с колонками: Показатель, Пол, Возраст, Код по МКБ X пересмотра, Год, Значение
    """
    
    # Создаем копию
    df = df_1000.copy()
    
    # Удаляем лишние строки и столбцы
    df = df.iloc[:42, :18]
    df = df.replace(0, np.nan)
    df = df.dropna(axis=1, how='all')
    
    # Объединяем и заменяем исходные столбцы
    df[df.columns[0]] = df.iloc[:, 0].combine_first(df.iloc[:, 1])
    df = df.drop(columns=[df.columns[1]])
    
    # Заполняем пропущенные поля в заголовках по горизонтали
    df.iloc[0] = df.iloc[0].ffill()
    df.iloc[1] = df.iloc[1].ffill()
    
    # Склеиваем три строки в один заголовок (через пробел)
    df.columns = (df.iloc[0].fillna('') + ' ' + df.iloc[1].fillna('') + ' ' + df.iloc[2].fillna('')).str.strip()

    # Берём строку с графой, заполняем NaN пустыми строками, преобразуем в строки
    prefix = df.iloc[3].fillna('').astype(str)
    # Добавляем префикс к текущим именам столбцов (например, через пробел)
    new_columns = prefix + '_' + df.columns.astype(str)
    # Очищаем от лишних пробелов
    new_columns = new_columns.str.strip()
    # Присваиваем новые имена столбцам
    df.columns = new_columns

    
    # Удаляем эти три строки
    df = df.iloc[4:].reset_index(drop=True)
    
    # Заполняем пропущенные поля в заголовках по вертикали
    df.iloc[:, 0] = df.iloc[:, 0].ffill()
    df.iloc[:, 2] = df.iloc[:, 2].ffill()
    
    df_clean = df
    
    # Преобразуем из широкого формата в длинный
    df_long = df_clean.melt(
        id_vars=['1_Наименование показателя', '2_Пол', '3_№ строки', '4_Код по МКБ X пересмотра'],
        var_name='Возраст_long',
        value_name='Значение'
    )
    
    # Добавляем колонку с годом
    df_long['Год'] = year
    
    # Переименовываем для соответствия примеру
    df_long = df_long.rename(columns={'1_Наименование показателя': 'Показатель'})

    df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)

    # Очищаем колонку с возрастной группой от лишнего текста
    df_long['Возраст'] = df_long['Возраст_long'].str.replace(
        'Число больных с впервые в жизни установленным диагнозом активного туберкулеза', '', regex=False)
    df_long['Возраст'] = df_long['Возраст'].str.replace(
        'в том числе в возрасте ', '', regex=False)
    
    # Переставляем колонки в нужном порядке
    df_long = df_long[['Показатель', '2_Пол', 'Возраст', '3_№ строки','4_Код по МКБ X пересмотра', 'Год', 'Значение', 'Возраст_long']]
    
    return df_long


# Пример использования:
df_2017_1000 = exdfs_dict_2017['(1000)']
df_2017_1000_long = process_1000_to_long(df_2017_1000, 2017)
df_2017_1000_long.head()
# df_2018_1000 = exdfs_dict_2018['(1000)']
# df_2018_1000_long = process_1000_to_long(df_2018_1000, 2018)

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\3196728679.py:38: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prefix = df.iloc[3].fillna('').astype(str)
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\3196728679.py:52: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.iloc[:, 2] = df.iloc[:, 2].ffill()
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\3196728679.py:69: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)


,Показатель,2_Пол,Возраст,3_№ строки,4_Код по МКБ X пересмотра,Год,Значение,Возраст_long
0,Заболело туберкулезом - всего,М,5_ ВСЕГО,1,A15 - A19,2017,1535.0,5_Число больных с впервые в жизни установленным диагнозом активного туберкулеза ВСЕГО
1,Заболело туберкулезом - всего,Ж,5_ ВСЕГО,2,NaN,2017,789.0,5_Число больных с впервые в жизни установленным диагнозом активного туберкулеза ВСЕГО
2,"из них МБТ+, определяемый любым методом",М,5_ ВСЕГО,3,A15; A17 - A19 часть,2017,721.0,5_Число больных с впервые в жизни установленным диагнозом активного туберкулеза ВСЕГО
3,"из них МБТ+, определяемый любым методом",Ж,5_ ВСЕГО,4,NaN,2017,303.0,5_Число больных с впервые в жизни установленным диагнозом активного туберкулеза ВСЕГО
4,"Из числа больных всего (стр.01,02) – число больных туберкулезом органов дыхания",М,5_ ВСЕГО,5,A15; A16; A19 часть,2017,1485.0,5_Число больных с впервые в жизни установленным диагнозом активного туберкулеза ВСЕГО


#### Обработка всех таблиц 1000 за 2016-2024 и приведение к длинному формату

In [11]:
# Список годов
years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]  

# Список для хранения имен созданных датафреймов
created_dfs_1000 = []

# Цикл по всем годам
for year in years:
    dict_name = f'exdfs_dict_{year}'
    df_name = f'df_{year}_1000'
    df_long_name = f'df_{year}_1000_long'
    
    # Достаем таблицу 1000 из словаря
    locals()[df_name] = locals()[dict_name]['(1000)']
    
    # Обрабатываем и приводим к длинному формату
    locals()[df_long_name] = process_1000_to_long(locals()[df_name], year)
    
    # Добавляем в список созданных датафреймов
    created_dfs_1000.append(df_long_name)
    
    print(f'\n=== {year} год ===')
    print(locals()[df_long_name].head())

# Выводим список всех созданных датафреймов
print('\n=== Список созданных датафреймов для формы 1000 ===')
for df_name in created_dfs_1000:
    display(f'- {df_name}')

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\3196728679.py:38: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prefix = df.iloc[3].fillna('').astype(str)
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\3196728679.py:52: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.iloc[:, 2] = df.iloc[:, 2].ffill()
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\3196728679.py:69: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)
C:\Users\uri


=== 2016 год ===
                                                                        Показатель  \
0                                                    Заболело туберкулезом - всего   
1                                                    Заболело туберкулезом - всего   
2                                          из них МБТ+, определяемый любым методом   
3                                          из них МБТ+, определяемый любым методом   
4  Из числа больных всего (стр.01,02) – число больных туберкулезом органов дыхания   

  2_Пол   Возраст  3_№ строки 4_Код по МКБ X пересмотра   Год  Значение  \
0     М  5_ ВСЕГО           1                 A15 - A19  2016    1699.0   
1     Ж  5_ ВСЕГО           2                       NaN  2016     916.0   
2     М  5_ ВСЕГО           3      A15; A17 - A19 часть  2016     731.0   
3     Ж  5_ ВСЕГО           4                       NaN  2016     362.0   
4     М  5_ ВСЕГО           5       A15; A16; A19 часть  2016    1634.0   

              

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\3196728679.py:38: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prefix = df.iloc[3].fillna('').astype(str)
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\3196728679.py:52: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.iloc[:, 2] = df.iloc[:, 2].ffill()
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\3196728679.py:69: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)


'- df_2016_1000_long'

'- df_2017_1000_long'

...

'- df_2020_1000_long'

'- df_2021_1000_long'

'- df_2022_1000_long'

'- df_2023_1000_long'

'- df_2024_1000_long'

### Таблица 2700

#### Создание функции для таблицы 2700

In [12]:

def process_2700_to_long(df_2700, year):
    """
    Обрабатывает DataFrame формы 2700 и возвращает длинную таблицу
    
    Parameters:
    -----------
    df_2700 : pandas.DataFrame
        DataFrame с данными формы 2700
    year : int
        Год данных
    
    Returns:
    --------
    pandas.DataFrame
        Длинная таблица с колонками: Показатель, Категория выявления, Год, Значение
    """
    
    # Создаем копию
    df = df_2700.copy()
    
    # Удаляем лишние строки и столбцы
    df = df.iloc[:15, :10]
    df = df.dropna(axis=1, how='all')
    
    # Заполняем значениями подзаголовка
    df.iloc[0] = df.iloc[0].ffill()
    
    # Объединяем две верхние строки в заголовок
    df.columns = df.iloc[0].fillna('') + ' ' + df.iloc[1].fillna('')
    df.columns = df.columns.str.strip()

    # Берём строку с графой, заполняем NaN пустыми строками, преобразуем в строки
    prefix = df.iloc[2].fillna('').astype(str)
    # Добавляем префикс к текущим именам столбцов (например, через пробел)
    new_columns = prefix + '_' + df.columns.astype(str)
    # Очищаем от лишних пробелов
    new_columns = new_columns.str.strip()
    # Присваиваем новые имена столбцам
    df.columns = new_columns
    
    # Удаляем ненужные строки и столбцы
    df = df.drop(index=[0, 1, 2]).reset_index(drop=True)
    #df = df.drop(columns=['№ строки'], errors='ignore')
    df = df.rename(columns={'2_№ строки': '№ строки'})
    # Очистка ячеек и заголовков
    df = df.applymap(lambda x: x.strip().replace('\n', ' ').replace('\r', ' ') 
                     if isinstance(x, str) else x)
    
    # Замена 'Х' и 'X' на 0
    df = df.replace('Х', 0).replace('X', 0)
    
    # Очистка заголовков столбцов
    df.columns = df.columns.str.strip().str.replace('\n', ' ', regex=False).str.replace('\r', ' ', regex=False).str.replace('-', '', regex=False)
    
    # Делаем метку отличия для дублирующихся наименований
    if len(df) > 5:
        df.iloc[5, 0] = str(df.iloc[5, 0]) + '*'
    if len(df) > 6:
        df.iloc[6, 0] = str(df.iloc[6, 0]) + '*'
    if len(df) > 7:
        df.iloc[7, 0] = str(df.iloc[7, 0]) + '*'
    
    df_clean = df
    
    # Преобразуем из широкого формата в длинный
    df_long = df_clean.melt(
        id_vars=['1_Наименование', '№ строки'],
        var_name='Категория выявления',
        value_name='Значение'
    )
    
    # Добавляем колонку с годом
    df_long['Год'] = year
    
    # Переименовываем
    df_long = df_long.rename(columns={'1_Наименование': 'Показатель'})
    
    # Удаляем лишние пробелы
    df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)
    
    # Переставляем колонки в нужном порядке
    df_long = df_long[['Показатель', 'Категория выявления', '№ строки', 'Год', 'Значение']]
    
    return df_long


# Пример использования:
df_2017_2700 = exdfs_dict_2017['(2700)']
df_2017_2700_long = process_2700_to_long(df_2017_2700, 2017)
df_2017_2700_long.head()

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\1676964687.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prefix = df.iloc[2].fillna('').astype(str)
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\1676964687.py:46: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip().replace('\n', ' ').replace('\r', ' ')
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\1676964687.py:50: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace('Х', 0).replace('X', 0)
C:\Users\u

,Показатель,Категория выявления,№ строки,Год,Значение
0,Взято на учет в предыдущем году,3_Впервые выявленные больные Всего,1,2017,2293.0
1,Выбыло в другие территории,3_Впервые выявленные больные Всего,2,2017,128.0
2,Умерло от туберкулеза,3_Впервые выявленные больные Всего,3,2017,92.0
3,Умерло от других причин,3_Впервые выявленные больные Всего,4,2017,239.0
4,Диагноз туберкулеза снят,3_Впервые выявленные больные Всего,5,2017,16.0


In [13]:
df_2017_2700_long.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Показатель           96 non-null     object 
 1   Категория выявления  96 non-null     object 
 2   № строки             96 non-null     int64  
 3   Год                  96 non-null     int64  
 4   Значение             90 non-null     float64
dtypes: float64(1), int64(2), object(2)
memory usage: 3.9+ KB


#### Обработка всех таблиц 2700 за 2017-2024 и приведение к длинному формату

In [14]:
# Список годов
years = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]  

# Список для хранения имен созданных датафреймов
created_dfs_2700 = []

# Цикл по всем годам
for year in years:
    dict_name = f'exdfs_dict_{year}'
    df_name = f'df_{year}_2700'
    df_long_name = f'df_{year}_2700_long'
    
    # Достаем таблицу 2700 из словаря
    locals()[df_name] = locals()[dict_name]['(2700)']
    
    # Обрабатываем и приводим к длинному формату
    locals()[df_long_name] = process_2700_to_long(locals()[df_name], year)
    
    # Добавляем в список созданных датафреймов
    created_dfs_2700.append(df_long_name)
    
    print(f'\n=== {year} год ===')
    print(locals()[df_long_name].head())

# Выводим список всех созданных датафреймов
print('\n=== Список созданных датафреймов для формы 2700 ===')
for df_name in created_dfs_2700:
    print(f'- {df_name}')


=== 2017 год ===
                        Показатель                 Категория выявления  \
0  Взято на учет в предыдущем году  3_Впервые выявленные больные Всего   
1       Выбыло в другие территории  3_Впервые выявленные больные Всего   
2            Умерло от туберкулеза  3_Впервые выявленные больные Всего   
3          Умерло от других причин  3_Впервые выявленные больные Всего   
4         Диагноз туберкулеза снят  3_Впервые выявленные больные Всего   

   № строки   Год  Значение  
0         1  2017    2293.0  
1         2  2017     128.0  
2         3  2017      92.0  
3         4  2017     239.0  
4         5  2017      16.0  

=== 2018 год ===
                        Показатель                 Категория выявления  \
0  Взято на учет в предыдущем году  3_Впервые выявленные больные Всего   
1       Выбыло в другие территории  3_Впервые выявленные больные Всего   
2            Умерло от туберкулеза  3_Впервые выявленные больные Всего   
3          Умерло от других причин  3_Вперв

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\1676964687.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prefix = df.iloc[2].fillna('').astype(str)
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\1676964687.py:46: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip().replace('\n', ' ').replace('\r', ' ')
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\1676964687.py:50: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace('Х', 0).replace('X', 0)
C:\Users\u

#### Обработка таблицы 2700 за 2016 года. Файл имеет другой формат, поэтому обработаем вручную 

In [15]:
df_2016_2700 = exdfs_dict_2016['(2700)']
df_2016_2700.head()

,0,55,60,73,86,99,112,125,138,151,163
0,Наименование,№ стр.,Впервые выявленные больные,NaN,NaN,NaN,Больные с рецидивом туберкулеза,NaN,NaN,NaN,NaN
1,NaN,NaN,всего,в том числе бактерио-выделители,с деструкцией легочной ткани,без бактерио-выделения и деструкции,всего,в том числе бактерио-выделители,с деструкцией легочной ткани,без бактерио-выделения и деструкции,NaN
2,1,2,3,4,5,6,7,8,9,10,NaN
3,Взято на учёт в предыдущем году,1,2541,1143,1071,994,336,174,174,106,NaN
4,Выбыло в другие территории,2,147,53,42,53,17,8,6,4,NaN


In [16]:
 # Создаем копию
df = df_2016_2700.copy()
    
    # Удаляем лишние строки и столбцы
df = df.iloc[:15, :10]
df = df.dropna(axis=1, how='all')

# Заполняем значениями подзаголовка
df.iloc[0] = df.iloc[0].ffill()

# Объединяем две верхние строки в заголовок
df.columns = df.iloc[0].fillna('') + ' ' + df.iloc[1].fillna('')
df.columns = df.columns.str.strip()

# Берём строку с графой, заполняем NaN пустыми строками, преобразуем в строки
prefix = df.iloc[2].fillna('').astype(str)
# Добавляем префикс к текущим именам столбцов (например, через пробел)
new_columns = prefix + '_' + df.columns.astype(str)
# Очищаем от лишних пробелов
new_columns = new_columns.str.strip()
# Присваиваем новые имена столбцам
df.columns = new_columns
    
# Удаляем ненужные строки и столбцы
df = df.drop(index=[0,1,2]).reset_index()
df = df.drop(columns=['index'])
#df = df.drop(columns=['№ стр.'])
# Вариант 1: Переименовать конкретный столбец
df = df.rename(columns={'2_№ стр.': '№ строки'})
# Очистка ячеек и заголовков одной командой + замена X на 0
df = df.applymap(lambda x: x.strip().replace('\n', ' ').replace('\r', ' ') 
                 if isinstance(x, str) else x)

df = df.replace('Х', 0).replace('X', 0)  # Замена 'X' на 0 во всем датафрейме

df.columns = df.columns.str.strip().str.replace('\n', ' ', regex=False).str.replace('\r', ' ', regex=False).str.replace('-', '', regex=False)

# Делаем метку отличия для дублирующихся наименований
df.loc[5, df.columns[0]] = df.loc[5, df.columns[0]] + '*'
df.loc[6, df.columns[0]] = df.loc[6, df.columns[0]] + '*'
df.loc[7, df.columns[0]] = df.loc[7, df.columns[0]] + '*'

df_2016_2700_clean = df 
df_2016_2700_clean.head()

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\1626810896.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prefix = df.iloc[2].fillna('').astype(str)
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\1626810896.py:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip().replace('\n', ' ').replace('\r', ' ')


,1_Наименование,№ строки,3_Впервые выявленные больные всего,4_Впервые выявленные больные в том числе бактериовыделители,5_Впервые выявленные больные с деструкцией легочной ткани,6_Впервые выявленные больные без бактериовыделения и деструкции,7_Больные с рецидивом туберкулеза всего,8_Больные с рецидивом туберкулеза в том числе бактериовыделители,9_Больные с рецидивом туберкулеза с деструкцией легочной ткани,10_Больные с рецидивом туберкулеза без бактериовыделения и деструкции
0,Взято на учёт в предыдущем году,1,2541.0,1143.0,1071.0,994.0,336.0,174.0,174.0,106.0
1,Выбыло в другие территории,2,147.0,53.0,42.0,53.0,17.0,8.0,6.0,4.0
2,Умерло от туберкулеза,3,119.0,74.0,80.0,13.0,22.0,17.0,12.0,2.0
3,Умерло от других причин,4,302.0,138.0,132.0,97.0,48.0,25.0,20.0,12.0
4,Диагноз туберкулеза снят,5,30.0,2.0,12.0,18.0,1.0,NaN,1.0,NaN


In [17]:
    # Преобразуем из широкого формата в длинный
df_long = df_2016_2700_clean.melt(
    id_vars=['1_Наименование', '№ строки'],
    var_name='Категория выявления',
    value_name='Значение'
)
    
    # Добавляем колонку с годом
df_long['Год'] = 2016
    
    # Переименовываем для соответствия примеру
df_long = df_long.rename(columns={'1_Наименование': 'Показатель'})

# Удаляем лишние пробелы
df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)

    # Переставляем колонки в нужном порядке
df_2016_2700_long = df_long[['Показатель', 'Категория выявления', '№ строки', 'Год', 'Значение']]
df_2016_2700_long.head()

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\393405235.py:15: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)


,Показатель,Категория выявления,№ строки,Год,Значение
0,Взято на учёт в предыдущем году,3_Впервые выявленные больные всего,1,2016,2541.0
1,Выбыло в другие территории,3_Впервые выявленные больные всего,2,2016,147.0
2,Умерло от туберкулеза,3_Впервые выявленные больные всего,3,2016,119.0
3,Умерло от других причин,3_Впервые выявленные больные всего,4,2016,302.0
4,Диагноз туберкулеза снят,3_Впервые выявленные больные всего,5,2016,30.0


### Таблица 2100

#### Создание функции для таблицы 2100

In [18]:
def process_2100_to_long(df_2100, year):
    """
    Обрабатывает DataFrame формы 2100 и возвращает длинную таблицу
    
    Parameters:
    -----------
    df_2100 : pandas.DataFrame
        DataFrame с данными формы 2100
    year : int
        Год данных
    
    Returns:
    --------
    pandas.DataFrame
        Длинная таблица с колонками: Формы туберкулеза, Возраст, Код по МКБ-Х пересмотра, Год, Значение
    """
    
    # Создаем копию
    df = df_2100.copy()
    
    # Заменить все значения 0 на NaN во всем датафрейме
    df = df.replace(0, np.nan)
    # Удалить все столбцы, где все значения NaN
    df = df.dropna(axis=1, how='all')

    # Удаляем лишние строки и столбцы
    df = df.iloc[:17, :9]
    df = df.dropna(axis=1, how='all')
    
    # Заполняем значениями подзаголовка
    df.iloc[0] = df.iloc[0].ffill()
    df.iloc[1] = df.iloc[1].ffill()
    
    # Склеиваем три строки в один заголовок (через пробел)
    df.columns = (df.iloc[0].fillna('') + ' ' + df.iloc[1].fillna('') + ' ' + df.iloc[2].fillna('')).str.strip()

    # Берём строку с графой, заполняем NaN пустыми строками, преобразуем в строки
    prefix = df.iloc[3].fillna('').astype(str)
    # Добавляем префикс к текущим именам столбцов (например, через пробел)
    new_columns = prefix + '_' + df.columns.astype(str)
    # Очищаем от лишних пробелов
    new_columns = new_columns.str.strip()
    # Присваиваем новые имена столбцам
    df.columns = new_columns
    
    # Удаляем эти три строки
    df = df.iloc[4:].reset_index(drop=True)
    
    # Очистка ячеек и заголовков
    df = df.applymap(lambda x: x.strip().replace('\n', ' ').replace('\r', ' ') 
                     if isinstance(x, str) else x)
    
    # Удаляем ненужные столбцы
    #if '№ стро-ки' in df.columns:
     #   df = df.drop(columns=['№ стро-ки'])
    #if '№ строки' in df.columns:
      #  df = df.drop(columns=['№ строки'])
        # Переименовать столбец
    df = df.rename(columns={'2_№ стро-ки': '№ строки'})
    df = df.rename(columns={'2_№ стр.': '№ строки'})
    df = df.rename(columns={'3_Код по МКБ-Х пересмотра': 'Код по МКБ-Х пересмотра'})

    df_clean = df


    # Преобразуем из широкого формата в длинный
    df_long = df_clean.melt(
        id_vars=['1_Формы туберкулеза', 'Код по МКБ-Х пересмотра', '№ строки'],
        var_name='Возраст',
        value_name='Значение'
    )
    
    # Добавляем колонку с годом
    df_long['Год'] = year
    
    # Удаляем лишние пробелы
    df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)
    
    # Переставляем колонки в нужном порядке
    df_long = df_long[['1_Формы туберкулеза', 'Возраст', '№ строки', 'Код по МКБ-Х пересмотра', 'Год', 'Значение']]
    
    return df_long


# Пример использования:
df_2020_2100 = exdfs_dict_2020['(2100)']
df_2020_2100_long = process_2100_to_long(df_2020_2100, 2017)
df_2020_2100_long.head()

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\680519791.py:22: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(0, np.nan)
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\680519791.py:38: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prefix = df.iloc[3].fillna('').astype(str)
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\680519791.py:50: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip().replace('\n', ' ').replace('\r', ' ')
C:\Users\urize\AppData\Loc

,1_Формы туберкулеза,Возраст,№ строки,Код по МКБ-Х пересмотра,Год,Значение
0,Туберкулез органов дыхания - всего,4_Взято на учет в отчетном году больных с первые в жизни установленным диагнозом всего,1,А15; А16; А19 часть,2017,1230.0
1,в том числе туберкулез легких,4_Взято на учет в отчетном году больных с первые в жизни установленным диагнозом всего,2,А15.0-А15.3; А15.7 часть; А16.0-А16.2; А16.7 часть; А19 часть,2017,1180.0
2,из него: фиброзно-кавернозный,4_Взято на учет в отчетном году больных с первые в жизни установленным диагнозом всего,3,А15.0-А15.3; А16.0-А16.2,2017,21.0
3,Из общего числа больных туберкулезом легких выявлено в фазе распада,4_Взято на учет в отчетном году больных с первые в жизни установленным диагнозом всего,4,NaN,2017,578.0
4,Из общего числа больных туберкулезом легких выявлено без распада и без бактериовыделения,4_Взято на учет в отчетном году больных с первые в жизни установленным диагнозом всего,5,NaN,2017,395.0


#### Обработка всех таблиц 2100 за 2017-2024 и приведение к длинному формату

In [19]:
# Список годов
years = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]  

# Список для хранения имен созданных датафреймов
created_dfs_2100 = []

# Цикл по всем годам
for year in years:
    dict_name = f'exdfs_dict_{year}'
    df_name = f'df_{year}_2100'
    df_long_name = f'df_{year}_2100_long'
    
    # Достаем таблицу 2100 из словаря
    locals()[df_name] = locals()[dict_name]['(2100)']
    
    # Обрабатываем и приводим к длинному формату
    locals()[df_long_name] = process_2100_to_long(locals()[df_name], year)
    
    # Добавляем в список созданных датафреймов
    created_dfs_2100.append(df_long_name)
    
    print(f'\n=== {year} год ===')
    print(locals()[df_long_name].head())

# Выводим список всех созданных датафреймов
print('\n=== Список созданных датафреймов для формы 2100 ===')
for df_name in created_dfs_2100:
    print(f'- {df_name}')


=== 2017 год ===
                                                                        1_Формы туберкулеза  \
0                                                        Туберкулез органов дыхания - всего   
1                                                             в том числе туберкулез легких   
2                                                             из него: фиброзно-кавернозный   
3                       Из общего числа больных туберкулезом легких выявлено в фазе распада   
4  Из общего числа больных туберкулезом легких выявлено без распада и без бактериовыделения   

                                                                                  Возраст  \
0  4_Взято на учет в отчетном году больных с первые в жизни установленным диагнозом всего   
1  4_Взято на учет в отчетном году больных с первые в жизни установленным диагнозом всего   
2  4_Взято на учет в отчетном году больных с первые в жизни установленным диагнозом всего   
3  4_Взято на учет в отчетном году боль

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\680519791.py:38: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prefix = df.iloc[3].fillna('').astype(str)
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\680519791.py:50: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip().replace('\n', ' ').replace('\r', ' ')
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\680519791.py:77: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\680519791.py:38: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and w

#### Обработка таблицы 2100 за 2016 года. Файл имеет другой формат, поэтому обработаем вручную 

In [20]:
df_2016_2100 = exdfs_dict_2016['(2100)']
df_2016_2100.head()

,0,46,53,78,95,110,125,141,155,168
0,Формы туберкулёза,№ стр.,Код по МКБ X пересмотра,Взято на учёт в отчётном году пациентов с впервые в жизни установленным диагнозом,NaN,NaN,Контингенты больных на конец отчётного года,NaN,NaN,NaN
1,NaN,NaN,NaN,всего,в том числе,NaN,всего,в том числе,NaN,NaN
2,NaN,NaN,NaN,NaN,дети 0-14 лет,подростков 15-17 лет,NaN,дети 0-14 лет,подростков 15-17 лет,NaN
3,1,2,3,4,5,6,7,8,9,NaN
4,Туберкулёз органов дыхания-всего,01,A15;A16;A19 часть,2293,89,31,5751,110,33,NaN


In [21]:
 # Создаем копию
df = df_2016_2100.copy()
    
    # Удаляем лишние строки и столбцы
df = df.iloc[:17, :9]
df = df.dropna(axis=1, how='all')
# Заполняем значениями подзаголовка
df.iloc[0] = df.iloc[0].ffill()
df.iloc[1] = df.iloc[1].ffill()
# Склеиваем три строки в один заголовок (через пробел)
df.columns = (df.iloc[0].fillna('') + ' ' + df.iloc[1].fillna('') + ' ' + df.iloc[2].fillna('')).str.strip()

# Берём строку с графой, заполняем NaN пустыми строками, преобразуем в строки
prefix = df.iloc[3].fillna('').astype(str)
# Добавляем префикс к текущим именам столбцов (например, через пробел)
new_columns = prefix + '_' + df.columns.astype(str)
# Очищаем от лишних пробелов
new_columns = new_columns.str.strip()
# Присваиваем новые имена столбцам
df.columns = new_columns


# Удаляем эти три строки
df = df.iloc[4:].reset_index(drop=True)
# Очистка ячеек и заголовков от лишних пробелов, переносов строк и множественных пробелов
df = df.applymap(lambda x: ' '.join(x.strip().replace('\n', ' ').replace('\r', ' ').split()) 
                 if isinstance(x, str) else x)
# Удаляем ненужные строки и столбцы
#df = df.drop(columns=['№ стр.'])
df = df.rename(columns={'2_№ стр.': '№ строки'})

df = df.replace('ё', 'е', regex=True)
df.columns = df.columns.str.replace('ё', 'е', regex=False)

df.columns.values[2] = 'Код по МКБ-Х пересмотра'


df_2016_2100_clean = df
df_2016_2100_clean.head()

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\3321481850.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prefix = df.iloc[3].fillna('').astype(str)
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\3321481850.py:26: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: ' '.join(x.strip().replace('\n', ' ').replace('\r', ' ').split())


,1_Формы туберкулеза,№ строки,Код по МКБ-Х пересмотра,4_Взято на учет в отчетном году пациентов с впервые в жизни установленным диагнозом всего,5_Взято на учет в отчетном году пациентов с впервые в жизни установленным диагнозом в том числе дети 0-14 лет,6_Взято на учет в отчетном году пациентов с впервые в жизни установленным диагнозом в том числе подростков 15-17 лет,7_Контингенты больных на конец отчетного года всего,8_Контингенты больных на конец отчетного года в том числе дети 0-14 лет,9_Контингенты больных на конец отчетного года в том числе подростков 15-17 лет
0,Туберкулез органов дыхания-всего,01,A15;A16;A19 часть,2293,89.0,31.0,5751,110.0,33.0
1,в том числе туберкулез легких,02,A15.0-A15.3; A15.7 часть;A16.0-A16.2;A16.7 часть;A19 часть,2171,41.0,29.0,5574,49.0,31.0
2,из него: фиброзно-кавернозный,03,A15.0-A15.3; A16.0-A16.2,16,NaN,NaN,957,NaN,NaN
3,Из общего числа больных туберкулезом легких выявлено в фазе распада,04,NaN,929,1.0,13.0,2780,1.0,10.0
4,Из общего числа больных туберкулезом легких выявлено без распада и без бактериовыделения,05,NaN,968,40.0,14.0,2018,47.0,21.0


In [22]:
    # Преобразуем из широкого формата в длинный
df_long = df_2016_2100_clean.melt(
    id_vars=['1_Формы туберкулеза', 'Код по МКБ-Х пересмотра', '№ строки'],
    var_name='Возраст',
    value_name='Значение'
)
    
    # Добавляем колонку с годом
df_long['Год'] = 2016
    
    # Переименовываем для соответствия примеру
#df_long = df_long.rename(columns={'Наименование показателя': 'Показатель'})

# Удаляем лишние пробелы
df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)
   
    # Очищаем колонку с возрастной группой от лишнего текста
#df_long['Возраст'] = df_long['Возраст_long'].str.replace(
#        'Число больных с впервые в жизни установленным диагнозом активного туберкулеза', '', regex=False)
#df_long['Возраст'] = df_long['Возраст'].str.replace(
#       ' в том числе в возрасте', '', regex=False)

    # Переставляем колонки в нужном порядке
df_2016_2100_long = df_long[['1_Формы туберкулеза', 'Возраст', '№ строки', 'Код по МКБ-Х пересмотра', 'Год', 'Значение']]
df_2016_2100_long.head()

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\629057051.py:15: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)


,1_Формы туберкулеза,Возраст,№ строки,Код по МКБ-Х пересмотра,Год,Значение
0,Туберкулез органов дыхания-всего,4_Взято на учет в отчетном году пациентов с впервые в жизни установленным диагнозом всего,01,A15;A16;A19 часть,2016,2293.0
1,в том числе туберкулез легких,4_Взято на учет в отчетном году пациентов с впервые в жизни установленным диагнозом всего,02,A15.0-A15.3; A15.7 часть;A16.0-A16.2;A16.7 часть;A19 часть,2016,2171.0
2,из него: фиброзно-кавернозный,4_Взято на учет в отчетном году пациентов с впервые в жизни установленным диагнозом всего,03,A15.0-A15.3; A16.0-A16.2,2016,16.0
3,Из общего числа больных туберкулезом легких выявлено в фазе распада,4_Взято на учет в отчетном году пациентов с впервые в жизни установленным диагнозом всего,04,NaN,2016,929.0
4,Из общего числа больных туберкулезом легких выявлено без распада и без бактериовыделения,4_Взято на учет в отчетном году пациентов с впервые в жизни установленным диагнозом всего,05,NaN,2016,968.0


### Таблица 2200

#### Создание функции для таблицы 2200

In [23]:
def process_2200_to_long(df_2200, year):
    """
    Обрабатывает DataFrame формы 2200 и возвращает длинную таблицу
    
    Parameters:
    -----------
    df_2200 : pandas.DataFrame
        DataFrame с данными формы 2200
    year : int
        Год данных
    
    Returns:
    --------
    pandas.DataFrame
        Длинная таблица с колонками: Показатель, Возраст, №, Год, Значение
    """
    
    # Создаем копию
    df = df_2200.copy()
    
    # Объединяем две верхние строки в заголовок
    df.columns = df.iloc[0].fillna('') + ' ' + df.iloc[1].fillna('')
    df.columns = df.columns.str.strip()

    # Берём строку с графой, заполняем NaN пустыми строками, преобразуем в строки
    prefix = df.iloc[2].fillna('').astype(str)
    # Добавляем префикс к текущим именам столбцов (например, через пробел)
    new_columns = prefix + '_' + df.columns.astype(str)
    # Очищаем от лишних пробелов
    new_columns = new_columns.str.strip()
    # Присваиваем новые имена столбцам
    df.columns = new_columns

    
    # Удаляем первые три строки (заголовочные)
    df = df.iloc[3:].reset_index(drop=True)
    
    # Заполняем пропущенные поля в первом столбце по вертикали
    #df.iloc[:, 0] = df.iloc[:, 0].ffill()
    
    # Объединяем первый и второй столбцы (индексы 0 и 1) через пробел
    df['1_Наименование показателей'] = df.iloc[:, 0].fillna('').astype(str) + ' ' + df.iloc[:, 1].fillna('').astype(str)
    df['1_Наименование показателей'] = df['1_Наименование показателей'].str.strip()
    
    # Удаляем второй столбец (индекс 1)
    df = df.drop(columns=df.columns[1])
    df.columns.values[1] = 'Строка'

    df_clean = df
    
    # Преобразуем из широкого формата в длинный
    df_long = df_clean.melt(
        id_vars=['1_Наименование показателей', 'Строка'],
        var_name='Возраст',
        value_name='Значение'
    )
    
    # Добавляем колонку с годом
    df_long['Год'] = year
    
    # Переименовываем
    df_long = df_long.rename(columns={'1_Наименование показателей': 'Показатель'})
    
    # Переставляем колонки в нужном порядке
    df_long = df_long[['Показатель', 'Возраст', 'Строка', 'Год', 'Значение']]
    
    return df_long


# Пример использования:
df_2017_2200 = exdfs_dict_2017['(2200)']
df_2017_2200_long = process_2200_to_long(df_2017_2200, 2017)
df_2017_2200_long.head()

,Показатель,Возраст,Строка,Год,Значение
0,Впервые выявлено больных туберкулезом из числа осмотренных на туберкулез,3_Всего,(4/01,2017,1443
1,из них с применением: туберкулинодиагностики,3_Всего,2,2017,66
2,в том числе аллергена туберкулезного рекомбинантного в стандартном разведении,3_Всего,3,2017,NaN
3,флюрографии,3_Всего,4,2017,1366
4,бактериологических методов,3_Всего,5,2017,11


#### Обработка всех таблиц 2200 за 2017-2024 и приведение к длинному формату

In [24]:
# Список годов
years = [2017, 2018, 2019, 2020, 2022, 2023, 2024]  

# Список для хранения имен созданных датафреймов
created_dfs_2200 = []

# Цикл по всем годам
for year in years:
    dict_name = f'exdfs_dict_{year}'
    df_name = f'df_{year}_2200'
    df_long_name = f'df_{year}_2200_long'
    
    # Достаем таблицу 2200 из словаря
    locals()[df_name] = locals()[dict_name]['(2200)']
    
    # Обрабатываем и приводим к длинному формату
    locals()[df_long_name] = process_2200_to_long(locals()[df_name], year)
    
    # Добавляем в список созданных датафреймов
    created_dfs_2200.append(df_long_name)
    
    print(f'\n=== {year} год ===')
    print(locals()[df_long_name].head())

# Выводим список всех созданных датафреймов
print('\n=== Список созданных датафреймов для формы 2200 ===')
for df_name in created_dfs_2200:
    print(f'- {df_name}')


=== 2017 год ===
                                                                      Показатель  \
0       Впервые выявлено больных туберкулезом из числа осмотренных на туберкулез   
1                                   из них с применением: туберкулинодиагностики   
2  в том числе аллергена туберкулезного рекомбинантного в стандартном разведении   
3                                                                    флюрографии   
4                                                     бактериологических методов   

   Возраст Строка   Год Значение  
0  3_Всего  (4/01  2017     1443  
1  3_Всего      2  2017       66  
2  3_Всего      3  2017      NaN  
3  3_Всего      4  2017     1366  
4  3_Всего      5  2017       11  

=== 2018 год ===
                                                                      Показатель  \
0       Впервые выявлено больных туберкулезом из числа осмотренных на туберкулез   
1                                   из них с применением: туберкулинодиагностики 

#### Обработка таблицы 2200 за 2016 года. Файл имеет другой формат, поэтому обработаем вручную 

In [25]:
df_2016_2200 = exdfs_dict_2016['(2200)']
df_2016_2200.head()

,0,83,94,117,138
0,Наименование показателей,NaN,Всего,из них:,NaN
1,NaN,NaN,NaN,детей 0-14 лет,подростков 15-17 лет
2,1,2.0,3,4,5
3,Впервые выявлено больных туберкулезом из числа осмотренных на туберкулез,1.0,1601,86,27
4,из них с применением: туберкулинодиагностики,2.0,97,86,11


In [26]:
 # Создаем копию
df = df_2016_2200.copy()

# Объединяем две верхние строки в заголовок
df.columns = df.iloc[0].fillna('') + ' ' + df.iloc[1].fillna('')
df.columns = df.columns.str.strip()

# Берём строку с графой, заполняем NaN пустыми строками, преобразуем в строки
prefix = df.iloc[2].fillna('').astype(int).astype(str)
# Добавляем префикс к текущим именам столбцов (например, через пробел)
new_columns = prefix + '_' + df.columns.astype(str)
# Очищаем от лишних пробелов
new_columns = new_columns.str.strip()
# Присваиваем новые имена столбцам
df.columns = new_columns


# Удаляем эти 2 строки
df = df.iloc[3:].reset_index(drop=True)
    # Удаляем лишние строки и столбцы
df = df.iloc[:12, :5]
df = df.dropna(axis=1, how='all')
# Пример: переименовать первый столбец (индекс 0)
df.columns.values[1] = 'Строка'
df_2016_2200_clean = df
df_2016_2200_clean.head()

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\212924151.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prefix = df.iloc[2].fillna('').astype(int).astype(str)


,1_Наименование показателей,Строка,3_Всего,4_из них: детей 0-14 лет,5_подростков 15-17 лет
0,Впервые выявлено больных туберкулезом из числа осмотренных на туберкулез,1.0,1601,86,27
1,из них с применением: туберкулинодиагностики,2.0,97,86,11
2,в том числе аллергена туберкулезного рекомбинантного в стандартном разведении,3.0,NaN,NaN,NaN
3,флюрографии,4.0,1472,NaN,16
4,бактериологических методов,5.0,32,NaN,NaN


In [27]:
# Преобразуем из широкого формата в длинный
# Расплавление: каждый столбец возрастной группы становится отдельной строкой
df_long = df_2016_2200_clean.melt(
    id_vars=['1_Наименование показателей', 'Строка'],
    var_name='Возраст',
    value_name='Значение'
)

# Добавляем колонку с годом
df_long['Год'] = 2016

# Переименовываем для соответствия примеру
df_long = df_long.rename(columns={'1_Наименование показателей': 'Показатель'})

# Переставляем колонки в нужном порядке
df_2016_2200_long = df_long[['Показатель', 'Возраст', 'Строка', 'Год', 'Значение']]
df_2016_2200_long.head()

,Показатель,Возраст,Строка,Год,Значение
0,Впервые выявлено больных туберкулезом из числа осмотренных на туберкулез,3_Всего,1.0,2016,1601
1,из них с применением: туберкулинодиагностики,3_Всего,2.0,2016,97
2,в том числе аллергена туберкулезного рекомбинантного в стандартном разведении,3_Всего,3.0,2016,NaN
3,флюрографии,3_Всего,4.0,2016,1472
4,бактериологических методов,3_Всего,5.0,2016,32


#### Обработка таблицы 2200 за 2021 год. Файл имеет другой формат, поэтому обработаем вручную 

In [28]:
df_2021_2200 = exdfs_dict_2021['(2200)']

In [29]:
# Обработка df_2021_2200
year = 2021
df = df_2021_2200.copy()

# Удаляем второй столбец (индекс 1)
df = df.drop(columns=df.columns[1])
# Объединяем две верхние строки в заголовок
df.columns = df.iloc[0].fillna('') + ' ' + df.iloc[1].fillna('')
df.columns = df.columns.str.strip()



# Берём строку с графой, заполняем NaN пустыми строками, преобразуем в строки
prefix = df.iloc[2].fillna('').astype(str)
# Добавляем префикс к текущим именам столбцов (через пробел)
new_columns = prefix + '_' + df.columns.astype(str)
# Очищаем от лишних пробелов
new_columns = new_columns.str.strip()
# Присваиваем новые имена столбцам
df.columns = new_columns

# Удаляем первые три строки (заголовочные)
df = df.iloc[3:].reset_index(drop=True)

# Объединяем первый и второй столбцы (индексы 0 и 1) через пробел
df['1_Наименование показателей'] = df.iloc[:, 0].fillna('').astype(str) + ' ' + df.iloc[:, 1].fillna('').astype(str)
df['1_Наименование показателей'] = df['1_Наименование показателей'].str.strip()

# Удаляем второй столбец (индекс 1)
df = df.drop(columns=df.columns[1])
df.columns.values[1] = 'Строка'

df_clean = df

df_clean.head()

,1_Наименование показателей,Строка,3_Всего,4_из них: детей 0-14 лет,5_подростков 15-17 лет
0,Впервые выявлено больных туберкулезом из числа осмотренных на туберкулез,(4/01,732,24,9
1,из них с применением: туберкулинодиагностики,2,26,24,2
2,в том числе аллергена туберкулезного рекомбинантного в стандартном разведении,3,15,13,2
3,флюрографии,4,698,0,7
4,бактериологических методов,5,8,0,0


In [30]:
# Преобразуем из широкого формата в длинный
df_long = df_clean.melt(
    id_vars=['1_Наименование показателей', 'Строка'],
    var_name='Возраст',
    value_name='Значение'
)

# Добавляем колонку с годом
df_long['Год'] = year

# Переименовываем
df_long = df_long.rename(columns={'1_Наименование показателей': 'Показатель'})

# Переставляем колонки в нужном порядке
df_2021_2200_long = df_long[['Показатель', 'Возраст', 'Строка', 'Год', 'Значение']]

# Проверяем результат
print(f"Обработано строк: {len(df_2021_2200_long)}")
print(f"Колонки: {list(df_2021_2200_long.columns)}")
print(f"Год: {df_2021_2200_long['Год'].unique()[0]}")
print(f"Уникальные значения в 'Строка': {sorted(df_2021_2200_long['Строка'].astype(str).unique())}")

df_2021_2200_long.head()

Обработано строк: 36
Колонки: ['Показатель', 'Возраст', 'Строка', 'Год', 'Значение']
Год: 2021
Уникальные значения в 'Строка': ['(4/01', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9']


,Показатель,Возраст,Строка,Год,Значение
0,Впервые выявлено больных туберкулезом из числа осмотренных на туберкулез,3_Всего,(4/01,2021,732
1,из них с применением: туберкулинодиагностики,3_Всего,2,2021,26
2,в том числе аллергена туберкулезного рекомбинантного в стандартном разведении,3_Всего,3,2021,15
3,флюрографии,3_Всего,4,2021,698
4,бактериологических методов,3_Всего,5,2021,8


### Таблица 2300

#### Создание функции для таблицы 2300

In [31]:
def process_2300_to_long(df_2300, year):
    """
    Обрабатывает DataFrame формы 2300 и возвращает длинную таблицу
    
    Parameters:
    -----------
    df_2300 : pandas.DataFrame
        DataFrame с данными формы 2300
    year : int
        Год данных
    
    Returns:
    --------
    pandas.DataFrame
        Длинная таблица с колонками: показатель, формы, Строка, Год, Значение
    """
    
    # Создаем копию
    df = df_2300.copy()
    
    # Заполняем пропущенные поля в заголовках по горизонтали
    df.iloc[0] = df.iloc[0].ffill()
    
    # Объединяем и заменяем исходные столбцы
    df[df.columns[0]] = df.iloc[:, 0].combine_first(df.iloc[:, 1])
    df = df.drop(columns=[df.columns[1]])
    
    # Заполняем конкретные ячейки
    if len(df.columns) > 5:
        df.iloc[0, 5] = 'Туберкулез легких'
    if len(df.columns) > 6:
        df.iloc[0, 6] = 'Туберкулез легких'
    if len(df.columns) > 7:
        df.iloc[0, 7] = 'Туберкулез легких'
    
    # Объединяем верхние строки в заголовок (первая и третья)
    df.columns = df.iloc[0].fillna('') + ' ' + df.iloc[2].fillna('')
    df.columns = df.columns.str.strip()

    prefix = df.iloc[3].fillna('').astype(str)
    # Добавляем префикс к текущим именам столбцов (например, через пробел)
    new_columns = prefix + '_' + df.columns.astype(str)
    # Очищаем от лишних пробелов
    new_columns = new_columns.str.strip()
    # Присваиваем новые имена столбцам
    df.columns = new_columns

    # Удаляем ненужные строки
    df = df.drop(index=[0, 1, 2, 3]).reset_index(drop=True)
    
    # Переименовываем столбец '№'
    if '2_№' in df.columns:
        df = df.rename(columns={'2_№': 'Строка'})
    
    # Дать имя первому столбцу
    df.columns.values[0] = 'показатель'
    
    df_clean = df
    
    # Преобразуем из широкого формата в длинный
    df_long = df_clean.melt(
        id_vars=['показатель', 'Строка'],
        var_name='формы',
        value_name='Значение'
    )
    
    # Добавляем колонку с годом
    df_long['Год'] = year
    
    # Удаляем лишние пробелы
    df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)
    
    # Очищаем колонку показатель (замена Ш на III)
    df_long['показатель'] = df_long['показатель'].str.replace('Ш', 'III', regex=False)
    
    # Переставляем колонки в нужном порядке
    df_long = df_long[['показатель', 'формы', 'Строка', 'Год', 'Значение']]
    
    return df_long


# Пример использования:
# df_2017_2300 = exdfs_dict_2017['(2300)']
# df_2017_2300_long = process_2300_to_long(df_2017_2300, 2017)

#### Обработка всех таблиц 2300 за 2017-2024 и приведение к длинному формату

In [32]:
# Список годов
years = [2017, 2018, 2019, 2021, 2022, 2023, 2024]  

# Список для хранения имен созданных датафреймов
created_dfs_2300 = []

# Цикл по всем годам
for year in years:
    dict_name = f'exdfs_dict_{year}'
    df_name = f'df_{year}_2300'
    df_long_name = f'df_{year}_2300_long'
    
    # Достаем таблицу 2300 из словаря
    locals()[df_name] = locals()[dict_name]['(2300)']
    
    # Обрабатываем и приводим к длинному формату
    locals()[df_long_name] = process_2300_to_long(locals()[df_name], year)
    
    # Добавляем в список созданных датафреймов
    created_dfs_2300.append(df_long_name)
    
    print(f'\n=== {year} год ===')
    print(locals()[df_long_name].head())

# Выводим список всех созданных датафреймов
print('\n=== Список созданных датафреймов для формы 2300 ===')
for df_name in created_dfs_2300:
    print(f'- {df_name}')


=== 2017 год ===
                 показатель                               формы  Строка   Год  \
0   Взято на учет рецидивов  3_Туберкулез органов дыхания Всего       1  2017   
1      из них из III группы  3_Туберкулез органов дыхания Всего       2  2017   
2                   Прибыло  3_Туберкулез органов дыхания Всего       3  2017   
3   Переведено в III группу  3_Туберкулез органов дыхания Всего       4  2017   
4  Диагноз туберкулеза снят  3_Туберкулез органов дыхания Всего       5  2017   

   Значение  
0     277.0  
1      95.0  
2    1238.0  
3    2362.0  
4      14.0  

=== 2018 год ===
                 показатель                               формы  Строка   Год  \
0   Взято на учет рецидивов  3_Туберкулез органов дыхания Всего       1  2018   
1      из них из III группы  3_Туберкулез органов дыхания Всего       2  2018   
2                   Прибыло  3_Туберкулез органов дыхания Всего       3  2018   
3   Переведено в III группу  3_Туберкулез органов дыхания Всего      

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\2824766697.py:40: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prefix = df.iloc[3].fillna('').astype(str)
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\2824766697.py:71: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\2824766697.py:40: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prefix = df.iloc[3].fillna('').astype(str)
C:\User

#### Обработка таблицы 2300 за 2016 года. Файл имеет другой формат, поэтому обработаем вручную 

In [33]:
df_2016_2300 = exdfs_dict_2016['(2300)']
df_2016_2300.head()

,0,36,37,50,94,108,123,137,151,168,182
0,NaN,№ стр,Туберкулез органов дыхания,NaN,NaN,NaN,NaN,NaN,Другие формы туберкулеза,NaN,NaN
1,NaN,NaN,Всего,NaN,NaN,из них туберкулез легких,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,Всего,дети 0-14 лет,подростки 15-17 лет,Всего,дети 0-14 лет,подростки 15-17 лет,Всего,дети 0-14 лет,подростки 15-17 лет
3,1,2,3,4,5,6,7,8,9,10,11
4,Взято на учет рецидивов,01,285,NaN,1,274,NaN,1,10,NaN,NaN


In [34]:
 # Создаем копию
df = df_2016_2300.copy()
    # Заполняем пропущенные поля в заголовках по горизонтали
df.iloc[0] = df.iloc[0].ffill()
#df.iloc[1] = df.iloc[1].ffill()
# Объединяем и заменяем исходные столбцы
#df[df.columns[0]] = df.iloc[:, 0].combine_first(df.iloc[:, 1])
#df = df.drop(columns=[df.columns[1]])
df.iloc[0, 5] = 'Туберкулез легких' 
df.iloc[0, 6] = 'Туберкулез легких' 
df.iloc[0, 7] = 'Туберкулез легких' 


# Объединяем две верхние строки в заголовок
df.columns = df.iloc[0].fillna('') + ' ' + df.iloc[2].fillna('')
df.columns = df.columns.str.strip()

# Берём строку с графой, заполняем NaN пустыми строками, преобразуем в строки
prefix = df.iloc[3].fillna('').astype(str)
# Добавляем префикс к текущим именам столбцов (например, через пробел)
new_columns = prefix + '_' + df.columns.astype(str)
# Очищаем от лишних пробелов
new_columns = new_columns.str.strip()
# Присваиваем новые имена столбцам
df.columns = new_columns

# Удаляем ненужные строки
df = df.drop(index=[0,1,2,3])
df = df.rename(columns={'2_№ стр': 'Строка'})
df = df.reset_index()
# Или если нужно удалить и перезаписать df
df.drop(columns='index', inplace=True)
# Дать имя первому столбцу
df.columns.values[0] = 'показатель'
df['Строка'] = df['Строка'].astype(int)
df_2016_2300_clean = df
df_2016_2300_clean.head()

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\2871295444.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prefix = df.iloc[3].fillna('').astype(str)


,показатель,Строка,3_Туберкулез органов дыхания Всего,4_Туберкулез органов дыхания дети 0-14 лет,5_Туберкулез органов дыхания подростки 15-17 лет,6_Туберкулез легких Всего,7_Туберкулез легких дети 0-14 лет,8_Туберкулез легких подростки 15-17 лет,9_Другие формы туберкулеза Всего,10_Другие формы туберкулеза дети 0-14 лет,11_Другие формы туберкулеза подростки 15-17 лет
0,Взято на учет рецидивов,1,285,NaN,1,274,NaN,1,10,NaN,NaN
1,из них из III группы,2,96,NaN,1,93,NaN,1,5,NaN,NaN
2,Прибыло,3,1098,18,4,926,11,4,61,3,1
3,Переведено в III группу,4,2531,127,26,2393,48,21,106,3,NaN
4,Диагноз туберкулеза снят,5,30,3,NaN,26,NaN,NaN,1,NaN,NaN


In [35]:
    # Преобразуем из широкого формата в длинный
df_long = df_2016_2300_clean.melt(
    id_vars=['показатель', 'Строка'],
    var_name='формы',
    value_name='Значение'
)
    
    # Добавляем колонку с годом
df_long['Год'] = 2016
    
# Удаляем лишние пробелы
df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)
   
    # Очищаем колонку с возрастной группой от лишнего текста
df_long['показатель'] = df_long['показатель'].str.replace(
       'Ш', 'III', regex=False)

    # Переставляем колонки в нужном порядке
df_2016_2300_long = df_long[['показатель', 'формы', 'Строка', 'Год', 'Значение']]
df_2016_2300_long.head()

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\3248409336.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)


,показатель,формы,Строка,Год,Значение
0,Взято на учет рецидивов,3_Туберкулез органов дыхания Всего,1,2016,285.0
1,из них из III группы,3_Туберкулез органов дыхания Всего,2,2016,96.0
2,Прибыло,3_Туберкулез органов дыхания Всего,3,2016,1098.0
3,Переведено в III группу,3_Туберкулез органов дыхания Всего,4,2016,2531.0
4,Диагноз туберкулеза снят,3_Туберкулез органов дыхания Всего,5,2016,30.0


#### Обработка таблицы 2300 за 2020 год. Файл имеет другой формат, поэтому обработаем вручную 

In [36]:
df_2020_2300 = exdfs_dict_2020['(2300)']
df_2020_2300.head()

,0,2,3,4,5,6,7,8,9,10,11,12
0,NaN,NaN,№,Туберкулез органов дыхания,NaN,NaN,NaN,NaN,NaN,Другие формы туберкулеза,NaN,NaN
1,NaN,NaN,NaN,Всего,NaN,NaN,из них туберкулез легких,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,Всего,дети 0-14 лет,подростки 15-17 лет,Всего,дети 0-14 лет,подростки 15-17 лет,Всего,дети 0-14 лет,подростки 15-17 лет
3,1,NaN,2,3,4,5,6,7,8,9,10,11
4,Взято на учет рецидивов,NaN,1,297,0,0,289,0,0,18,0,0


In [37]:
 # Создаем копию
df = df_2020_2300.copy()
    # Заполняем пропущенные поля в заголовках по горизонтали
df.iloc[0] = df.iloc[0].ffill()
df.iloc[1] = df.iloc[1].ffill()
 #Объединяем и заменяем исходные столбцы
df[df.columns[0]] = df.iloc[:, 0].combine_first(df.iloc[:, 1])
df = df.drop(columns=[df.columns[1]])
df.iloc[0, 5] = 'Туберкулез легких' 
df.iloc[0, 6] = 'Туберкулез легких' 
df.iloc[0, 7] = 'Туберкулез легких' 
# Объединяем две верхние строки в заголовок
df.columns = df.iloc[0].fillna('') + ' ' + df.iloc[2].fillna('')
df.columns = df.columns.str.strip()

# Берём строку с графой, заполняем NaN пустыми строками, преобразуем в строки
prefix = df.iloc[3].fillna('').astype(str)
# Добавляем префикс к текущим именам столбцов (например, через пробел)
new_columns = prefix + '_' + df.columns.astype(str)
# Очищаем от лишних пробелов
new_columns = new_columns.str.strip()
# Присваиваем новые имена столбцам
df.columns = new_columns

# Удаляем ненужные строки
df = df.drop(index=[0,1,2,3,12])
df = df.rename(columns={'2_№': 'Строка'})
df = df.reset_index()
df.drop(columns='index', inplace=True)
df.columns.values[0] = 'показатель'
df['Строка'] = df['Строка'].astype(int)

df_2020_2300_clean = df
df_2020_2300_clean.head()

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\2868460414.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prefix = df.iloc[3].fillna('').astype(str)


,показатель,Строка,3_Туберкулез органов дыхания Всего,4_Туберкулез органов дыхания дети 0-14 лет,5_Туберкулез органов дыхания подростки 15-17 лет,6_Туберкулез легких Всего,7_Туберкулез легких дети 0-14 лет,8_Туберкулез легких подростки 15-17 лет,9_Другие формы туберкулеза Всего,10_Другие формы туберкулеза дети 0-14 лет,11_Другие формы туберкулеза подростки 15-17 лет
0,Взято на учет рецидивов,1,297,0,0,289,0,0,18,0,0
1,из них из Ш группы,2,87,0,0,79,0,0,9,0,0
2,Прибыло,3,469,0,6,458,0,6,41,0,0
3,Переведено в Ш группу,4,1833,46,18,1772,24,16,168,4,1
4,Диагноз туберкулеза снят,5,12,0,0,10,0,0,0,0,0


In [38]:
    # Преобразуем из широкого формата в длинный
df_long = df_2020_2300_clean.melt(
    id_vars=['показатель', 'Строка'],
    var_name='формы',
    value_name='Значение'
)
    
    # Добавляем колонку с годом
df_long['Год'] = 2020
    
# Удаляем лишние пробелы
df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)
   
    # Очищаем колонку с возрастной группой от лишнего текста
df_long['показатель'] = df_long['показатель'].str.replace(
       'Ш', 'III', regex=False)

    # Переставляем колонки в нужном порядке
df_2020_2300_long = df_long[['показатель', 'формы', 'Строка', 'Год', 'Значение']]
df_2020_2300_long.head()

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\636649680.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)


,показатель,формы,Строка,Год,Значение
0,Взято на учет рецидивов,3_Туберкулез органов дыхания Всего,1,2020,297
1,из них из III группы,3_Туберкулез органов дыхания Всего,2,2020,87
2,Прибыло,3_Туберкулез органов дыхания Всего,3,2020,469
3,Переведено в III группу,3_Туберкулез органов дыхания Всего,4,2020,1833
4,Диагноз туберкулеза снят,3_Туберкулез органов дыхания Всего,5,2020,12


### Таблица 2400

#### Создание функции для таблицы 2400

In [39]:
def process_2400_to_long(df_2400, year):
    """
    Обрабатывает DataFrame формы 2400 и возвращает длинную таблицу
    
    Parameters:
    -----------
    df_2400 : pandas.DataFrame
        DataFrame с данными формы 2400
    year : int
        Год данных
    
    Returns:
    --------
    pandas.DataFrame
        Длинная таблица с колонками: группа_учета, состоит, Строка, Год, Значение
    """
    
    # Создаем копию
    df = df_2400.copy()
    
    # Объединяем и заменяем исходные столбцы
    df[df.columns[1]] = df.iloc[:, 1].combine_first(df.iloc[:, 2])
    df = df.drop(columns=[df.columns[2]])
    
    # Назначить первую строку названием столбцов
    df.columns = df.iloc[0].values
    
    # Берём строку с графой, заполняем NaN пустыми строками, преобразуем в строки
    prefix = df.iloc[1].fillna('').astype(str)
    # Добавляем префикс к текущим именам столбцов (например, через пробел)
    new_columns = prefix + '_' + df.columns.astype(str)
    # Очищаем от лишних пробелов
    new_columns = new_columns.str.strip()
    # Присваиваем новые имена столбцам
    df.columns = new_columns
    
    df = df[1:].reset_index(drop=True)
    
    # Заменить первый столбец объединенным значением
    df.iloc[:, 0] = df.iloc[:, 0].fillna('').astype(str) + ' ' + df.iloc[:, 1].fillna('').astype(str)
    df.iloc[:, 0] = df.iloc[:, 0].str.strip()
    
    # Затем удалить второй столбец
    df = df.drop(columns=df.columns[1])
    
    # Удаляем лишние строки и столбцы
    df = df.iloc[:16, :9]
    df = df.drop(index=0).reset_index(drop=True)
    
    # Переименовываем столбцы
    if '2_№' in df.columns:
        df = df.rename(columns={'2_№': 'Строка'})
    if '1_Состоит по группам учета' in df.columns:
        df = df.rename(columns={'1_Состоит по группам учета': 'группа_учета'})
    
    df_clean = df
    
    # Преобразуем из широкого формата в длинный
    df_long = df_clean.melt(
        id_vars=['группа_учета', 'Строка'],
        var_name='состоит',
        value_name='Значение'
    )
    
    # Добавляем колонку с годом
    df_long['Год'] = year
    
    # Удаляем лишние пробелы
    df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)
    
    # Переставляем колонки в нужном порядке
    df_long = df_long[['группа_учета', 'состоит', 'Строка', 'Год', 'Значение']]
    
    return df_long


# Пример использования:
# df_2017_2400 = exdfs_dict_2017['(2400)']
# df_2017_2400_long = process_2400_to_long(df_2017_2400, 2017)

#### Обработка всех таблиц 2400 за 2017-2024 и приведение к длинному формату

In [40]:
# Список годов
years = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]  

# Список для хранения имен созданных датафреймов
created_dfs_2400 = []

# Цикл по всем годам
for year in years:
    dict_name = f'exdfs_dict_{year}'
    df_name = f'df_{year}_2400'
    df_long_name = f'df_{year}_2400_long'
    
    # Достаем таблицу 2400 из словаря
    locals()[df_name] = locals()[dict_name]['(2400)']
    
    # Обрабатываем и приводим к длинному формату
    locals()[df_long_name] = process_2400_to_long(locals()[df_name], year)
    
    # Добавляем в список созданных датафреймов
    created_dfs_2400.append(df_long_name)
    
    print(f'\n=== {year} год ===')
    print(locals()[df_long_name].head())

# Выводим список всех созданных датафреймов
print('\n=== Список созданных датафреймов для формы 2400 ===')
for df_name in created_dfs_2400:
    print(f'- {df_name}')


=== 2017 год ===
                                                                                   группа_учета  \
0        Взрослые, из них нуждающиеся в определении активности туберкулезного процесса (гр. 0А)   
1                            в проведении дифференциально-диагностических мероприятий (гр. 0 Б)   
2                    Взрослые с неактивным туб процессом после клинического излечения (гр. III)   
3      Взрослые состоящие в бытовом и производственном контакте с бактериовыделителем (гр. IVА)   
4  в бытовом и производственном контакте с больным туберкулезом без бактериовыделения (гр. IVА)   

                  состоит  Строка   Год  Значение  
0  3_Взято в текущем году       1  2017     411.0  
1  3_Взято в текущем году       2  2017     381.0  
2  3_Взято в текущем году       3  2017    2597.0  
3  3_Взято в текущем году       4  2017    1192.0  
4  3_Взято в текущем году       5  2017    1167.0  

=== 2018 год ===
                                                         

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\2498830255.py:69: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\2498830255.py:69: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\2498830255.py:69: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\2498830255.py:69: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)
C:\Users\urize\AppData\Local\Temp\ipykernel_

#### Обработка таблицы 2400 за 2016 года. Файл имеет другой формат, поэтому обработаем вручную 

In [41]:
df_2016_2400 = exdfs_dict_2016['(2400)']
df_2016_2400.head()

,0,37,44,94,117,138,161,182,205,225
0,Состоит по группам учета,№ стр,Взято в текущем году,Подлежало ХП или пробному лечению,Прошли курс ХП пробного лечения,Впервые выявлено больных с активным ТБ,Снято с учета,Выбыло,Состоит на конец года,NaN
1,1,2,3,4,5,6,7,8,9,NaN
2,"Взрослые, из них нуждающиеся: в определении активности туберкулезного процесса (гр.0А)",01,441,431,391,150,289,21,109,NaN
3,в проведении дифференциально-диагностических мероприятий (гр.0Б),02,450,245,210,184,281,11,45,NaN
4,Взрослые с неактивным туберкулезным процессом после клинического излечения (гр.III),03,2531,3418,3170,101,2312,377,4948,NaN


In [42]:
 # Создаем копию
df = df_2016_2400.copy()

# Назначить первую строку названием столбцов
df.columns = df.iloc[0].values

# Берём строку с графой, заполняем NaN пустыми строками, преобразуем в строки
prefix = df.iloc[1].fillna('').astype(str)
# Добавляем префикс к текущим именам столбцов (например, через пробел)
new_columns = prefix + '_' + df.columns.astype(str)
# Очищаем от лишних пробелов
new_columns = new_columns.str.strip()
# Присваиваем новые имена столбцам
df.columns = new_columns

df = df[1:].reset_index(drop=True)
    # Удаляем лишние строки и столбцы
df = df.iloc[:15, :9]
df = df.drop(index=0)
df = df.rename(columns={'2_№ стр': 'Строка'})
df = df.rename(columns={'1_Состоит по группам учета': 'группа_учета'})
df_2016_2400_clean = df
df_2016_2400_clean.head()

,группа_учета,Строка,3_Взято в текущем году,4_Подлежало ХП или пробному лечению,5_Прошли курс ХП пробного лечения,6_Впервые выявлено больных с активным ТБ,7_Снято с учета,8_Выбыло,9_Состоит на конец года
1,"Взрослые, из них нуждающиеся: в определении активности туберкулезного процесса (гр.0А)",01,441,431,391,150,289,21,109
2,в проведении дифференциально-диагностических мероприятий (гр.0Б),02,450,245,210,184,281,11,45
3,Взрослые с неактивным туберкулезным процессом после клинического излечения (гр.III),03,2531,3418,3170,101,2312,377,4948
4,Взрослые состоящие :в бытовом и производственном контакте с бактериовыделителем (гр.IVА),04,1211,1743,1575,14,1008,194,2622
5,в бытовом и производственном контакте с больным туберкулезом без бактериовыделния (гр.IVА),05,1329,1272,1122,1,1379,311,1981


In [43]:
    # Преобразуем из широкого формата в длинный
df_long = df_2016_2400_clean.melt(
    id_vars=['группа_учета', 'Строка'],
    var_name='состоит',
    value_name='Значение'
)
    
    # Добавляем колонку с годом
df_long['Год'] = 2016
    
# Удаляем лишние пробелы
df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)
   

    # Переставляем колонки в нужном порядке
df_2016_2400_long = df_long[['группа_учета', 'состоит', 'Строка', 'Год', 'Значение']]
df_2016_2400_long.head()

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\105310402.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)


,группа_учета,состоит,Строка,Год,Значение
0,"Взрослые, из них нуждающиеся: в определении активности туберкулезного процесса (гр.0А)",3_Взято в текущем году,01,2016,441.0
1,в проведении дифференциально-диагностических мероприятий (гр.0Б),3_Взято в текущем году,02,2016,450.0
2,Взрослые с неактивным туберкулезным процессом после клинического излечения (гр.III),3_Взято в текущем году,03,2016,2531.0
3,Взрослые состоящие :в бытовом и производственном контакте с бактериовыделителем (гр.IVА),3_Взято в текущем году,04,2016,1211.0
4,в бытовом и производственном контакте с больным туберкулезом без бактериовыделния (гр.IVА),3_Взято в текущем году,05,2016,1329.0


### Таблица 2500

#### Создание функции для таблицы 2500

In [44]:
def process_2500_to_long(df_2500, year):
    """
    Обрабатывает DataFrame формы 2500 и возвращает длинную таблицу
    
    Parameters:
    -----------
    df_2500 : pandas.DataFrame
        DataFrame с данными формы 2500
    year : int
        Год данных
    
    Returns:
    --------
    pandas.DataFrame
        Длинная таблица с колонками: группы больных, форма_заболевания, Строка, Год, Значение
    """
    
    # Создаем копию
    df = df_2500.copy()
    
    # Заполняем конкретные ячейки
    if len(df.columns) > 6:
        df.iloc[1, 6] = 'состоящих на учете'
    if len(df.columns) > 7:
        df.iloc[1, 7] = 'состоящих на учете'
    if len(df.columns) > 8:
        df.iloc[1, 8] = 'состоящих на учете'
    
    # Удаляем лишние строки и столбцы
    df = df.iloc[:9, :16]
    
    # Склеиваем три строки в один заголовок (через пробел)
    df.columns = (df.iloc[0].fillna('').astype(str) + ' ' + 
                  df.iloc[1].fillna('').astype(str) + ' ' + 
                  df.iloc[2].fillna('').astype(str)).str.strip()
    
    
    # Берём строку с графой, заполняем NaN пустыми строками, преобразуем в строки
    prefix = df.iloc[3].fillna('').astype(str)
    # Добавляем префикс к текущим именам столбцов (например, через пробел)
    new_columns = prefix + '_' + df.columns.astype(str)
    # Очищаем от лишних пробелов
    new_columns = new_columns.str.strip()
    # Присваиваем новые имена столбцам
    df.columns = new_columns
    
    # Удаляем эти три строки
    df = df.iloc[4:].reset_index(drop=True)
    
    # Переименовываем столбцы
    df = df.rename(columns={'2_№': 'Строка'})
    df['Строка'] = pd.to_numeric(df['Строка'], errors='coerce').fillna(0).astype(int)
    df = df.rename(columns={'1_Группы больных': 'форма_заболевания'})
    
    df_clean = df
    
    # Преобразуем из широкого формата в длинный
    df_long = df_clean.melt(
        id_vars=['форма_заболевания', 'Строка'],
        var_name='группы больных',
        value_name='Значение'
    )
    
    # Добавляем колонку с годом
    df_long['Год'] = year
    
    # Удаляем лишние пробелы
    df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)
    
    # Переставляем колонки в нужном порядке
    df_long = df_long[['группы больных', 'форма_заболевания', 'Строка', 'Год', 'Значение']]
    
    return df_long


# Пример использования:
# df_2017_2500 = exdfs_dict_2017['(2500)']
# df_2017_2500_long = process_2500_to_long(df_2017_2500, 2017)

#### Обработка всех таблиц 2500 за 2017-2024 и приведение к длинному формату

In [45]:
# Список годов
years = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]  

# Список для хранения имен созданных датафреймов
created_dfs_2500 = []

# Цикл по всем годам
for year in years:
    dict_name = f'exdfs_dict_{year}'
    df_name = f'df_{year}_2500'
    df_long_name = f'df_{year}_2500_long'
    
    # Достаем таблицу 2500 из словаря
    locals()[df_name] = locals()[dict_name]['(2500)']
    
    # Обрабатываем и приводим к длинному формату
    locals()[df_long_name] = process_2500_to_long(locals()[df_name], year)
    
    # Добавляем в список созданных датафреймов
    created_dfs_2500.append(df_long_name)
    
    print(f'\n=== {year} год ===')
    print(locals()[df_long_name].head())

# Выводим список всех созданных датафреймов
print('\n=== Список созданных датафреймов для формы 2500 ===')
for df_name in created_dfs_2500:
    print(f'- {df_name}')

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\88083996.py:39: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prefix = df.iloc[3].fillna('').astype(str)



=== 2017 год ===
                                                             группы больных  \
0  3_Обнаружено из числа больных с впервые в жизни уста-новленным диагнозом   
1  3_Обнаружено из числа больных с впервые в жизни уста-новленным диагнозом   
2  3_Обнаружено из числа больных с впервые в жизни уста-новленным диагнозом   
3  3_Обнаружено из числа больных с впервые в жизни уста-новленным диагнозом   
4  3_Обнаружено из числа больных с впервые в жизни уста-новленным диагнозом   

                                форма_заболевания  Строка   Год  Значение  
0                      Туберкулез органов дыхания       1  2017     960.0  
1                 Обследовано на МЛУ (из стр. 01)       2  2017     707.0  
2                             из них выявлена МЛУ       3  2017     171.0  
3              Туберкулез внелегочных локализаций       4  2017      11.0  
4  из них сельских жителей (из суммы строк 01+04)       5  2017     277.0  

=== 2018 год ===
                                 

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\88083996.py:68: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\88083996.py:39: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prefix = df.iloc[3].fillna('').astype(str)
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\88083996.py:68: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\88083996.py:39: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecat

#### Обработка таблицы 2500 за 2016 года. Файл имеет другой формат, поэтому обработаем вручную 

In [46]:
df_2016_2500 = exdfs_dict_2016['(2500)']
df_2016_2500.head()

,0,40,47,58,68,78,91,104,117,130,140,157,169,180,193,203,209
0,Группы больных,№ стр.,Обнаружено из числа больных,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Умерло,NaN,Перестало выделять МБТ,Выбыло из района обслуживания,Состоит на конец отчетного года,NaN
1,NaN,NaN,С впервые в жизни установленным диагнозом,из них методом посева,из них методом микроскопии,Состоящих на учёте,NaN,NaN,NaN,NaN,NaN,от туберкулеза,от других причин,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,в I группе,из них в IБ группе,в II группе,в III группе,Снятых ранее с учета,"Переведено из других учреждений больных, выделяющих МБТ",NaN,NaN,NaN,NaN,NaN,NaN
3,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,NaN
4,Туберкулёз органов дыхания,01,1027,337,707,253,57,155,44,103,383,248,390,1441,296,2601,NaN


In [47]:
 # Создаем копию
df = df_2016_2500.copy()
    # Заполняем пропущенные поля в заголовках по горизонтали
# df.iloc[0] = df.iloc[0].ffill()
#df.iloc[1] = df.iloc[1].ffill()
df.iloc[1, 6] = 'состоящих на учете' 
df.iloc[1, 7] = 'состоящих на учете' 
df.iloc[1, 8] = 'состоящих на учете' 
    # Удаляем лишние строки и столбцы
df = df.iloc[:9, :16]
# Склеиваем три строки в один заголовок (через пробел)
df.columns = (df.iloc[0].fillna('') + ' ' + df.iloc[1].fillna('') + ' ' + df.iloc[2].fillna('')).str.strip()

# Берём строку с графой, заполняем NaN пустыми строками, преобразуем в строки
prefix = df.iloc[3].fillna('').astype(str)
# Добавляем префикс к текущим именам столбцов (например, через пробел)
new_columns = prefix + '_' + df.columns.astype(str)
# Очищаем от лишних пробелов
new_columns = new_columns.str.strip()
# Присваиваем новые имена столбцам
df.columns = new_columns

# Удаляем эти три строки
df = df.iloc[4:].reset_index(drop=True)
df = df.rename(columns={'2_№ стр.': 'Строка'})
df['Строка'] = df['Строка'].astype(int)
df = df.rename(columns={'1_Группы больных': 'форма_заболевания'})

df_2016_2500_clean = df
df_2016_2500_clean.head()

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\2875589472.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prefix = df.iloc[3].fillna('').astype(str)


,форма_заболевания,Строка,3_Обнаружено из числа больных С впервые в жизни установленным диагнозом,4_из них методом посева,5_из них методом микроскопии,6_Состоящих на учёте в I группе,7_состоящих на учете из них в IБ группе,8_состоящих на учете в II группе,9_состоящих на учете в III группе,10_Снятых ранее с учета,"11_Переведено из других учреждений больных, выделяющих МБТ",12_Умерло от туберкулеза,13_от других причин,14_Перестало выделять МБТ,15_Выбыло из района обслуживания,16_Состоит на конец отчетного года
0,Туберкулёз органов дыхания,1,1027,337,707,253,57,155,44,103,383,248,390,1441,296,2601
1,Обследовано на МЛУ (из стр.01),2,1019,318,701,253,56,152,44,100,383,230,388,1414,291,2596
2,из них выявлена МЛУ,3,167,154,13,140,32,108,17,34,173,96,154,302,107,1076
3,Туберкулез внелегочных локализаций,4,9,6,3,2,1,3,NaN,NaN,2,NaN,1,10,4,30
4,из них сельских жителей (из суммы строк 01+04),5,296,97,199,50,20,37,9,24,67,89,88,389,111,632


In [48]:
    # Преобразуем из широкого формата в длинный
df_long = df_2016_2500_clean.melt(
    id_vars=['форма_заболевания', 'Строка'],
    var_name='группы больных',
    value_name='Значение'
)
    
    # Добавляем колонку с годом
df_long['Год'] = 2016
    
# Удаляем лишние пробелы
df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)
   

    # Переставляем колонки в нужном порядке
df_2016_2500_long = df_long[['группы больных', 'форма_заболевания', 'Строка', 'Год', 'Значение']]
df_2016_2500_long.head()

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\21783814.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)


,группы больных,форма_заболевания,Строка,Год,Значение
0,3_Обнаружено из числа больных С впервые в жизни установленным диагнозом,Туберкулёз органов дыхания,1,2016,1027.0
1,3_Обнаружено из числа больных С впервые в жизни установленным диагнозом,Обследовано на МЛУ (из стр.01),2,2016,1019.0
2,3_Обнаружено из числа больных С впервые в жизни установленным диагнозом,из них выявлена МЛУ,3,2016,167.0
3,3_Обнаружено из числа больных С впервые в жизни установленным диагнозом,Туберкулез внелегочных локализаций,4,2016,9.0
4,3_Обнаружено из числа больных С впервые в жизни установленным диагнозом,из них сельских жителей (из суммы строк 01+04),5,2016,296.0


### Таблица 2600

#### Создание функции для таблицы 2600

In [49]:
def process_2600_to_long(df_2600, year):
    """
    Обрабатывает DataFrame формы 2600 и возвращает длинную таблицу
    
    Parameters:
    -----------
    df_2600 : pandas.DataFrame
        DataFrame с данными формы 2600
    year : int
        Год данных
    
    Returns:
    --------
    pandas.DataFrame
        Длинная таблица с колонками: Вид помощи, группы больных, Строка, Год, Значение
    """
    
    # Создаем копию
    df = df_2600.copy()
    
    # Удаляем лишние строки и столбцы
    df = df.iloc[:18, :10]
    df = df.dropna(axis=1, how='all')
    
    # Объединяем и заменяем исходные столбцы
    df[df.columns[0]] = df.iloc[:, 0].combine_first(df.iloc[:, 1])
    df = df.drop(columns=[df.columns[1]])
    
    # Заполняем конкретные ячейки
    if len(df.columns) > 2:
        df.iloc[0, 2] = 'Больных, состоящих на учете'
    if len(df.columns) > 5:
        df.iloc[0, 5] = 'Впервые установленным диагнозом'
    if len(df.columns) > 7:
        df.iloc[3, 7] = 'детей до 14 лет'
    if len(df.columns) > 3:
        df.iloc[2, 3] = ''
    
    # Удаляем столбец по индексу 6, если он существует
    if len(df.columns) > 6:
        df = df.drop(columns=[df.columns[6]])
    
    # Заполняем пропущенные поля в заголовках по горизонтали
    df.iloc[0] = df.iloc[0].ffill()
    
    # Объединяем верхние строки в заголовок (строки 0, 3, 2)
    df.columns = (df.iloc[0].fillna('').astype(str) + ' ' + 
                  df.iloc[3].fillna('').astype(str) + ' ' + 
                  df.iloc[2].fillna('').astype(str)).str.strip()

    # Берём строку с графой, заполняем NaN пустыми строками, преобразуем в строки
    prefix = df.iloc[5].fillna('').astype(str)
    # Добавляем префикс к текущим именам столбцов (например, через пробел)
    new_columns = prefix + '_' + df.columns.astype(str)
    # Очищаем от лишних пробелов
    new_columns = new_columns.str.strip()
    # Присваиваем новые имена столбцам
    df.columns = new_columns
  
    # Удаляем эти строки
    df = df.iloc[6:].reset_index(drop=True)
    
    # Переименовываем второй столбец (индекс 1)
    df.columns.values[1] = 'Строка'
    df.columns.values[0] = 'Вид помощи'

    df_clean = df
    
    # Преобразуем из широкого формата в длинный
    df_long = df_clean.melt(
        id_vars=['Вид помощи', 'Строка'],
        var_name='группы больных',
        value_name='Значение'
    )
    
    # Добавляем колонку с годом
    df_long['Год'] = year
    
    # Удаляем лишние пробелы
    df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)
    
    # Переставляем колонки в нужном порядке
    df_long = df_long[['Вид помощи', 'группы больных', 'Строка', 'Год', 'Значение']]
    
    return df_long


# Пример использования:
# df_2017_2600 = exdfs_dict_2017['(2600)']
# df_2017_2600_long = process_2600_to_long(df_2017_2600, 2017)

#### Обработка всех таблиц 2600 за 2017-2024 и приведение к длинному формату

In [50]:
# Список годов
years = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]  

# Список для хранения имен созданных датафреймов
created_dfs_2600 = []

# Цикл по всем годам
for year in years:
    dict_name = f'exdfs_dict_{year}'
    df_name = f'df_{year}_2600'
    df_long_name = f'df_{year}_2600_long'
    
    # Достаем таблицу 2600 из словаря
    locals()[df_name] = locals()[dict_name]['(2600)']
    
    # Обрабатываем и приводим к длинному формату
    locals()[df_long_name] = process_2600_to_long(locals()[df_name], year)
    
    # Добавляем в список созданных датафреймов
    created_dfs_2600.append(df_long_name)
    
    print(f'\n=== {year} год ===')
    print(locals()[df_long_name].head())

# Выводим список всех созданных датафреймов
print('\n=== Список созданных датафреймов для формы 2600 ===')
for df_name in created_dfs_2600:
    print(f'- {df_name}')

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\1099999654.py:52: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prefix = df.iloc[5].fillna('').astype(str)
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\1099999654.py:80: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\1099999654.py:52: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prefix = df.iloc[5].fillna('').astype(str)
C:\User


=== 2017 год ===
                                        Вид помощи  \
0                          Госпитализировано всего   
1                       из них бактериовыделителей   
2  в том числе в дневные стационары (из строки 01)   
3           в том числе в санатории (из строки 01)   
4   Применены хирургические методы лечения (всего)   

                        группы больных  Строка   Год  Значение  
0  3_Больных, состоящих на учете всего       1  2017    3967.0  
1  3_Больных, состоящих на учете всего       2  2017    1985.0  
2  3_Больных, состоящих на учете всего       3  2017     307.0  
3  3_Больных, состоящих на учете всего       4  2017      65.0  
4  3_Больных, состоящих на учете всего       5  2017     482.0  

=== 2018 год ===
                                        Вид помощи  \
0                          Госпитализировано всего   
1                       из них бактериовыделителей   
2  в том числе в дневные стационары (из строки 01)   
3           в том числе в санатор

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\1099999654.py:52: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prefix = df.iloc[5].fillna('').astype(str)
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\1099999654.py:80: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_long = df_long.applymap(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\1099999654.py:52: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prefix = df.iloc[5].fillna('').astype(str)
C:\User

#### Обработка таблицы 2600 за 2016 года. Файл имеет другой формат, поэтому обработаем вручную 

In [51]:
df_2016_2600 = exdfs_dict_2016['(2600)']
df_2016_2600.head()

,0,4,5,6,7,8,9,10,11,12
0,Вид помощи,NaN,№ строки,"Больных, состоящих на учете, всего",NaN,NaN,из них с впервые в жизни установленным диагнозом,NaN,NaN,NaN
1,NaN,NaN,NaN,всего,из них:,NaN,всего,из них:,NaN,NaN
2,NaN,NaN,NaN,NaN,детей до 14 лет,подростков 15-17 лет,NaN,детей до 14 лет,NaN,подростков 15-17 лет
3,1,NaN,2,3,4,5,6,6,7.0,8
4,Госпитализировано всего,NaN,1,4276,109,35,1842,NaN,93.0,31


In [52]:
# ======================================================================
# ЧАСТЬ 1: ОЧИСТКА И ПОДГОТОВКА ТАБЛИЦЫ 2600 ДЛЯ 2016 ГОДА
# ======================================================================

df = df_2016_2600.copy()

# Объединяем и заменяем исходные столбцы
df[df.columns[0]] = df.iloc[:, 0].combine_first(df.iloc[:, 1])
df = df.drop(columns=[df.columns[1]])

# Заполняем конкретные ячейки
if len(df.columns) > 2:
    df.iloc[0, 2] = 'Больных, состоящих на учете'
if len(df.columns) > 5:
    df.iloc[0, 5] = 'Впервые установленным диагнозом'
if len(df.columns) > 7:
    df.iloc[2, 7] = 'детей до 14 лет'
if len(df.columns) > 3:
    df.iloc[1, 3] = ''

# Удаляем столбец по индексу 6, если он существует
if len(df.columns) > 6:
    df = df.drop(columns=[df.columns[6]])

# Заполняем пропущенные поля в заголовках по горизонтали
df.iloc[0] = df.iloc[0].ffill()

# Объединяем верхние строки в заголовок (строки 0, 3, 2)
df.columns = (df.iloc[0].fillna('').astype(str) + ' ' + 
              df.iloc[1].fillna('').astype(str) + ' ' + 
              df.iloc[2].fillna('').astype(str)).str.strip()

# Берём строку с графой (строка 3), преобразуем числа в целые без .0
prefix = df.iloc[3].fillna('').astype(str)
# Убираем .0 из чисел
prefix = prefix.str.replace(r'\.0$', '', regex=True)

# Добавляем префикс к текущим именам столбцов
new_columns = prefix + '_' + df.columns.astype(str)
new_columns = new_columns.str.strip()

# Присваиваем новые имена столбцам
df.columns = new_columns

# Удаляем служебные строки
df = df.iloc[4:].reset_index(drop=True)

# Переименовываем первые два столбца
df.columns.values[1] = 'Строка'
df.columns.values[0] = 'Вид помощи'

df_clean = df
print("После очистки:")
display(df_clean.head())

После очистки:


C:\Users\urize\AppData\Local\Temp\ipykernel_15856\3234836919.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'детей до 14 лет' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.iloc[2, 7] = 'детей до 14 лет'
C:\Users\urize\AppData\Local\Temp\ipykernel_15856\3234836919.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prefix = df.iloc[3].fillna('').astype(str)


,Вид помощи,Строка,"3_Больных, состоящих на учете всего","4_Больных, состоящих на учете детей до 14 лет","5_Больных, состоящих на учете подростков 15-17 лет",6_Впервые установленным диагнозом всего,7_Впервые установленным диагнозом детей до 14 лет,8_Впервые установленным диагнозом подростков 15-17 лет
0,Госпитализировано всего,1,4276,109,35,1842,93.0,31
1,из них бактериовыделителей,2,2222,2,10,928,NaN,9
2,в том числе в дневные стационары (из строки 01),3,335,NaN,NaN,122,NaN,NaN
3,в том числе в санатории (из строки 01),4,88,1,NaN,12,NaN,NaN
4,Применены хирургические методы лечения (всего),5,480,6,4,349,4.0,3


In [53]:
# ======================================================================
# ЧАСТЬ 2: ПРИВЕДЕНИЕ К ДЛИННОМУ ФОРМАТУ
# ======================================================================

df_2016_2600_long = df.melt(
    id_vars=['Вид помощи', 'Строка'],
    var_name='группы больных',
    value_name='Значение'
)

# Добавляем колонку с годом
df_2016_2600_long['Год'] = 2016

# Удаляем лишние пробелы в строковых колонках
for col in ['Вид помощи', 'группы больных']:
    df_2016_2600_long[col] = df_2016_2600_long[col].apply(lambda x: ' '.join(x.split()) if isinstance(x, str) else x)

# Переставляем колонки в нужном порядке
df_2016_2600_long = df_2016_2600_long[['Вид помощи', 'группы больных', 'Строка', 'Год', 'Значение']]

print("\nПосле приведения к длинному формату:")
display(df_2016_2600_long.head())
print(f"\nВсего строк: {len(df_2016_2600_long)}")


После приведения к длинному формату:


,Вид помощи,группы больных,Строка,Год,Значение
0,Госпитализировано всего,"3_Больных, состоящих на учете всего",1,2016,4276
1,из них бактериовыделителей,"3_Больных, состоящих на учете всего",2,2016,2222
2,в том числе в дневные стационары (из строки 01),"3_Больных, состоящих на учете всего",3,2016,335
3,в том числе в санатории (из строки 01),"3_Больных, состоящих на учете всего",4,2016,88
4,Применены хирургические методы лечения (всего),"3_Больных, состоящих на учете всего",5,2016,480



Всего строк: 72


### Таблица 2513

#### Создание функции извлечения таблицы 2513 из docx файла

In [54]:

def extract_table_2513_from_docx(file_path):
    if not os.path.exists(file_path):
        print(f"Файл не найден: {file_path}")
        return None
    
    doc = Document(file_path)
    found = False
    table = None
    
    for element in doc.element.body:
        if element.tag.endswith('p') and not found:
            for para in doc.paragraphs:
                if para._element is element and '(2513)' in para.text:
                    found = True
                    break
        elif element.tag.endswith('tbl') and found:
            for t in doc.tables:
                if t._element is element:
                    table = t
                    break
            break
    
    if table is None:
        print(f"Таблица (2513) не найдена в {file_path}")
        return None
    
    data = []
    for row in table.rows:
        row_data = []
        for cell in row.cells:
            text = cell.text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
            text = ' '.join(text.split())
            row_data.append(text)
        data.append(row_data)
    
    # Заголовки из первых двух строк
    headers = []
    for i in range(max(len(data[0]), len(data[1]))):
        val1 = data[0][i] if i < len(data[0]) else ''
        val2 = data[1][i] if i < len(data[1]) else ''
        if val1 == val2 or not val1 or not val2:
            header = val1 if val1 else val2
        else:
            header = f"{val1} {val2}".strip()
        headers.append(header if header else f"Column_{i+1}")
    
    df = pd.DataFrame(data[2:], columns=headers)
    
    # Объединяем строки с пустым '№ строки' со следующей строкой
    col_name = '№ строки'
    if col_name in df.columns:
        new_rows = []
        i = 0
        while i < len(df):
            row = df.iloc[i].copy()
            is_empty = pd.isna(row[col_name]) or (isinstance(row[col_name], str) and row[col_name].strip() == '')
            if is_empty and i + 1 < len(df):
                next_row = df.iloc[i + 1]
                for col in df.columns:
                    curr = row[col]
                    nxt = next_row[col]
                    if pd.notna(curr) and str(curr).strip() != '':
                        if pd.notna(nxt) and str(nxt).strip() != '':
                            row[col] = f"{curr} {nxt}"
                    else:
                        row[col] = nxt
                new_rows.append(row)
                i += 2
            else:
                new_rows.append(row)
                i += 1
        df = pd.DataFrame(new_rows, columns=df.columns)
    
    print(f"Загружена {os.path.basename(file_path)}: {df.shape}")
    return df

# Загружаем и сохраняем все годы
for year in range(2016, 2025):
    df = extract_table_2513_from_docx(f"ИО 30 {year}.docx")
    if df is not None:
        df.to_csv(f"df_{year}_2513.csv", index=False, encoding='utf-8-sig')
        globals()[f"df_{year}_2513"] = df

print("Готово!")

Загружена ИО 30 2016.docx: (10, 6)
Загружена ИО 30 2017.docx: (10, 6)
Загружена ИО 30 2018.docx: (10, 6)
Загружена ИО 30 2019.docx: (10, 6)
Загружена ИО 30 2020.docx: (10, 6)
Загружена ИО 30 2021.docx: (10, 6)
Загружена ИО 30 2022.docx: (10, 6)
Загружена ИО 30 2023.docx: (10, 6)
Загружена ИО 30 2024.docx: (10, 6)
Готово!


In [55]:
# Сохранение всех датасетов в текущую директорию
#for year in range(2016, 2025):
#    df = dfs_2513.get(f"df_{year}_2513")
#    if df is not None:
#        df.to_csv(f"df_{year}_2513.csv", index=False, encoding='utf-8-sig')
#        print(f"Сохранён: df_{year}_2513.csv")

In [56]:
 # Создаем копию
df = df_2016_2513.copy()


# Берём строку с графой, заполняем NaN пустыми строками, преобразуем в строки
prefix = df.iloc[0].fillna('').astype(str)
# Добавляем префикс к текущим именам столбцов (например, через пробел)
new_columns = prefix + '_' + df.columns.astype(str)
# Очищаем от лишних пробелов
new_columns = new_columns.str.strip()
# Присваиваем новые имена столбцам
df.columns = new_columns

# Удаляем эти 2 строки
df = df.iloc[4:].reset_index(drop=True)
# переименовать столбец 
df.columns.values[1] = 'Строка'
df.columns.values[0] = 'Профилактические осмотры на туберкулез'

#df['Строка'] = df['Строка'].astype(int)
df_2016_2513_clean = df
df_2016_2513_clean.head()

,Профилактические осмотры на туберкулез,Строка,3_Всего,4_из них сельских жителей,5_Выявлен туберкулез Всего,6_Выявлен туберкулез из них: у сельских жителей
0,15-17 лет включительно,1.3,77945,17127,44,7
1,Из числа осмотренных (стр.1) обследовано: флюорографически,2,1411232,275966,1523,415
2,бактериоскопически,3,11666,2353,40,9
3,Из числа осмотренных детей (стр. 1.1+1.2+1.3) проведены: иммунодиагностика с применением аллергена бактерий с 2 туберкулиновыми единицами очищенного туберкулина в стандартном разведении,4,455644,111134,83,24
4,иммунодиагностика с применением аллергена туберкулезного рекомбинантного в стандартном разведении,5,,,,


#### Обработка всех таблиц 2513 за 2016-2024 и приведение к длинному формату

In [57]:
def process_2513_to_long(df, year):
    """
    Обрабатывает DataFrame формы 2513 и возвращает длинную таблицу
    """
    df = df.copy()
    
    # Берём первую строку как префикс для всех колонок
    prefix = df.iloc[0].fillna('').astype(str)
    new_columns = prefix + '_' + df.columns.astype(str)
    new_columns = new_columns.str.strip()
    df.columns = new_columns
    
    # Удаляем первые 1
    df = df.iloc[1:].reset_index(drop=True)
    
    # Переименовываем первые две колонки
    df.columns.values[0] = 'Профилактические осмотры на туберкулез'
    df.columns.values[1] = 'Строка'
    
    # Преобразуем из широкого в длинный
    id_vars = ['Профилактические осмотры на туберкулез', 'Строка']
    value_vars = [col for col in df.columns if col not in id_vars]
    
    df_long = df.melt(
        id_vars=id_vars,
        value_vars=value_vars,
        var_name='Категория',
        value_name='Значение'
    )
    
    df_long['Год'] = year
    
    # Очистка строк
    for col in ['Профилактические осмотры на туберкулез', 'Строка', 'Категория']:
        df_long[col] = df_long[col].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
    
    df_long['Значение'] = pd.to_numeric(df_long['Значение'], errors='coerce')
    
    # Перестановка колонок
    df_long = df_long[['Профилактические осмотры на туберкулез', 'Категория', 'Год', 'Значение', 'Строка']]
    
    return df_long

In [58]:
# Загружаем и преобразуем все таблицы 2513
for year in range(2016, 2025):
    df_orig = globals().get(f"df_{year}_2513")  # если переменные уже созданы
    if df_orig is not None:
        df_long = process_2513_to_long(df_orig, year)
        # Сохраняем
        #df_long.to_csv(f"df_{year}_2513_long.csv", index=False, encoding='utf-8-sig')
        # Создаём переменную в глобальной области (по желанию)
        globals()[f"df_{year}_2513_long"] = df_long
        print(f"Преобразована df_{year}_2513_long: {df_long.shape}")

Преобразована df_2016_2513_long: (36, 5)
Преобразована df_2017_2513_long: (36, 5)
Преобразована df_2018_2513_long: (36, 5)
Преобразована df_2019_2513_long: (36, 5)
Преобразована df_2020_2513_long: (36, 5)
Преобразована df_2021_2513_long: (36, 5)
Преобразована df_2022_2513_long: (36, 5)
Преобразована df_2023_2513_long: (36, 5)
Преобразована df_2024_2513_long: (36, 5)


### Таблицы 1100 и 3100 из формы 30

In [59]:
# Путь к файлу
file_path = r"C:\Users\urize\Ирткутск_туберкулез\from form 30.xlsx"

# Загружаем первый лист в df_1100
df_1100 = pd.read_excel(file_path, sheet_name=0)  # или sheet_name="1100", если лист так называется
print(f"Лист 1 (1100): {df_1100.shape[0]} строк, {df_1100.shape[1]} столбцов")
print(f"Колонки: {df_1100.columns.tolist()}")
print("\nПервые 3 строки:")
display(df_1100.head(3))

# Загружаем второй лист в df_3100
df_3100 = pd.read_excel(file_path, sheet_name=1)  # или sheet_name="3100"
print(f"\nЛист 2 (3100): {df_3100.shape[0]} строк, {df_3100.shape[1]} столбцов")
print(f"Колонки: {df_3100.columns.tolist()}")
print("\nПервые 3 строки:")
display(df_3100.head(3))

Лист 1 (1100): 18 строк, 18 столбцов
Колонки: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]

Первые 3 строки:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,2016,фтизиатры,111,209.00,197.75,123.75,117.75,85.25,80,119,65,54,33,11.0,2.0,118,NaN,NaN
1,2016,из них фтизиатры участковые,112,75.00,73.25,75.00,73.25,X,X,36,36,X,7,8.0,NaN,36,NaN,NaN
2,2017,фтизиатры,111,246.25,229.25,126.25,117.75,120,111.5,132,62,70,39,14.0,2.0,131,NaN,NaN



Лист 2 (3100): 18 строк, 10 столбцов
Колонки: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

Первые 3 строки:


,0,1,2,3,4,5,6,7,8,9
0,2016,туберкулезные для взрослых,57,1413,188.0,1405,5927,2037,13,862.0
1,2016,туберкулезные для детей,58,150,NaN,150,527,197,527,NaN
2,2017,туберкулезные для взрослых,57,1363,185.0,1356,5551,2010,8,1141.0


In [60]:
# Обновляем исходный df_3100
# Заменяем строки для 2018 года (взрослые)
mask_adult_2018 = (df_3100[0] == 2018) & (df_3100[1] == 'туберкулезные для взрослых')
df_3100.loc[mask_adult_2018, [3,4,5,6,7,8,9]] = [1333, 214, 1334, 5544, 2008, 18, 1144]

# Заменяем строки для 2018 года (дети)
mask_child_2018 = (df_3100[0] == 2018) & (df_3100[1] == 'туберкулезные для детей')
df_3100.loc[mask_child_2018, [3,4,5,6,7,8,9]] = [105, None, 143, 352, 158, 352, None]

print("Исправленные данные для 2018 года:")
print(df_3100[df_3100[0] == 2018])

Исправленные данные для 2018 года:
      0                           1   2     3      4     5     6     7    8  \
4  2018  туберкулезные для взрослых  57  1333  214.0  1334  5544  2008   18   
5  2018     туберкулезные для детей  58   105    NaN   143   352   158  352   

        9  
4  1144.0  
5     NaN  


In [61]:
# Приводим df_1100 в длинный формат
df_all_1100 = pd.melt(
    df_1100,
    id_vars=[0, 1, 2],
    var_name='Графа',
    value_name='Значение'
)

# Приводим df_3100 в длинный формат
df_all_3100 = pd.melt(
    df_3100,
    id_vars=[0, 1, 2],
    var_name='Графа',
    value_name='Значение'
)

# Переименовываем колонки
df_all_1100 = df_all_1100.rename(columns={0: 'Год', 1: 'Должность', 2: 'Строка'})
df_all_3100 = df_all_3100.rename(columns={0: 'Год', 1: 'Тип_коек', 2: 'Строка'})

# Приводим к числовым типам (нечисловое -> NaN)
df_all_1100['Год'] = pd.to_numeric(df_all_1100['Год'], errors='coerce')
df_all_1100['Строка'] = pd.to_numeric(df_all_1100['Строка'], errors='coerce')
df_all_1100['Графа'] = pd.to_numeric(df_all_1100['Графа'], errors='coerce')
df_all_1100['Значение'] = pd.to_numeric(df_all_1100['Значение'], errors='coerce')

df_all_3100['Год'] = pd.to_numeric(df_all_3100['Год'], errors='coerce')
df_all_3100['Строка'] = pd.to_numeric(df_all_3100['Строка'], errors='coerce')
df_all_3100['Графа'] = pd.to_numeric(df_all_3100['Графа'], errors='coerce')
df_all_3100['Значение'] = pd.to_numeric(df_all_3100['Значение'], errors='coerce')

print("Готово!")
print(f"df_all_1100: {df_all_1100.shape}")
print(f"df_all_3100: {df_all_3100.shape}")

Готово!
df_all_1100: (270, 5)
df_all_3100: (126, 5)


In [62]:
df_all_1100.head()

,Год,Должность,Строка,Графа,Значение
0,2016,фтизиатры,111,3,209.00
1,2016,из них фтизиатры участковые,112,3,75.00
2,2017,фтизиатры,111,3,246.25
3,2017,из них фтизиатры участковые,112,3,78.25
4,2018,фтизиатры,111,3,230.25


In [63]:
df_all_3100.head()

,Год,Тип_коек,Строка,Графа,Значение
0,2016,туберкулезные для взрослых,57,3,1413.0
1,2016,туберкулезные для детей,58,3,150.0
2,2017,туберкулезные для взрослых,57,3,1363.0
3,2017,туберкулезные для детей,58,3,150.0
4,2018,туберкулезные для взрослых,57,3,1333.0


### 1.3. Сведение таблиц по годам в единую таблицу. 

#### Создание функции snake_case для полей датасетов


In [64]:
def to_snake_case(df):
    """
    Преобразует названия столбцов DataFrame в snake_case,
    удаляя начальный числовой префикс (например, '5_', '12_').
    """
    df = df.copy()
    new_columns = []
    for col in df.columns:
        name = str(col)
        # удаляем префикс "цифры_"
        name = re.sub(r'^\d+_', '', name)
        # заменяем пробелы и дефисы на подчеркивания
        name = name.replace(' ', '_').replace('-', '_')
        name = name.replace('№', 'номер')
        # убираем множественные подчеркивания
        name = re.sub(r'_+', '_', name)
        name = name.lower().strip('_')
        new_columns.append(name)
    df.columns = new_columns
    return df

#### Создание функции экстракции префикса в поле графа

In [65]:
def extract_numeric_prefix_from_dataset(df, sep='_', new_col='Графа'):
    """
    Автоматически находит первую колонку с типом object (строки),
    в которой встречается шаблон 'цифры_текст', извлекает числовой префикс в новую колонку.
    ПРЕФИКС УДАЛЯЕТСЯ из исходной колонки.
    """
    df = df.copy()
    
    # Ищем подходящую колонку
    target_col = None
    for col in df.select_dtypes(include='object').columns:
        sample = df[col].dropna()
        if len(sample) == 0:
            continue
        sample = sample.astype(str)
        # Проверяем, есть ли значения с цифрами и разделителем
        if sample.str.match(r'^\d+_', na=False).any():
            target_col = col
            print(f"Выбрана колонка для извлечения префикса: '{target_col}'")
            break
    
    if target_col is None:
        print(f"Предупреждение: не найдена колонка с шаблоном 'число{sep}текст'")
        df[new_col] = None
        return df
    
    # Извлекаем префикс и остаток
    split_df = df[target_col].astype(str).str.split(sep, n=1, expand=True)
    mask = split_df[0].str.isdigit()
    
    # Создаём новую колонку с числовым префиксом
    df[new_col] = pd.to_numeric(split_df[0], errors='coerce')
    
    # УДАЛЯЕМ префикс из исходной колонки (оставляем только текст после разделителя)
    df[target_col] = df[target_col].astype(str)
    df.loc[mask, target_col] = split_df[1].loc[mask]
    
    # Очищаем от лишних пробелов
    df[target_col] = df[target_col].str.strip()
    
    # Заменяем пустые строки на None
    df[target_col] = df[target_col].replace('', None)
    
    print(f"Из колонки '{target_col}' извлечено {df[new_col].notna().sum()} числовых префиксов")
    
    return df

#### Сведение 2800 +

In [66]:
# Список всех датафреймов 2800 за разные годы
df_list = [
    df_2016_2800_long,
    df_2017_2800_long,
    df_2018_2800_long,
    df_2019_2800_long,
    df_2020_2800_long,
    df_2021_2800_long,
    df_2022_2800_long,
    df_2023_2800_long,
    df_2024_2800_long
]

# Объединяем все таблицы в одну
df_all_2800 = pd.concat(df_list, ignore_index=True)

# Меняем значения 0 на nan
df_all_2800 = df_all_2800.replace(0, np.nan)
#Преобразуем названия столбцов DataFrame в стиль snake_case
#df_all_2800 = to_snake_case(df_all_2800)
# Проверяем результат
print(f"Объединено строк: {len(df_all_2800)}")
print(f"Колонки: {list(df_all_2800.columns)}")

# Берем префикс и переносим значение в поле Графа
df_all_2800 = extract_numeric_prefix_from_dataset(df_all_2800)


df_all_2800.head()

Объединено строк: 162
Колонки: ['Показатель', 'Возраст', '2_№ строки', 'Год', 'Значение']
Выбрана колонка для извлечения префикса: 'Возраст'
Из колонки 'Возраст' извлечено 162 числовых префиксов


C:\Users\urize\AppData\Local\Temp\ipykernel_15856\1028202725.py:18: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_all_2800 = df_all_2800.replace(0, np.nan)


,Показатель,Возраст,2_№ строки,Год,Значение,Графа
0,из них с МБТ+,Всего,2,2016,6.0,3
1,Наблюдалось в отчетном году,Всего,3,2016,7.0,3
2,Лечились в стационаре,Всего,4,2016,4.0,3
3,Лечились амбулаторно,Всего,5,2016,3.0,3
4,Умерло от туберкулеза всего,Всего,6,2016,NaN,3


In [67]:
df_all_2800 = df_all_2800.rename(columns={
    '2_№ строки': 'Строка',
})[['Показатель', 'Возраст', 'Год', 'Значение', 'Строка', 'Графа']]


In [68]:
df_all_2800.columns

Index(['Показатель', 'Возраст', 'Год', 'Значение', 'Строка', 'Графа'], dtype='object')

In [69]:
df_all_2800['Строка'] = pd.to_numeric(df_all_2800['Строка'], errors='coerce')


In [70]:
replace_dict = {
    'из них с МБТ+': 'Выявлено в текущем году, из них с МБТ+',
    'из них в стационаре': 'Умерло от туберкулеза всего, из них в стационаре'
}

df_all_2800['Показатель'] = df_all_2800['Показатель'].replace(replace_dict)

In [71]:
df_all_2800['Показатель'].unique()

array(['Выявлено в текущем году, из них с МБТ+',
       'Наблюдалось в отчетном году', 'Лечились в стационаре',
       'Лечились амбулаторно', 'Умерло от туберкулеза всего',
       'Умерло от туберкулеза всего, из них в стационаре'], dtype=object)

In [72]:
replace_dict = {
    'из них: детей            0-14 лет': 'Детей от 0 до 14 лет',
    'из них: подростков 15-17 лет': 'Подростков 15-17 лет'
}

df_all_2800['Возраст'] = df_all_2800['Возраст'].replace(replace_dict)

In [73]:
df_all_2800['Возраст'].unique()

array(['Всего', 'Детей от 0 до 14 лет', 'Подростков 15-17 лет'],
      dtype=object)

In [74]:
df_all_2800 = df_all_2800.drop_duplicates()

In [75]:
df_all_2800.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 162 entries, 0 to 161
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Показатель  162 non-null    object 
 1   Возраст     162 non-null    object 
 2   Год         162 non-null    int64  
 3   Значение    34 non-null     float64
 4   Строка      162 non-null    int64  
 5   Графа       162 non-null    int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 7.7+ KB


#### Сведение  1000 +

In [76]:
# Список всех датафреймов 1000 за разные годы
df_list_1000 = [
    df_2016_1000_long,
    df_2017_1000_long,
    df_2018_1000_long,
    df_2019_1000_long,
    df_2020_1000_long,
    df_2021_1000_long,
    df_2022_1000_long,
    df_2023_1000_long,
    df_2024_1000_long
]

# Объединяем все таблицы в одну
df_all_1000 = pd.concat(df_list_1000, ignore_index=True)
#Преобразуем названия столбцов DataFrame в стиль snake_case
#df_all_1000 = to_snake_case(df_all_1000)
# Проверяем результат
print(f"Объединено строк: {len(df_all_1000)}")
print(f"Колонки: {list(df_all_1000.columns)}")

# Берем префикс и переносим значение в поле Графа
df_all_1000 = extract_numeric_prefix_from_dataset(df_all_1000)

df_all_1000.head()

Объединено строк: 3762
Колонки: ['Показатель', '2_Пол', 'Возраст', '3_№ строки', '4_Код по МКБ X пересмотра', 'Год', 'Значение', 'Возраст_long']
Выбрана колонка для извлечения префикса: 'Возраст'
Из колонки 'Возраст' извлечено 3762 числовых префиксов


,Показатель,2_Пол,Возраст,3_№ строки,4_Код по МКБ X пересмотра,Год,Значение,Возраст_long,Графа
0,Заболело туберкулезом - всего,М,ВСЕГО,1,A15 - A19,2016,1699.0,5_Число больных с впервые в жизни установленным диагнозом активного туберкулеза ВСЕГО,5
1,Заболело туберкулезом - всего,Ж,ВСЕГО,2,NaN,2016,916.0,5_Число больных с впервые в жизни установленным диагнозом активного туберкулеза ВСЕГО,5
2,"из них МБТ+, определяемый любым методом",М,ВСЕГО,3,A15; A17 - A19 часть,2016,731.0,5_Число больных с впервые в жизни установленным диагнозом активного туберкулеза ВСЕГО,5
3,"из них МБТ+, определяемый любым методом",Ж,ВСЕГО,4,NaN,2016,362.0,5_Число больных с впервые в жизни установленным диагнозом активного туберкулеза ВСЕГО,5
4,"Из числа больных всего (стр.01,02) – число больных туберкулезом органов дыхания",М,ВСЕГО,5,A15; A16; A19 часть,2016,1634.0,5_Число больных с впервые в жизни установленным диагнозом активного туберкулеза ВСЕГО,5


In [77]:
df_all_1000.columns

Index(['Показатель', '2_Пол', 'Возраст', '3_№ строки',
       '4_Код по МКБ X пересмотра', 'Год', 'Значение', 'Возраст_long',
       'Графа'],
      dtype='object')

In [78]:
# Сначала проверьте текущие названия колонок
print(df_all_1000.columns.tolist())

# Исправленный вариант (с учетом пробела в 'МКБ X')
df_all_1000 = df_all_1000.rename(columns={
    'Показатель': 'Формы туберкулеза',
    '3_№ строки': 'Строка',
    '2_Пол': 'Пол',
    '4_Код по МКБ X пересмотра': 'Код по МКБ-X пересмотра'
})[['Формы туберкулеза', 'Код по МКБ-X пересмотра', 'Пол', 'Возраст', 'Год', 'Значение', 'Строка', 'Графа']]


['Показатель', '2_Пол', 'Возраст', '3_№ строки', '4_Код по МКБ X пересмотра', 'Год', 'Значение', 'Возраст_long', 'Графа']


In [79]:
df_all_1000['Строка'] = pd.to_numeric(df_all_1000['Строка'], errors='coerce')


In [80]:
replace_dict = {
    'ВСЕГО': 'всего',
    '0-4 года': '0-4',
    '5-6 лет': '5-6',
    '7-14 лет': '7-14',
    '15-17 лет': '15-17',
    '18-24 года': '18-24',
    '25-34 года': '25-34',
    '35-44 года': '35-44',
    '45-54 года': '45-54',
    '55-64 года': '55-64',
    '65 лет и более': '65+'
}

df_all_1000['Возраст'] = df_all_1000['Возраст'].replace(replace_dict)

In [81]:
print(df_all_1000['Возраст'].unique())


['всего' '0-4' '5-6' '7-14' '15-17' '18-24' '25-34' '35-44' '45-54'
 '55-64' '65+']


Видим дублирующиеся значения, относящиеся к разным категориям: строка 3 и 4 к новым заболеваниям, строки 37 и 38 к рецидивам. Делаем префикс для последних

In [82]:
def add_prefix(row):
    if row['Строка'] in [37, 38]:
        return 'Больные с рецидивом, ' + row['Формы туберкулеза']
    else:
        return row['Формы туберкулеза']

df_all_1000['Формы туберкулеза'] = df_all_1000.apply(add_prefix, axis=1)

In [83]:
# Проверяем уникальные значения после изменений
print("Уникальные значения в 'Формы туберкулеза' после изменений:")
print(df_all_1000['Формы туберкулеза'].unique())

Уникальные значения в 'Формы туберкулеза' после изменений:
['Заболело туберкулезом - всего' 'из них МБТ+, определяемый любым методом'
 'Из числа больных всего (стр.01,02) – число больных туберкулезом органов дыхания'
 'Из числа больных туберкулезом органов дыхания больные туберкулезом легких'
 'в том числе МБТ+ только культуральным методом, независимо от результатов микроскопии'
 'МБТ+ методом бактериоскопии, независимо от результатов посева'
 'Фиброзно-кавернозный туберкулез'
 'Из числа больных всего (стр.01,02) – число больных туберкулезом внелегочных локализаций'
 'из них: мозговых оболочек и ЦНС' 'костей и суставов'
 'мочеполовых органов' 'в т.ч.: женских половых органов'
 'периферических лимфатических узлов'
 'Из числа больных (стр.01,02) – сельских жителей'
 'Из общего числа больных (стр.01,02) иностранных жителей'
 'Из общего числа больных (стр.01,02) больные в подразделениях УИН'
 'Из общего числа больных (стр.01,02) лица БОМЖ'
 'Из общего числа больных (стр.01,02) диагностиров

In [84]:
def fix_punctuation(text):
    if pd.isna(text):
        return text
    s = str(text)
    
    # 1. Длинное тире → короткое
    s = s.replace('–', '-')
    
    # 2. Убираем пробелы вокруг дефиса/тире
    s = re.sub(r'\s*-\s*', '-', s)
    
    # 3. Убираем пробелы внутри скобок (например (стр. 01, 02) -> (стр.01,02))
    s = re.sub(r'\(\s*', '(', s)
    s = re.sub(r'\s*\)', ')', s)
    s = re.sub(r'\(\s*([^)]+?)\s*\)', r'(\1)', s)
    
    # 4. Убираем пробелы перед запятой, точкой, двоеточием, точкой с запятой
    s = re.sub(r'\s+([,.:;])', r'\1', s)
    
    # 5. Убираем пробелы после открывающей кавычки или скобки (если есть)
    s = re.sub(r'([\(\[])\s+', r'\1', s)
    
    # 6. Множественные пробелы → один пробел
    s = re.sub(r'\s+', ' ', s)
    
    # 7. Убираем пробелы в начале и конце
    s = s.strip()
    
    return s

# Применяем
df_all_1000['Формы туберкулеза'] = df_all_1000['Формы туберкулеза'].apply(fix_punctuation)

# Проверяем уникальные значения после правки пунктуации
df_all_1000['Формы туберкулеза'].unique()

array(['Заболело туберкулезом-всего',
       'из них МБТ+, определяемый любым методом',
       'Из числа больных всего (стр.01,02)-число больных туберкулезом органов дыхания',
       'Из числа больных туберкулезом органов дыхания больные туберкулезом легких',
       'в том числе МБТ+ только культуральным методом, независимо от результатов микроскопии',
       'МБТ+ методом бактериоскопии, независимо от результатов посева',
       'Фиброзно-кавернозный туберкулез',
       'Из числа больных всего (стр.01,02)-число больных туберкулезом внелегочных локализаций',
       'из них: мозговых оболочек и ЦНС', 'костей и суставов',
       'мочеполовых органов', 'в т.ч.: женских половых органов',
       'периферических лимфатических узлов',
       'Из числа больных (стр.01,02)-сельских жителей',
       'Из общего числа больных (стр.01,02) иностранных жителей',
       'Из общего числа больных (стр.01,02) больные в подразделениях УИН',
       'Из общего числа больных (стр.01,02) лица БОМЖ',
       'И

In [85]:
# Целевой список разрешённых значений (включая NaN)
target_codes = [
    'A15 - A19',
    'A15; A17 - A19 часть',
    'А17; А18; А19 часть',
    'A15 - A16',
    'A15; A16; A19 часть',
    'A18.1 часть',
    'A18.0',
    'A15.0',
    'A15.1-A15.2 часть',
    'A17',
    '45851',
    '45881',
    'A15.0 - А15.3; А16.0-А16.2',
    np.nan
]

# Словарь для приведения различных вариантов к каноническому виду
replace_dict = {
    # A15 - A19
    'A15 - A19': 'A15 - A19',
    'А15-А19': 'A15 - A19',
    'А15-А19': 'A15 - A19',  # кириллица

    # A15; A17 - A19 часть
    'A15;A17 - A19 часть': 'A15; A17 - A19 часть',
    'A15; A17 - A19 часть': 'A15; A17 - A19 часть',
    'А15; A17-А19часть': 'A15; A17 - A19 часть',

    # А17; А18; А19 часть
    'A17;A18; A19 часть': 'А17; А18; А19 часть',
    'А17; А18; А19 часть': 'А17; А18; А19 часть',

    # A15 - A16
    'A15 - A16': 'A15 - A16',
    'А15-А16': 'A15 - A16',

    # A15; A16; A19 часть
    'A15;A16;A19 часть': 'A15; A16; A19 часть',
    'A15; A16; A19 часть': 'A15; A16; A19 часть',

    # A18.1 часть
    'A18.1часть': 'A18.1 часть',
    'A18.1 часть': 'A18.1 часть',

    # A18.0
    'A18.0': 'A18.0',

    # A15.0
    'A15.0': 'A15.0',
    'А15.0': 'A15.0',

    # A15.1-A15.2 часть
    'A15.1-A15.2 часть': 'A15.1-A15.2 часть',
    'А15.1-А15.2 часть': 'A15.1-A15.2 часть',

    # A17
    'A17': 'A17',

    # 45851, 45881
    '45851': '45851',
    '45881': '45881',

    # A15.0 - А15.3; А16.0-А16.2
    'A15.0 - A15.3; A16.0 - A16.2': 'A15.0 - А15.3; А16.0-А16.2',
    'A15.0 - А15.3; А16.0-А16.2': 'A15.0 - А15.3; А16.0-А16.2',
}

# Применяем замену
df_all_1000['Код по МКБ-X пересмотра'] = df_all_1000['Код по МКБ-X пересмотра'].replace(replace_dict)

# Оставляем только строки, где значение входит в целевой список (или NaN)
df_all_1000 = df_all_1000[df_all_1000['Код по МКБ-X пересмотра'].isin(target_codes) | df_all_1000['Код по МКБ-X пересмотра'].isna()]

# Проверяем результат
print(df_all_1000['Код по МКБ-X пересмотра'].unique())

['A15 - A19' nan 'A15; A17 - A19 часть' 'A15; A16; A19 часть' 'A15 - A16'
 'A15.1-A15.2 часть' 'A15.0' 'A15.0 - А15.3; А16.0-А16.2'
 'А17; А18; А19 часть' 'A17' 'A18.0' 'A18.1 часть']


In [86]:
df_all_1000 = df_all_1000.drop_duplicates()

In [87]:
df_all_1000.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3564 entries, 0 to 3761
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Формы туберкулеза        3564 non-null   object 
 1   Код по МКБ-X пересмотра  1782 non-null   object 
 2   Пол                      3465 non-null   object 
 3   Возраст                  3564 non-null   object 
 4   Год                      3564 non-null   int64  
 5   Значение                 2235 non-null   float64
 6   Строка                   3564 non-null   int64  
 7   Графа                    3564 non-null   int64  
dtypes: float64(1), int64(3), object(4)
memory usage: 250.6+ KB


In [88]:
print(df_all_1000['Строка'].unique())
print(df_all_1000['Графа'].unique())
print(df_all_1000['Год'].unique())


[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 22 23 25 26
 27 28 29 30 31 32 33 34 35 36 37 38]
[ 5  6  7  8  9 10 11 12 13 14 15]
[2016 2017 2018 2019 2020 2021 2022 2023 2024]


#### Сведение 2700 +

In [89]:
# Список всех датафреймов 2700 за разные годы
df_list_2700 = [
    df_2016_2700_long,
    df_2017_2700_long,
    df_2018_2700_long,
    df_2019_2700_long,
    df_2020_2700_long,
    df_2021_2700_long,
    df_2022_2700_long,
    df_2023_2700_long,
    df_2024_2700_long
]

# Объединяем все таблицы в одну
df_all_2700 = pd.concat(df_list_2700, ignore_index=True)

# Меняем значения 0 на nan
df_all_2700 = df_all_2700.replace(0, np.nan)
#Преобразуем названия столбцов DataFrame в стиль snake_case
#df_all_2700 = to_snake_case(df_all_2700)

# Берем префикс и переносим значение в поле Графа
df_all_2700 = extract_numeric_prefix_from_dataset(df_all_2700)

# Проверяем результат
print(f"Объединено строк: {len(df_all_2700)}")
print(f"Колонки: {list(df_all_2700.columns)}")
df_all_2700.head()

Выбрана колонка для извлечения префикса: 'Категория выявления'
Из колонки 'Категория выявления' извлечено 864 числовых префиксов
Объединено строк: 864
Колонки: ['Показатель', 'Категория выявления', '№ строки', 'Год', 'Значение', 'Графа']


,Показатель,Категория выявления,№ строки,Год,Значение,Графа
0,Взято на учёт в предыдущем году,Впервые выявленные больные всего,1,2016,2541.0,3
1,Выбыло в другие территории,Впервые выявленные больные всего,2,2016,147.0,3
2,Умерло от туберкулеза,Впервые выявленные больные всего,3,2016,119.0,3
3,Умерло от других причин,Впервые выявленные больные всего,4,2016,302.0,3
4,Диагноз туберкулеза снят,Впервые выявленные больные всего,5,2016,30.0,3


In [90]:
df_all_2700.columns

Index(['Показатель', 'Категория выявления', '№ строки', 'Год', 'Значение',
       'Графа'],
      dtype='object')

In [91]:
df_all_2700 = df_all_2700.rename(columns={
    'Категория выявления': 'Уточнение',
    '№ строки': 'Строка'
})[['Показатель', 'Уточнение', 'Год', 'Значение','Строка', 'Графа']]

In [92]:
# Замена ё на е во всем датасете
df_all_2700 = df_all_2700.replace('ё', 'е', regex=True)
# преобразовать № строки в число
df_all_2700['Строка'] = pd.to_numeric(df_all_2700['Строка'], errors='coerce')


In [93]:
df_all_2700['Уточнение'] = df_all_2700['Уточнение'].replace({
    'Впервые выявленные больные Всего': 'Впервые выявленные больные всего',
    'Больные с рецидивом туберкулеза Всего': 'Больные с рецидивом туберкулеза всего',
    'Впервые выявленные больные с деструкцией легочной ткани': 'Впервые выявленные больные в том числе с деструкцией легочной ткани',
    'Больные с рецидивом туберкулеза с деструкцией легочной ткани': 'Больные с рецидивом туберкулеза в том числе с деструкцией легочной ткани'
})

In [94]:
print(df_all_2700['Уточнение'].unique())

['Впервые выявленные больные всего'
 'Впервые выявленные больные в том числе бактериовыделители'
 'Впервые выявленные больные в том числе с деструкцией легочной ткани'
 'Впервые выявленные больные без бактериовыделения и деструкции'
 'Больные с рецидивом туберкулеза всего'
 'Больные с рецидивом туберкулеза в том числе бактериовыделители'
 'Больные с рецидивом туберкулеза в том числе с деструкцией легочной ткани'
 'Больные с рецидивом туберкулеза без бактериовыделения и деструкции']


In [95]:
print(df_all_2700['Показатель'].unique())

['Взято на учет в предыдущем году' 'Выбыло в другие территории'
 'Умерло от туберкулеза' 'Умерло от других причин'
 'Диагноз туберкулеза снят' 'Выделение МБТ прекратилось*'
 'Полость распада закрылась*' 'Переведено в III группу*'
 'Прибыло из других территорий' 'Выделение МБТ прекратилось'
 'Полость распада закрылась' 'Переведено в III группу']


In [96]:
print(df_all_2700['Год'].unique())
print(df_all_2700['Строка'].unique())
print(df_all_2700['Графа'].unique())

[2016 2017 2018 2019 2020 2021 2022 2023 2024]
[ 1  2  3  4  5  6  7  8  9 10 11 12]
[ 3  4  5  6  7  8  9 10]


In [97]:
df_all_2700 = df_all_2700.drop_duplicates()

In [98]:
df_all_2700.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 864 entries, 0 to 863
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Показатель  864 non-null    object 
 1   Уточнение   864 non-null    object 
 2   Год         864 non-null    int64  
 3   Значение    579 non-null    float64
 4   Строка      864 non-null    int64  
 5   Графа       864 non-null    int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 40.6+ KB


#### Сведение 2100 +

In [99]:
# Список всех датафреймов 2100 за разные годы
df_list_2100 = [
    df_2016_2100_long,
    df_2017_2100_long,
    df_2018_2100_long,
    df_2019_2100_long,
    df_2020_2100_long,
    df_2021_2100_long,
    df_2022_2100_long,
    df_2023_2100_long,
    df_2024_2100_long
]

# Объединяем все таблицы в одну
df_all_2100 = pd.concat(df_list_2100, ignore_index=True)
#Преобразуем названия столбцов DataFrame в стиль snake_case
#df_all_2100 = to_snake_case(df_all_2100)

# Берем префикс и переносим значение в поле Графа
df_all_2100 = extract_numeric_prefix_from_dataset(df_all_2100)

# Проверяем результат
print(f"Объединено строк: {len(df_all_2100)}")
print(f"Колонки: {list(df_all_2100.columns)}")
df_all_2100.head()

Выбрана колонка для извлечения префикса: 'Возраст'
Из колонки 'Возраст' извлечено 702 числовых префиксов
Объединено строк: 702
Колонки: ['1_Формы туберкулеза', 'Возраст', '№ строки', 'Код по МКБ-Х пересмотра', 'Год', 'Значение', 'Графа']


,1_Формы туберкулеза,Возраст,№ строки,Код по МКБ-Х пересмотра,Год,Значение,Графа
0,Туберкулез органов дыхания-всего,Взято на учет в отчетном году пациентов с впервые в жизни установленным диагнозом всего,01,A15;A16;A19 часть,2016,2293.0,4
1,в том числе туберкулез легких,Взято на учет в отчетном году пациентов с впервые в жизни установленным диагнозом всего,02,A15.0-A15.3; A15.7 часть;A16.0-A16.2;A16.7 часть;A19 часть,2016,2171.0,4
2,из него: фиброзно-кавернозный,Взято на учет в отчетном году пациентов с впервые в жизни установленным диагнозом всего,03,A15.0-A15.3; A16.0-A16.2,2016,16.0,4
3,Из общего числа больных туберкулезом легких выявлено в фазе распада,Взято на учет в отчетном году пациентов с впервые в жизни установленным диагнозом всего,04,NaN,2016,929.0,4
4,Из общего числа больных туберкулезом легких выявлено без распада и без бактериовыделения,Взято на учет в отчетном году пациентов с впервые в жизни установленным диагнозом всего,05,NaN,2016,968.0,4


In [100]:
print(df_all_2100['Возраст'].unique())


['Взято на учет в отчетном году пациентов с впервые в жизни установленным диагнозом всего'
 'Взято на учет в отчетном году пациентов с впервые в жизни установленным диагнозом в том числе дети 0-14 лет'
 'Взято на учет в отчетном году пациентов с впервые в жизни установленным диагнозом в том числе подростков 15-17 лет'
 'Контингенты больных на конец отчетного года всего'
 'Контингенты больных на конец отчетного года в том числе дети 0-14 лет'
 'Контингенты больных на конец отчетного года в том числе подростков 15-17 лет'
 'Взято на учет в отчетном году больных с первые в жизни установленным диагнозом всего'
 'Взято на учет в отчетном году больных с первые в жизни установленным диагнозом из них: детей 0-14 лет'
 'Взято на учет в отчетном году больных с первые в жизни установленным диагнозом из них: подростков 15-17 лет'
 'Контингенты больных на конец отчетного года в том числе: детей 0-14 лет'
 'Контингенты больных на конец отчетного года в том числе: подростков 15-17 лет']


In [101]:
# Словарь для столбца 'Уточнение' (общая категория)
type_mapping = {
    'Взято на учет в отчетном году пациентов с впервые в жизни установленным диагнозом всего': 'Взято на учет в отчетном году больных с впервые в жизни установленным диагнозом',
    'Взято на учет в отчетном году пациентов с впервые в жизни установленным диагнозом в том числе дети 0-14 лет': 'Взято на учет в отчетном году больных с впервые в жизни установленным диагнозом',
    'Взято на учет в отчетном году пациентов с впервые в жизни установленным диагнозом в том числе подростков 15-17 лет': 'Взято на учет в отчетном году больных с впервые в жизни установленным диагнозом',
    'Контингенты больных на конец отчетного года всего': 'Контингенты больных на конец отчетного года',
    'Контингенты больных на конец отчетного года в том числе дети 0-14 лет': 'Контингенты больных на конец отчетного года',
    'Контингенты больных на конец отчетного года в том числе подростков 15-17 лет': 'Контингенты больных на конец отчетного года',
    'Взято на учет в отчетном году больных с первые в жизни установленным диагнозом всего': 'Взято на учет в отчетном году больных с впервые в жизни установленным диагнозом',
    'Взято на учет в отчетном году больных с первые в жизни установленным диагнозом из них: детей 0-14 лет': 'Взято на учет в отчетном году больных с впервые в жизни установленным диагнозом',
    'Взято на учет в отчетном году больных с первые в жизни установленным диагнозом из них: подростков 15-17 лет': 'Взято на учет в отчетном году больных с впервые в жизни установленным диагнозом',
    'Контингенты больных на конец отчетного года в том числе: детей 0-14 лет': 'Контингенты больных на конец отчетного года',
    'Контингенты больных на конец отчетного года в том числе: подростков 15-17 лет': 'Контингенты больных на конец отчетного года',
}

# Словарь для столбца 'Еще уточнение' (детализация)
detail_mapping = {
    'Взято на учет в отчетном году пациентов с впервые в жизни установленным диагнозом всего': 'всего',
    'Взято на учет в отчетном году пациентов с впервые в жизни установленным диагнозом в том числе дети 0-14 лет': 'из них детей от 0 до 14 лет',
    'Взято на учет в отчетном году пациентов с впервые в жизни установленным диагнозом в том числе подростков 15-17 лет': 'из них подростков 15-17 лет',
    'Контингенты больных на конец отчетного года всего': 'всего',
    'Контингенты больных на конец отчетного года в том числе дети 0-14 лет': 'из них детей от 0 до 14 лет',
    'Контингенты больных на конец отчетного года в том числе подростков 15-17 лет': 'из них подростков 15-17 лет',
    'Взято на учет в отчетном году больных с первые в жизни установленным диагнозом всего': 'всего',
    'Взято на учет в отчетном году больных с первые в жизни установленным диагнозом из них: детей 0-14 лет': 'из них детей от 0 до 14 лет',
    'Взято на учет в отчетном году больных с первые в жизни установленным диагнозом из них: подростков 15-17 лет': 'из них подростков 15-17 лет',
    'Контингенты больных на конец отчетного года в том числе: детей 0-14 лет': 'из них детей от 0 до 14 лет',
    'Контингенты больных на конец отчетного года в том числе: подростков 15-17 лет': 'из них подростков 15-17 лет',
}

# Применяем отображение
df_all_2100['Уточнение'] = df_all_2100['Возраст'].map(type_mapping)
df_all_2100['Еще уточнение'] = df_all_2100['Возраст'].map(detail_mapping)

# Удаляем старый столбец
df_all_2100 = df_all_2100.drop(columns=['Возраст'])


In [102]:
df_all_2100 = df_all_2100.rename(columns={
    '1_Формы туберкулеза': 'Формы туберкулеза',
    '№ строки': 'Строка'
})[['Формы туберкулеза', 'Уточнение', 'Еще уточнение', 'Код по МКБ-Х пересмотра', 'Год', 'Значение', 'Строка', 'Графа']]

In [103]:
df_all_2100['Строка'] = pd.to_numeric(df_all_2100['Строка'], errors='coerce')
df_all_2100['Строка'] = df_all_2100['Строка'].astype(int)

In [104]:
df_all_2100['Формы туберкулеза'] = df_all_2100['Формы туберкулеза'].replace({
    'Туберкулез органов дыхания-всего': 'Туберкулез органов дыхания - всего',
    'Туберкулез органов дыхания - всего': 'Туберкулез органов дыхания - всего',
    'в том числе туберкулез легких': 'Туберкулез органов дыхания - в том числе туберкулез легких',
    'из него: фиброзно-кавернозный': 'Туберкулез органов дыхания - из него: фиброзно-кавернозный',
    'Из общего числа больных туберкулезом легких выявлено в фазе распада': 'Из общего числа больных туберкулезом легких выявлено в фазе распада',
    'Из общего числа больных туберкулезом легких выявлено без распада и без бактериовыделения': 'Из общего числа больных туберкулезом легких выявлено без распада и без бактериовыделения',
    'Другие локализации туберкулеза': 'Другие локализации туберкулеза',
    'Итого (сумма строк 01, 06)': 'Итого (сумма строк 01, 06)',
    'Имеют инвалидность в связи с туберкулезом': 'Имеют инвалидность в связи с туберкулезом',
    'в том числе : первой группы': 'Имеют инвалидность в связи с туберкулезом в том числе первой группы',
    'в том числе: первой группы': 'Имеют инвалидность в связи с туберкулезом в том числе первой группы',
    'второй группы': 'Имеют инвалидность в связи с туберкулезом в том числе второй группы',
    'Обследовано на АТ к ВИЧ': 'Обследовано на АТ к ВИЧ',
    'из них с положительным результатом методом иммунного блотинга': 'Обследовано на АТ к ВИЧ из них с положительным результатом методом иммунного блотинга',
    'Туберкулез в сочетании с ВИЧ': 'Туберкулез в сочетании с ВИЧ'
})

# Проверка
print(df_all_2100['Формы туберкулеза'].unique())

['Туберкулез органов дыхания - всего'
 'Туберкулез органов дыхания - в том числе туберкулез легких'
 'Туберкулез органов дыхания - из него: фиброзно-кавернозный'
 'Из общего числа больных туберкулезом легких выявлено в фазе распада'
 'Из общего числа больных туберкулезом легких выявлено без распада и без бактериовыделения'
 'Другие локализации туберкулеза' 'Итого (сумма строк 01, 06)'
 'Имеют инвалидность в связи с туберкулезом'
 'Имеют инвалидность в связи с туберкулезом в том числе первой группы'
 'Имеют инвалидность в связи с туберкулезом в том числе второй группы'
 'Обследовано на АТ к ВИЧ'
 'Обследовано на АТ к ВИЧ из них с положительным результатом методом иммунного блотинга'
 'Туберкулез в сочетании с ВИЧ']


In [105]:
print(df_all_2100['Уточнение'].unique())
print(df_all_2100['Еще уточнение'].unique())


['Взято на учет в отчетном году больных с впервые в жизни установленным диагнозом'
 'Контингенты больных на конец отчетного года']
['всего' 'из них детей от 0 до 14 лет' 'из них подростков 15-17 лет']


In [106]:
print(df_all_2100['Код по МКБ-Х пересмотра'].unique())


['A15;A16;A19 часть'
 'A15.0-A15.3; A15.7 часть;A16.0-A16.2;A16.7 часть;A19 часть'
 'A15.0-A15.3; A16.0-A16.2' nan 'A17;A18;A19 часть'
 'A15-A19 и B20.0-23.0; Z 21' 'А15; А16; А19 часть'
 'А15.0-А15.3; А15.7 часть; А16.0-А16.2; А16.7 часть; А19 часть'
 'А15.0-А15.3; А16.0-А16.2' 'А17; А18; А19 часть' 'А15-А19'
 'А15-А19 и В20.0 – 23.0; Z 21']


In [107]:
print(df_all_2100['Год'].unique())
print(df_all_2100['Строка'].unique())
print(df_all_2100['Графа'].unique())

[2016 2017 2018 2019 2020 2021 2022 2023 2024]
[ 1  2  3  4  5  6  7  8  9 10 11 12 13]
[4 5 6 7 8 9]


In [108]:
df_all_2100 = df_all_2100.drop_duplicates()

In [109]:
df_all_2100.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 702 entries, 0 to 701
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Формы туберкулеза        702 non-null    object 
 1   Уточнение                702 non-null    object 
 2   Еще уточнение            702 non-null    object 
 3   Код по МКБ-Х пересмотра  318 non-null    object 
 4   Год                      702 non-null    int64  
 5   Значение                 517 non-null    float64
 6   Строка                   702 non-null    int64  
 7   Графа                    702 non-null    int64  
dtypes: float64(1), int64(3), object(4)
memory usage: 44.0+ KB


#### Сведение 2200 +

In [110]:
# Список всех датафреймов 2200 за разные годы
df_list_2200 = [
    df_2016_2200_long,
    df_2017_2200_long,
    df_2018_2200_long,
    df_2019_2200_long,
    df_2020_2200_long,
    df_2021_2200_long,
    df_2022_2200_long,
    df_2023_2200_long,
    df_2024_2200_long
]

# Объединяем все таблицы в одну
df_all_2200 = pd.concat(df_list_2200, ignore_index=True)

# Берем префикс и переносим значение в поле Графа
df_all_2200 = extract_numeric_prefix_from_dataset(df_all_2200)

# БЕЗОПАСНАЯ обработка колонки "Строка"
# 1. Заменяем '(4/01' на '4' (как строку)
df_all_2200['Строка'] = df_all_2200['Строка'].replace('(4/01', '1')

# 2. Преобразуем в numeric с сохранением всех значений
#    (оставляем как float, чтобы избежать ошибок с NaN)
df_all_2200['Строка'] = pd.to_numeric(df_all_2200['Строка'], errors='coerce')
df_all_2200['Строка'] = df_all_2200['Строка'].astype(int)

df_all_2200.head()

Выбрана колонка для извлечения префикса: 'Возраст'
Из колонки 'Возраст' извлечено 324 числовых префиксов


,Показатель,Возраст,Строка,Год,Значение,Графа
0,Впервые выявлено больных туберкулезом из числа осмотренных на туберкулез,Всего,1,2016,1601,3
1,из них с применением: туберкулинодиагностики,Всего,2,2016,97,3
2,в том числе аллергена туберкулезного рекомбинантного в стандартном разведении,Всего,3,2016,NaN,3
3,флюрографии,Всего,4,2016,1472,3
4,бактериологических методов,Всего,5,2016,32,3


In [111]:
df_all_2200[df_all_2200['Год']== 2021]['Строка'].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12])

In [112]:
# Переименовываем колонки
df_all_2200 = df_all_2200.rename(columns={
    'Показатель': 'Наименование показателя',
    'Возраст': 'Уточнение',
    'значение': 'Значение'
})

# Выбираем колонки в нужном порядке
df_all_2200 = df_all_2200[['Наименование показателя', 'Уточнение', 'Год', 'Значение', 'Строка', 'Графа']]
df_all_2200.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 324 entries, 0 to 323
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Наименование показателя  324 non-null    object
 1   Уточнение                324 non-null    object
 2   Год                      324 non-null    int64 
 3   Значение                 252 non-null    object
 4   Строка                   324 non-null    int64 
 5   Графа                    324 non-null    int64 
dtypes: int64(3), object(3)
memory usage: 15.3+ KB


In [113]:
df_all_2200['Наименование показателя'] = df_all_2200['Наименование показателя'].replace({
    'из них с применением: туберкулинодиагностики': 'Впервые выявлено больных туберкулезом из числа осмотренных на туберкулез из них с применением туберкулинодиагностики',
    'в том числе аллергена туберкулезного рекомбинантного в стандартном  разведении ': 'Впервые выявлено больных туберкулезом из числа осмотренных на туберкулез в том числе аллергена туберкулезного рекомбинантного в стандартном разведении',
    'в том числе аллергена туберкулезного рекомбинантного в стандартном разведении': 'Впервые выявлено больных туберкулезом из числа осмотренных на туберкулез в том числе аллергена туберкулезного рекомбинантного в стандартном разведении',
    'флюрографии': 'Впервые выявлено больных туберкулезом из числа осмотренных на туберкулез из них с применением флюрографии',
    'бактериологических методов': 'Впервые выявлено больных туберкулезом из числа осмотренных на туберкулез из них с применением бактериологических методов',
    'в том числе методом бактериоскопии': 'Впервые выявлено больных туберкулезом из числа осмотренных на туберкулез в том числе методом бактериоскопии',
    'Взято на учет в III А группу диспансерного учета': 'Взято на учет в III группу диспансерного учета',
    'Взято на учет в IIIА группу диспансерного учета': 'Взято на учет в III группу диспансерного учета',
    'Взято на учет в V  группу диспансерного учета: Всего': 'Взято на учет в V группу диспансерного учета всего',
    'Взято на учет в V группу диспансерного учета:  Всего': 'Взято на учет в V группу диспансерного учета всего',
    'Взято на учет в V группу диспансерного учета:': 'Взято на учет в V группу диспансерного учета всего',
    '            в том числе в VA': 'Взято на учет в V группу диспансерного учета в том числе VA',
    'в том числе в  VА': 'Взято на учет в V группу диспансерного учета в том числе VA',
    '                               VБ': 'Взято на учет в V группу диспансерного учета в том числе VБ',
    'VБ': 'Взято на учет в V группу диспансерного учета в том числе VБ',
    'Кроме того умерло больных от туберкулеза постоянных жителей, диагноз у которых установлен посмертно ': 'Кроме того умерло больных от туберкулеза постоянных жителей, диагноз у которых установлен посмертно',
    'Кроме того умерло больных от туберкулеза постоянных жителей, диагноз у которых установлен посмертно Иркутская область черн верн': 'Кроме того умерло больных от туберкулеза постоянных жителей, диагноз у которых установлен посмертно',
    'Кроме того умерло больных от ВИЧ-инфекции постоянных жителей, диагноз у которых установлен посмертно ': 'Кроме того умерло больных от ВИЧ-инфекции постоянных жителей, диагноз туберкулеза у которых установлен посмертно',
})

#df_all_2200 = df_all_2200[df_all_2200['Наименование показателя'] != '']

In [114]:
df_all_2200['Наименование показателя'].unique()

array(['Впервые выявлено больных туберкулезом из числа осмотренных на туберкулез',
       'Впервые выявлено больных туберкулезом из числа осмотренных на туберкулез из них с применением туберкулинодиагностики',
       'Впервые выявлено больных туберкулезом из числа осмотренных на туберкулез в том числе аллергена туберкулезного рекомбинантного в стандартном разведении',
       'Впервые выявлено больных туберкулезом из числа осмотренных на туберкулез из них с применением флюрографии',
       'Впервые выявлено больных туберкулезом из числа осмотренных на туберкулез из них с применением бактериологических методов',
       'Впервые выявлено больных туберкулезом из числа осмотренных на туберкулез в том числе методом бактериоскопии',
       'Взято на учет в III группу диспансерного учета',
       'Взято на учет в V группу диспансерного учета всего',
       'Взято на учет в V группу диспансерного учета в том числе VA',
       'Взято на учет в V группу диспансерного учета в том числе VБ',
      

In [115]:
df_all_2200['Уточнение'] = df_all_2200['Уточнение'].replace({
    'Всего': 'Всего',
    'из них: детей 0-14 лет': 'Из них детей от 0 до 14 лет',
    'из них:  детей 0-14 лет': 'Из них детей от 0 до 14 лет',
    'подростков              15-17 лет': 'Из них подростков от 15 до 17 лет',
    'подростков 15-17 лет': 'Из них подростков от 15 до 17 лет'
})

print(df_all_2200['Уточнение'].unique())

['Всего' 'Из них детей от 0 до 14 лет' 'Из них подростков от 15 до 17 лет']


In [116]:
df_all_2200['Уточнение'].unique()

array(['Всего', 'Из них детей от 0 до 14 лет',
       'Из них подростков от 15 до 17 лет'], dtype=object)

In [117]:

print(df_all_2200['Год'].unique())
print(df_all_2200['Строка'].unique())
print(df_all_2200['Графа'].unique())


[2016 2017 2018 2019 2020 2021 2022 2023 2024]
[ 1  2  3  4  5  6  7  8  9 10 11 12]
[3 4 5]


In [118]:
df_all_2200 = df_all_2200.drop_duplicates()

In [119]:
df_all_2200.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 324 entries, 0 to 323
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Наименование показателя  324 non-null    object
 1   Уточнение                324 non-null    object
 2   Год                      324 non-null    int64 
 3   Значение                 252 non-null    object
 4   Строка                   324 non-null    int64 
 5   Графа                    324 non-null    int64 
dtypes: int64(3), object(3)
memory usage: 15.3+ KB


#### Сведение 2300 +

In [120]:
# Список всех датафреймов 2300 за разные годы
df_list_2300 = [
    df_2016_2300_long,
    df_2017_2300_long,
    df_2018_2300_long,
    df_2019_2300_long,
    df_2020_2300_long,
    df_2021_2300_long,
    df_2022_2300_long,
    df_2023_2300_long,
    df_2024_2300_long
]

# Объединяем все таблицы в одну
df_all_2300 = pd.concat(df_list_2300, ignore_index=True)

# Преобразуем названия столбцов DataFrame в стиль snake_case
#df_all_2300 = to_snake_case(df_all_2300)

# Берем префикс и переносим значение в поле Графа
df_all_2300 = extract_numeric_prefix_from_dataset(df_all_2300)

# Проверяем результат
print(f"Объединено строк: {len(df_all_2300)}")
print(f"Колонки: {list(df_all_2300.columns)}")
df_all_2300.head()

Выбрана колонка для извлечения префикса: 'формы'
Из колонки 'формы' извлечено 648 числовых префиксов
Объединено строк: 648
Колонки: ['показатель', 'формы', 'Строка', 'Год', 'Значение', 'Графа']


,показатель,формы,Строка,Год,Значение,Графа
0,Взято на учет рецидивов,Туберкулез органов дыхания Всего,1,2016,285.0,3
1,из них из III группы,Туберкулез органов дыхания Всего,2,2016,96.0,3
2,Прибыло,Туберкулез органов дыхания Всего,3,2016,1098.0,3
3,Переведено в III группу,Туберкулез органов дыхания Всего,4,2016,2531.0,3
4,Диагноз туберкулеза снят,Туберкулез органов дыхания Всего,5,2016,30.0,3


In [121]:
# Переименовываем колонки
df_all_2300 = df_all_2300.rename(columns={
    'показатель': 'Показатель',
    'год': 'Год',
    'значение': 'Значение'
})

# Определяем возможные суффиксы (возрастные группы или "Всего")
suffixes = ['Всего', 'дети 0-14 лет', 'подростки 15-17 лет']

# Словарь для замены суффиксов
suffix_mapping = {
    'Всего': 'Всего',
    'дети 0-14 лет': 'Дети от 0 до 14 лет',
    'подростки 15-17 лет': 'Подростки от 15 до 17 лет'
}

# Словарь для замены основной части
main_mapping = {
    'Туберкулез органов дыхания': 'Туберкулез органов дыхания',
    'Туберкулез легких': 'Туберкулез органов дыхания из них туберкулез легких',
    'Другие формы туберкулеза': 'Прочие формы туберкулеза'
}

# Функция для разделения колонки 'формы'
def split_form(value):
    value = str(value).strip()
    for suffix in suffixes:
        if value.endswith(suffix):
            # Убираем суффикс и лишние пробелы
            main_part = value[:-len(suffix)].strip()
            # Применяем маппинг к основной части
            main_part = main_mapping.get(main_part, main_part)
            # Применяем маппинг к суффиксу
            new_suffix = suffix_mapping.get(suffix, suffix)
            return main_part, new_suffix
    # Если не найдено, проверяем значение без суффикса
    main_part = main_mapping.get(value, value)
    return main_part, 'Всего'

# Применяем функцию
df_all_2300[['Уточнение', 'Еще уточнение']] = df_all_2300['формы'].apply(lambda x: pd.Series(split_form(x)))

# Выбираем нужные колонки в правильном порядке
df_all_2300 = df_all_2300[['Показатель', 'Уточнение', 'Еще уточнение', 'Год', 'Значение', 'Строка', 'Графа']]

# Проверяем результат
print(df_all_2300['Уточнение'].unique())
print(df_all_2300['Еще уточнение'].unique())

['Туберкулез органов дыхания'
 'Туберкулез органов дыхания из них туберкулез легких'
 'Прочие формы туберкулеза']
['Всего' 'Дети от 0 до 14 лет' 'Подростки от 15 до 17 лет']


In [122]:
df_all_2300['Показатель'] = df_all_2300['Показатель'].replace({
    'из них из III группы': 'Взято на учет рецидивов III группы'
})

In [123]:
print(df_all_2300['Показатель'].unique())

['Взято на учет рецидивов' 'Взято на учет рецидивов III группы' 'Прибыло'
 'Переведено в III группу' 'Диагноз туберкулеза снят' 'Выбыло'
 'Умерло от туберкулеза' 'Умерло от других причин']


In [124]:
df_all_2300 = df_all_2300.drop_duplicates()

In [125]:
df_all_2200.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 324 entries, 0 to 323
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Наименование показателя  324 non-null    object
 1   Уточнение                324 non-null    object
 2   Год                      324 non-null    int64 
 3   Значение                 252 non-null    object
 4   Строка                   324 non-null    int64 
 5   Графа                    324 non-null    int64 
dtypes: int64(3), object(3)
memory usage: 15.3+ KB


In [126]:
df_all_2300.head()

,Показатель,Уточнение,Еще уточнение,Год,Значение,Строка,Графа
0,Взято на учет рецидивов,Туберкулез органов дыхания,Всего,2016,285.0,1,3
1,Взято на учет рецидивов III группы,Туберкулез органов дыхания,Всего,2016,96.0,2,3
2,Прибыло,Туберкулез органов дыхания,Всего,2016,1098.0,3,3
3,Переведено в III группу,Туберкулез органов дыхания,Всего,2016,2531.0,4,3
4,Диагноз туберкулеза снят,Туберкулез органов дыхания,Всего,2016,30.0,5,3


#### Сведение 2400 +

In [127]:
# Список всех датафреймов 2400 за разные годы
df_list_2400 = [
    df_2016_2400_long,
    df_2017_2400_long,
    df_2018_2400_long,
    df_2019_2400_long,
    df_2020_2400_long,
    df_2021_2400_long,
    df_2022_2400_long,
    df_2023_2400_long,
    df_2024_2400_long
]

# Объединяем все таблицы в одну
df_all_2400 = pd.concat(df_list_2400, ignore_index=True)

# Преобразуем названия столбцов DataFrame в стиль snake_case
#df_all_2400 = to_snake_case(df_all_2400)

# Берем префикс и переносим значение в поле Графа
df_all_2400 = extract_numeric_prefix_from_dataset(df_all_2400)

# Проверяем результат
print(f"Объединено строк: {len(df_all_2400)}")
print(f"Колонки: {list(df_all_2400.columns)}")
df_all_2400.head()

Выбрана колонка для извлечения префикса: 'состоит'
Из колонки 'состоит' извлечено 882 числовых префиксов
Объединено строк: 882
Колонки: ['группа_учета', 'состоит', 'Строка', 'Год', 'Значение', 'Графа']


,группа_учета,состоит,Строка,Год,Значение,Графа
0,"Взрослые, из них нуждающиеся: в определении активности туберкулезного процесса (гр.0А)",Взято в текущем году,01,2016,441.0,3
1,в проведении дифференциально-диагностических мероприятий (гр.0Б),Взято в текущем году,02,2016,450.0,3
2,Взрослые с неактивным туберкулезным процессом после клинического излечения (гр.III),Взято в текущем году,03,2016,2531.0,3
3,Взрослые состоящие :в бытовом и производственном контакте с бактериовыделителем (гр.IVА),Взято в текущем году,04,2016,1211.0,3
4,в бытовом и производственном контакте с больным туберкулезом без бактериовыделния (гр.IVА),Взято в текущем году,05,2016,1329.0,3


In [128]:
df_all_2400['Строка'] = df_all_2400['Строка'].astype(int)

In [129]:
# Переименовываем столбцы и задаём порядок
df_all_2400 = df_all_2400.rename(columns={
    'группа_учета': 'Показатель',
    'состоит': 'Уточнение'
})[['Показатель', 'Уточнение', 'Год', 'Значение', 'Строка', 'Графа']]


In [130]:
df_all_2400['Показатель'] = df_all_2400['Показатель'].replace({
    'Взрослые, из них нуждающиеся: в определении активности туберкулезного процесса (гр.0А)': 'Взрослые, из них нуждающиеся в определении активности туберкулезного процесса (0А)',
    'в проведении дифференциально-диагностических мероприятий (гр.0Б)': 'Взрослые, из них нуждающиеся в проведении дифференциально-диагностических мероприятий (0Б)',
    'Взрослые с неактивным туберкулезным процессом после клинического излечения (гр.III)': 'Взрослые с неактивным туб процессом после канонического излечения (III)',
    'Взрослые состоящие :в бытовом и производственном контакте с бактериовыделителем (гр.IVА)': 'Взрослые состоящие в бытовом и производственном контакте с бактериовыделителем (IVA)',
    'в бытовом и производственном контакте с больным туберкулезом без бактериовыделния (гр.IVА)': 'Взрослые состоящие в бытовом и производственном контакте с больным туберкулезом без бактериовыделения (IVA)',
    'в прфессиональном контакте с источником инфекции (гр.IVБ)': 'Взрослые состоящие в профессиональном контакте с источником инфекции (IVБ)',
    'Дети от 0 до 17 лет, из них: нуждающиеся в уточнении характера туберкулиновой чувствительности, уточнении активности туберкулеза и диагностике (гр.0)': 'Дети от 0 до 17 лет, из них нуждающиеся в уточнении характера туберкулезной чувствительности, уточнении активности туберкулеза и диагностике (0)',
    'с остаточными посттуберкулезными изменениями ( гр.III А)': 'Дети от 0 до 17 лет с остаточными посттуберкулезными изменениями (IIIА)',
    'переведенные из I, II, III А групп (III Б)': 'Дети от 0 до 17 лет переведнные из I, II, IIIA групп (IIIБ)',
    'состоящие в контакте с бактериовыделителями (гр.IVА)': 'Дети от 0 до 17 лет, состоящие в контакте с бактериовыделителями (IVA)',
    'из контакта с больными туберкулезом без бактериовыделения, из семей животноводов или имеющих больных туберкулезом животных (гр.IVБ)': 'Дети от 0 до 17 лет из контакта с больными туберкулезом без бактериовыделения, из семей животноводов или имеющих больных туберкулезом животных (IVБ)',
    'в раннем периоде первичной туберкулезной инфекции (группа VIA)': 'Дети от 0 до 17 лет в ранеем периоде первичной туберкулезной инфекции (VIA)',
    'раннее инфицированные, с гиперергической реакцией на туберкулин, соц. групп риска с выраженными реакциями на туберкулин (гр.VIБ)': 'Дети от 0 до 17 лет ранее инфецированные, с гиперергической реакцией на туберкулин из соц групп риска с выраженными реакциями на туберкулин (VIБ)',
    'с усиливающейся туберкулиновой чувствительностью (гр.VIВ)': 'Дети от 0 до 17 лет с усиливающейся туберкулиновой чувствительностью (VIB)',
    # дублирующие варианты с другими пробелами/точками (если есть в уникальных)
    'Взрослые, из них нуждающиеся в определении активности туберкулезного процесса (гр. 0А)': 'Взрослые, из них нуждающиеся в определении активности туберкулезного процесса (0А)',
    'в проведении дифференциально-диагностических мероприятий (гр. 0 Б)': 'Взрослые, из них нуждающиеся в проведении дифференциально-диагностических мероприятий (0Б)',
    'Взрослые с неактивным туб процессом после клинического излечения (гр. III)': 'Взрослые с неактивным туб процессом после канонического излечения (III)',
    'Взрослые состоящие в бытовом и производственном контакте с бактериовыделителем (гр. IVА)': 'Взрослые состоящие в бытовом и производственном контакте с бактериовыделителем (IVA)',
    'в бытовом и производственном контакте с больным туберкулезом без бактериовыделения (гр. IVА)': 'Взрослые состоящие в бытовом и производственном контакте с больным туберкулезом без бактериовыделения (IVA)',
    'в профессиональном контакте с источником инфекции (гр. IVБ)': 'Взрослые состоящие в профессиональном контакте с источником инфекции (IVБ)',
    'Дети от 0 до 17 лет, из них нуждающиеся в уточнении характера туберкулиновой чувствительности, уточнении активности туберкулеза и диагностике (гр. 0)': 'Дети от 0 до 17 лет, из них нуждающиеся в уточнении характера туберкулезной чувствительности, уточнении активности туберкулеза и диагностике (0)',
    'с остаточными посттуберкулезными изменениями (гр III А)': 'Дети от 0 до 17 лет с остаточными посттуберкулезными изменениями (IIIА)',
    'переведенные из I, II, IIIA групп (гр. III Б)': 'Дети от 0 до 17 лет переведнные из I, II, IIIA групп (IIIБ)',
    'состоящие в контакте с бактериовыделителями (гр. IVА)': 'Дети от 0 до 17 лет, состоящие в контакте с бактериовыделителями (IVA)',
    'из контакта с больными туберкулезом без бактериовыделения, из семей животноводов или имеющих больных туберкулезом животных (гр. IVБ)': 'Дети от 0 до 17 лет из контакта с больными туберкулезом без бактериовыделения, из семей животноводов или имеющих больных туберкулезом животных (IVБ)',
    'в раннем периоде первичной туберкулезной инфекции (группа VIА)': 'Дети от 0 до 17 лет в ранеем периоде первичной туберкулезной инфекции (VIA)',
    'ранее инфицированные, с гиперергической реакцией на туберкулин, из соц. групп риска с выраженными реакциями на туберкулин (гр. VIБ)': 'Дети от 0 до 17 лет ранее инфецированные, с гиперергической реакцией на туберкулин из соц групп риска с выраженными реакциями на туберкулин (VIБ)',
    'с усиливающейся туберкулиновой чувствительностью (гр. VIВ)': 'Дети от 0 до 17 лет с усиливающейся туберкулиновой чувствительностью (VIB)',
})

# Проверка
print(df_all_2400['Показатель'].unique())

['Взрослые, из них нуждающиеся в определении активности туберкулезного процесса (0А)'
 'Взрослые, из них нуждающиеся в проведении дифференциально-диагностических мероприятий (0Б)'
 'Взрослые с неактивным туб процессом после канонического излечения (III)'
 'Взрослые состоящие в бытовом и производственном контакте с бактериовыделителем (IVA)'
 'Взрослые состоящие в бытовом и производственном контакте с больным туберкулезом без бактериовыделения (IVA)'
 'Взрослые состоящие в профессиональном контакте с источником инфекции (IVБ)'
 'Дети от 0 до 17 лет, из них нуждающиеся в уточнении характера туберкулезной чувствительности, уточнении активности туберкулеза и диагностике (0)'
 'Дети от 0 до 17 лет с остаточными посттуберкулезными изменениями (IIIА)'
 'Дети от 0 до 17 лет переведнные из I, II, IIIA групп (IIIБ)'
 'Дети от 0 до 17 лет, состоящие в контакте с бактериовыделителями (IVA)'
 'Дети от 0 до 17 лет из контакта с больными туберкулезом без бактериовыделения, из семей животноводов или и

In [131]:
df_all_2400['Уточнение'] = df_all_2400['Уточнение'].replace({
    'Взято в текущем году': 'Взято на учет в текущем году',
    'Прошли курс ХП пробного лечения': 'Прошел курс ХП, пробного лечения',
    'Впервые выявлено больных с активным ТБ': 'Впервые выявлено больных с ТБ'
})

# Проверка
print(df_all_2400['Уточнение'].unique())

['Взято на учет в текущем году' 'Подлежало ХП или пробному лечению'
 'Прошел курс ХП, пробного лечения' 'Впервые выявлено больных с ТБ'
 'Снято с учета' 'Выбыло' 'Состоит на конец года'
 'Прошли курс ХП, пробного лечения']


In [132]:
df_all_2400.head()

,Показатель,Уточнение,Год,Значение,Строка,Графа
0,"Взрослые, из них нуждающиеся в определении активности туберкулезного процесса (0А)",Взято на учет в текущем году,2016,441.0,1,3
1,"Взрослые, из них нуждающиеся в проведении дифференциально-диагностических мероприятий (0Б)",Взято на учет в текущем году,2016,450.0,2,3
2,Взрослые с неактивным туб процессом после канонического излечения (III),Взято на учет в текущем году,2016,2531.0,3,3
3,Взрослые состоящие в бытовом и производственном контакте с бактериовыделителем (IVA),Взято на учет в текущем году,2016,1211.0,4,3
4,Взрослые состоящие в бытовом и производственном контакте с больным туберкулезом без бактериовыделения (IVA),Взято на учет в текущем году,2016,1329.0,5,3


In [133]:
print(df_all_2400['Строка'].unique())
print(df_all_2400['Графа'].unique())
print(df_all_2400['Год'].unique())

[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14]
[3 4 5 6 7 8 9]
[2016 2017 2018 2019 2020 2021 2022 2023 2024]


In [134]:
df_all_2400 = df_all_2400.drop_duplicates()

In [135]:
df_all_2400.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 882 entries, 0 to 881
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Показатель  882 non-null    object 
 1   Уточнение   882 non-null    object 
 2   Год         882 non-null    int64  
 3   Значение    775 non-null    float64
 4   Строка      882 non-null    int64  
 5   Графа       882 non-null    int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 41.5+ KB


#### Сведение 2500 +

In [136]:
# Список всех датафреймов 2500 за разные годы
df_list_2500 = [
    df_2016_2500_long,
    df_2017_2500_long,
    df_2018_2500_long,
    df_2019_2500_long,
    df_2020_2500_long,
    df_2021_2500_long,
    df_2022_2500_long,
    df_2023_2500_long,
    df_2024_2500_long
]

# Объединяем все таблицы в одну
df_all_2500 = pd.concat(df_list_2500, ignore_index=True)

# Преобразуем названия столбцов DataFrame в стиль snake_case
#df_all_2500 = to_snake_case(df_all_2500)

# Берем префикс и переносим значение в поле Графа
df_all_2500 = extract_numeric_prefix_from_dataset(df_all_2500)

# Приводим номер строки к целому числу
df_all_2500['Строка'] = df_all_2500['Строка'].astype(int)

# Проверяем результат
print(f"Объединено строк: {len(df_all_2500)}")
print(f"Колонки: {list(df_all_2500.columns)}")
df_all_2500.head()

Выбрана колонка для извлечения префикса: 'группы больных'
Из колонки 'группы больных' извлечено 630 числовых префиксов
Объединено строк: 630
Колонки: ['группы больных', 'форма_заболевания', 'Строка', 'Год', 'Значение', 'Графа']


,группы больных,форма_заболевания,Строка,Год,Значение,Графа
0,Обнаружено из числа больных С впервые в жизни установленным диагнозом,Туберкулёз органов дыхания,1,2016,1027.0,3
1,Обнаружено из числа больных С впервые в жизни установленным диагнозом,Обследовано на МЛУ (из стр.01),2,2016,1019.0,3
2,Обнаружено из числа больных С впервые в жизни установленным диагнозом,из них выявлена МЛУ,3,2016,167.0,3
3,Обнаружено из числа больных С впервые в жизни установленным диагнозом,Туберкулез внелегочных локализаций,4,2016,9.0,3
4,Обнаружено из числа больных С впервые в жизни установленным диагнозом,из них сельских жителей (из суммы строк 01+04),5,2016,296.0,3


In [137]:
# Переименовываем столбцы и задаём порядок
df_all_2500 = df_all_2500.rename(columns={
    'группы больных': 'Уточнение',
    'форма_заболевания': 'Показатель'
})[['Показатель', 'Уточнение', 'Год', 'Значение', 'Строка', 'Графа']]

In [138]:
df_all_2500.columns

Index(['Показатель', 'Уточнение', 'Год', 'Значение', 'Строка', 'Графа'], dtype='object')

In [139]:
df_all_2500['Показатель'] = df_all_2500['Показатель'].replace({
    'Туберкулёз органов дыхания': 'Туберкулез органов дыхания',
    'Обследовано на МЛУ (из стр.01)': 'Обследовано на МЛУ (из стр1)',
    'Обследовано на МЛУ (из стр. 01)': 'Обследовано на МЛУ (из стр1)',
    'из них выявлена МЛУ': 'Обследовано на МЛУ из них выявлена МЛУ',
    'из них сельских жителей (из суммы строк 01+04)': 'Туберкулез внелегочных локализаций, из них сельских жителей (из суммы строк 1 и 4)'
})

# Проверка
print(df_all_2500['Показатель'].unique())

['Туберкулез органов дыхания' 'Обследовано на МЛУ (из стр1)'
 'Обследовано на МЛУ из них выявлена МЛУ'
 'Туберкулез внелегочных локализаций'
 'Туберкулез внелегочных локализаций, из них сельских жителей (из суммы строк 1 и 4)']


In [140]:
df_all_2500['Уточнение'] = df_all_2500['Уточнение'].replace({
    'Обнаружено из числа больных С впервые в жизни установленным диагнозом': 'Обнаружено из числа больных с впервые в жизни установленным диагнозом',
    'Обнаружено из числа больных с впервые в жизни уста-новленным диагнозом': 'Обнаружено из числа больных с впервые в жизни установленным диагнозом',
    'из них методом посева': 'Обнаружено из числа больных с впервые в жизни установленным диагнозом из них методом посева',
    'из них методом микроскопии': 'Обнаружено из числа больных с впервые в жизни установленным диагнозом из них методом микроскопии',
    'Состоящих на учёте в I группе': 'Обнаружено из числа больных состоящих на учете в I группе',
    'состоящих на учете в I группе': 'Обнаружено из числа больных состоящих на учете в I группе',
    'состоящих на учете из них в IБ группе': 'Обнаружено из числа больных состоящих на учете в I группе из них в IБ группе',
    'состоящих на учете в II группе': 'Обнаружено из числа больных состоящих на учете в II группе',
    'состоящих на учете в III группе': 'Обнаружено из числа больных состоящих на учете в III группе',
    'Снятых ранее с учета': 'Обнаружено из числа больных снятых ранее с учета',
    'снятых ранее с учета': 'Обнаружено из числа больных снятых ранее с учета',
    'Переведено из других учреждений больных, выделяющих МБТ': 'Переведено из других учреждений бюджетных, выделяющих МБТ',
    'Переведено из дру-гих учреж-дений б-х, выделяющих МБТ': 'Переведено из других учреждений бюджетных, выделяющих МБТ',
    'от других причин': 'Умерло от других причин'
})

# Проверка результата
print(df_all_2500['Уточнение'].unique())

['Обнаружено из числа больных с впервые в жизни установленным диагнозом'
 'Обнаружено из числа больных с впервые в жизни установленным диагнозом из них методом посева'
 'Обнаружено из числа больных с впервые в жизни установленным диагнозом из них методом микроскопии'
 'Обнаружено из числа больных состоящих на учете в I группе'
 'Обнаружено из числа больных состоящих на учете в I группе из них в IБ группе'
 'Обнаружено из числа больных состоящих на учете в II группе'
 'Обнаружено из числа больных состоящих на учете в III группе'
 'Обнаружено из числа больных снятых ранее с учета'
 'Переведено из других учреждений бюджетных, выделяющих МБТ'
 'Умерло от туберкулеза' 'Умерло от других причин'
 'Перестало выделять МБТ' 'Выбыло из района обслуживания'
 'Состоит на конец отчетного года']


In [141]:
print(df_all_2500['Строка'].unique())
print(df_all_2500['Графа'].unique())
print(df_all_2500['Год'].unique())

[1 2 3 4 5]
[ 3  4  5  6  7  8  9 10 11 12 13 14 15 16]
[2016 2017 2018 2019 2020 2021 2022 2023 2024]


In [142]:
df_all_2500.head()

,Показатель,Уточнение,Год,Значение,Строка,Графа
0,Туберкулез органов дыхания,Обнаружено из числа больных с впервые в жизни установленным диагнозом,2016,1027.0,1,3
1,Обследовано на МЛУ (из стр1),Обнаружено из числа больных с впервые в жизни установленным диагнозом,2016,1019.0,2,3
2,Обследовано на МЛУ из них выявлена МЛУ,Обнаружено из числа больных с впервые в жизни установленным диагнозом,2016,167.0,3,3
3,Туберкулез внелегочных локализаций,Обнаружено из числа больных с впервые в жизни установленным диагнозом,2016,9.0,4,3
4,"Туберкулез внелегочных локализаций, из них сельских жителей (из суммы строк 1 и 4)",Обнаружено из числа больных с впервые в жизни установленным диагнозом,2016,296.0,5,3


In [143]:
df_all_2500 = df_all_2500.drop_duplicates()

In [144]:
df_all_2500.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630 entries, 0 to 629
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Показатель  630 non-null    object 
 1   Уточнение   630 non-null    object 
 2   Год         630 non-null    int64  
 3   Значение    613 non-null    float64
 4   Строка      630 non-null    int64  
 5   Графа       630 non-null    int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 29.7+ KB


#### Сведение 2600 +

In [145]:
# Список всех датафреймов 2600 за разные годы
df_list_2600 = [
    df_2016_2600_long,
    df_2017_2600_long,
    df_2018_2600_long,
    df_2019_2600_long,
    df_2020_2600_long,
    df_2021_2600_long,
    df_2022_2600_long,
    df_2023_2600_long,
    df_2024_2600_long
]

# Объединяем все таблицы в одну
df_all_2600 = pd.concat(df_list_2600, ignore_index=True)

# Преобразуем названия столбцов DataFrame в стиль snake_case
#df_all_2600 = to_snake_case(df_all_2600)

# Берем префикс и переносим значение в поле Графа
df_all_2600 = extract_numeric_prefix_from_dataset(df_all_2600)


# Приводим номер строки к целому числу
df_all_2600['Строка'] = pd.to_numeric(df_all_2600['Строка'], errors='coerce').fillna(0).astype(int)

# Проверяем результат
print(f"Объединено строк: {len(df_all_2600)}")
print(f"Колонки: {list(df_all_2600.columns)}")
df_all_2600.head()

Выбрана колонка для извлечения префикса: 'группы больных'
Из колонки 'группы больных' извлечено 648 числовых префиксов
Объединено строк: 648
Колонки: ['Вид помощи', 'группы больных', 'Строка', 'Год', 'Значение', 'Графа']


,Вид помощи,группы больных,Строка,Год,Значение,Графа
0,Госпитализировано всего,"Больных, состоящих на учете всего",1,2016,4276,3
1,из них бактериовыделителей,"Больных, состоящих на учете всего",2,2016,2222,3
2,в том числе в дневные стационары (из строки 01),"Больных, состоящих на учете всего",3,2016,335,3
3,в том числе в санатории (из строки 01),"Больных, состоящих на учете всего",4,2016,88,3
4,Применены хирургические методы лечения (всего),"Больных, состоящих на учете всего",5,2016,480,3


In [146]:
df_2016_2600_long.head()

,Вид помощи,группы больных,Строка,Год,Значение
0,Госпитализировано всего,"3_Больных, состоящих на учете всего",1,2016,4276
1,из них бактериовыделителей,"3_Больных, состоящих на учете всего",2,2016,2222
2,в том числе в дневные стационары (из строки 01),"3_Больных, состоящих на учете всего",3,2016,335
3,в том числе в санатории (из строки 01),"3_Больных, состоящих на учете всего",4,2016,88
4,Применены хирургические методы лечения (всего),"3_Больных, состоящих на учете всего",5,2016,480


In [147]:
# Переименовываем столбцы и задаём порядок
df_all_2600= df_all_2600.rename(columns={
    'Вид помощи': 'Показатель',
    'группы больных': 'Возраст'
})[['Показатель', 'Возраст', 'Год', 'Значение', 'Строка', 'Графа']]

In [148]:
df_all_2600.columns

Index(['Показатель', 'Возраст', 'Год', 'Значение', 'Строка', 'Графа'], dtype='object')

In [149]:

df_all_2600['Показатель'] = df_all_2600['Показатель'].replace({
    'Госпитализировано всего': 'Госпитализировано всего',
    'из них бактериовыделителей': 'Госпитализировано всего, из них бактериовыделителей',
    'в том числе в дневные стационары (из стр.01)': 'Госпитализировано всего, из них бактериовыделителей, в том числе дневные стационары',
    'в том числе в дневные стационары (из строки 01)': 'Госпитализировано всего, из них бактериовыделителей, в том числе дневные стационары',
    'в том числе в санатории (из стр.01)': 'Госпитализировано всего, из них бактериовыделителей, в том числе санатории',
    'в том числе в санатории (из строки 01)': 'Госпитализировано всего, из них бактериовыделителей, в том числе санатории',
    'Применены хирургические методы лечения (всего)': 'Применены хирургические методы лечения всего',
    'по поводу туберкулеза органов дыхания': 'Применены хирургические методы лечения по поводу туберкулеза органов дыхания',
    'из них по поводу ФКТ легких': 'Применены хирургические методы лечения по поводу ФКТ легких',
    'костно-суставного туберкулеза': 'Применены хирургические методы лечения костно-суставного туберкулеза',
    'туберкулеза мочеполовых органов': 'Применены хирургические методы лечения туберкулеза мочеполовых органов',
    'из них с туберкулезом женских половых органов': 'Применены хирургические методы лечения мочеполовых органов из низ с туберкулезом женских половых органов',
    'туберкулеза периферических лимфатических узлов': 'Применены хирургические методы лечения туберкулеза периферических лимфатических узлов',
    'Умерло в стационаре от туберкулеза больных, состоявших на учете': 'Умерло в стационаре от туберкулеза больных, состоявших на учете',
    'Иркутская область черн верн': ''  # заменим на пустую строку, затем удалим
})

# Удаляем строки с пустым значением в 'Показатель'
df_all_2600 = df_all_2600[df_all_2600['Показатель'] != '']

# Проверяем результат
print(df_all_2600['Показатель'].unique())


['Госпитализировано всего'
 'Госпитализировано всего, из них бактериовыделителей'
 'Госпитализировано всего, из них бактериовыделителей, в том числе дневные стационары'
 'Госпитализировано всего, из них бактериовыделителей, в том числе санатории'
 'Применены хирургические методы лечения всего'
 'Применены хирургические методы лечения по поводу туберкулеза органов дыхания'
 'Применены хирургические методы лечения по поводу ФКТ легких'
 'Применены хирургические методы лечения костно-суставного туберкулеза'
 'Применены хирургические методы лечения туберкулеза мочеполовых органов'
 'Применены хирургические методы лечения мочеполовых органов из низ с туберкулезом женских половых органов'
 'Применены хирургические методы лечения туберкулеза периферических лимфатических узлов'
 'Умерло в стационаре от туберкулеза больных, состоявших на учете']


In [150]:
df_all_2600['Возраст'] = df_all_2600['Возраст'].replace({
    'Больных, состоящих на учете всего': 'Больных состоящих на учете всего',
    'Больных, состоящих на учете детей до 14 лет': 'Больных состоящих на учете из них детей до 14 лет',
    'Больных, состоящих на учете подростков 15-17 лет': 'Больных состоящих на учете из них подростков 15-17 лет',
    'Впервые установленным диагнозом всего': 'Больных состоящих на учете всего, из них с впервые в жизни установленным диагнозом всего',
    'из них с впервые установленным диагнозом всего': 'Больных состоящих на учете всего, из них с впервые в жизни установленным диагнозом всего',
    'Впервые установленным диагнозом детей до 14 лет': 'Больных состоящих на учете всего, из них с впервые в жизни установленным диагнозом из них детей до 14 лет',
    'из них с впервые установленным диагнозом детей до 14 лет': 'Больных состоящих на учете всего, из них с впервые в жизни установленным диагнозом из них детей до 14 лет',
    'Впервые установленным диагнозом подростков 15-17 лет': 'Больных состоящих на учете всего, из них с впервые в жизни установленным диагнозом из них подростков 15-17 лет',
    'из них с впервые установленным диагнозом подростков 15-17 лет': 'Больных состоящих на учете всего, из них с впервые в жизни установленным диагнозом из них подростков 15-17 лет',
})

print(df_all_2600['Возраст'].unique())

['Больных состоящих на учете всего'
 'Больных состоящих на учете из них детей до 14 лет'
 'Больных состоящих на учете из них подростков 15-17 лет'
 'Больных состоящих на учете всего, из них с впервые в жизни установленным диагнозом всего'
 'Больных состоящих на учете всего, из них с впервые в жизни установленным диагнозом из них детей до 14 лет'
 'Больных состоящих на учете всего, из них с впервые в жизни установленным диагнозом из них подростков 15-17 лет']


In [151]:
print(df_all_2600['Строка'].unique())
print(df_all_2600['Графа'].unique())
print(df_all_2600['Год'].unique())

[ 1  2  3  4  5  6  7  8  9 10 11 12]
[3 4 5 6 7 8]
[2016 2017 2018 2019 2020 2021 2022 2023 2024]


In [152]:
df_all_2600 = df_all_2600.drop_duplicates()

In [153]:
df_all_2600.info()

<class 'pandas.core.frame.DataFrame'>
Index: 642 entries, 0 to 647
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Показатель  642 non-null    object
 1   Возраст     642 non-null    object
 2   Год         642 non-null    int64 
 3   Значение    562 non-null    object
 4   Строка      642 non-null    int64 
 5   Графа       642 non-null    int64 
dtypes: int64(3), object(3)
memory usage: 35.1+ KB


#### Сведение 2310 +

In [154]:
# ПОЛНЫЙ путь 
data_dir = r"C:\Users\urize\Ирткутск_туберкулез"

df_2310 = pd.read_excel(os.path.join(data_dir, "2310.xlsx"))

In [155]:
df_2310['Показатель'].unique()

array(['Умерших от туберкулеза', 'Умерших от других причин',
       'Умерло от туберкулеза', 'Умерло от других причин'], dtype=object)

In [156]:
df_2310['Уточнение'].unique()

array(['Состояли на учете менее 1 года', 'Сочетание ВИЧ/ТБ',
       'Не состояли на учете', 'Больные с сочетанием ВИЧ/ТБ',
       'Больные с сочетанием ВИЧ/ТБ, состоявшие на учете менее 1 года',
       'Не состояли на учете в ПТУ'], dtype=object)

In [157]:
df_2310['Еще уточнение'].unique()

array(['Всего', 'Состояли на учете менее 1 года', 'Взрослых',
       'Детей 0-14 лет', 'Детей 15-17 лет', 'Взрослые',
       'Дети от 0 до 14 лет', 'Подростки от 15 до 17 лет'], dtype=object)

In [158]:
# Копируем, чтобы не менять оригинал
df_clean = df_2310.copy()

# 1. Приводим 'Показатель'
df_clean['Показатель'] = df_clean['Показатель'].replace({
    'Умерло от туберкулеза': 'Умерших от туберкулеза',
    'Умерло от других причин': 'Умерших от других причин'
})

# 2. Приводим 'Уточнение'
df_clean['Уточнение'] = df_clean['Уточнение'].replace({
    'Больные с сочетанием ВИЧ/ТБ': 'Сочетание ВИЧ/ТБ',
    'Больные с сочетанием ВИЧ/ТБ, состоявшие на учете менее 1 года': 'Сочетание ВИЧ/ТБ',
    'Не состояли на учете в ПТУ': 'Не состояли на учете'
})

# 3. Приводим 'Еще уточнение'
df_clean['Еще уточнение'] = df_clean['Еще уточнение'].replace({
    'Взрослые': 'Взрослых',
    'Дети от 0 до 14 лет': 'Детей 0-14 лет',
    'Подростки от 15 до 17 лет': 'Детей 15-17 лет'
})

# 4. Для строк, где Уточнение = 'Сочетание ВИЧ/ТБ' и исходное 'Больные с сочетанием ВИЧ/ТБ, состоявшие на учете менее 1 года',
#    нужно установить Еще уточнение = 'Состояли на учете менее 1 года' (если оно еще не стоит)
#    Определим такие строки по наличию подстроки "состоявшие на учете менее 1 года" в исходном Уточнении
mask = df_2310['Уточнение'].str.contains('состоявшие на учете менее 1 года', na=False)
df_clean.loc[mask, 'Еще уточнение'] = 'Состояли на учете менее 1 года'
df_all_2310 = df_clean
df_all_2310.head()

,Показатель,Уточнение,Еще уточнение,Год,Значение,Строка,Графа,Таблица
0,Умерших от туберкулеза,Состояли на учете менее 1 года,Всего,2016,97.0,1,1,2310
1,Умерших от туберкулеза,Сочетание ВИЧ/ТБ,Всего,2016,15.0,1,2,2310
2,Умерших от туберкулеза,Сочетание ВИЧ/ТБ,Состояли на учете менее 1 года,2016,11.0,1,3,2310
3,Умерших от туберкулеза,Не состояли на учете,Взрослых,2016,3.0,1,4,2310
4,Умерших от туберкулеза,Не состояли на учете,Детей 0-14 лет,2016,NaN,1,5,2310


In [159]:
# Меняем местами колонки Строка и Графа в df_all_2310
df_all_2310 = df_all_2310.rename(columns={'Строка': 'Графа', 'Графа': 'Строка'})

# Проверяем результат
print("Новые колонки:", df_all_2310.columns.tolist())
print(df_all_2310.head())

Новые колонки: ['Показатель', 'Уточнение', 'Еще уточнение', 'Год', 'Значение', 'Графа', 'Строка', 'Таблица']
               Показатель                       Уточнение  \
0  Умерших от туберкулеза  Состояли на учете менее 1 года   
1  Умерших от туберкулеза                Сочетание ВИЧ/ТБ   
2  Умерших от туберкулеза                Сочетание ВИЧ/ТБ   
3  Умерших от туберкулеза            Не состояли на учете   
4  Умерших от туберкулеза            Не состояли на учете   

                    Еще уточнение   Год  Значение  Графа  Строка  Таблица  
0                           Всего  2016      97.0      1       1     2310  
1                           Всего  2016      15.0      1       2     2310  
2  Состояли на учете менее 1 года  2016      11.0      1       3     2310  
3                        Взрослых  2016       3.0      1       4     2310  
4                  Детей 0-14 лет  2016       NaN      1       5     2310  


#### Сведение 2513

In [160]:
# Список всех датафреймов 2513 long за разные годы
df_list_2513 = [
    df_2016_2513_long,
    df_2017_2513_long,
    df_2018_2513_long,
    df_2019_2513_long,
    df_2020_2513_long,
    df_2021_2513_long,
    df_2022_2513_long,
    df_2023_2513_long,
    df_2024_2513_long
]

# Объединяем все таблицы в одну
df_all_2513 = pd.concat(df_list_2513, ignore_index=True)

# Если нужно применить extract_numeric_prefix_from_dataset
df_all_2513 = extract_numeric_prefix_from_dataset(df_all_2513)
df_all_2513['Строка'] = pd.to_numeric(df_all_2513['Строка'], errors='coerce')

# Проверяем результат
print(f"Объединено строк: {len(df_all_2513)}")
print(f"Колонки: {list(df_all_2513.columns)}")
df_all_2513.head()

Выбрана колонка для извлечения префикса: 'Категория'
Из колонки 'Категория' извлечено 324 числовых префиксов
Объединено строк: 324
Колонки: ['Профилактические осмотры на туберкулез', 'Категория', 'Год', 'Значение', 'Строка', 'Графа']


,Профилактические осмотры на туберкулез,Категория,Год,Значение,Строка,Графа
0,"Осмотрено пациентов, всего",Всего,2016,1892276.0,1.0,3
1,из них детей: 1-7 лет включительно,Всего,2016,248618.0,1.1,3
2,8-14 лет включительно,Всего,2016,204640.0,1.2,3
3,15-17 лет включительно,Всего,2016,77945.0,1.3,3
4,Из числа осмотренных (стр.1) обследовано: флюорографически,Всего,2016,1411232.0,2.0,3


In [161]:
df_all_2513['Профилактические осмотры на туберкулез'].unique()

array(['Осмотрено пациентов, всего', 'из них детей: 1-7 лет включительно',
       '8-14 лет включительно', '15-17 лет включительно',
       'Из числа осмотренных (стр.1) обследовано: флюорографически',
       'бактериоскопически',
       'Из числа осмотренных детей (стр. 1.1+1.2+1.3) проведены: иммунодиагностика с применением аллергена бактерий с 2 туберкулиновыми единицами очищенного туберкулина в стандартном разведении',
       'иммунодиагностика с применением аллергена туберкулезного рекомбинантного в стандартном разведении',
       'рентгенологическое (флюорографическое) исследование органов грудной клетки',
       'из них детей: 0-7 лет включительно'], dtype=object)

In [162]:
df_all_2513['Категория'].unique()

array(['Всего', 'из них сельских жителей', 'Выявлен туберкулез Всего',
       'Выявлен туберкулез из них: у сельских жителей', 'Всего, чел',
       'Выявлен туберкулез, включая рецидив туберкулеза Всего',
       'Выявлен туберкулез, включая рецидив туберкулеза из них у сельских жителей',
       'Выявлен туберкулез из них у сельских жителей'], dtype=object)

In [163]:
# Нормализация колонки 'Категория'
category_mapping = {
    'Всего': 'всего',
    'Всего, чел': 'всего',
    'Выявлен туберкулез Всего': 'Выявлен туберкулез (всего)',
    'Выявлен туберкулез из них: у сельских жителей': 'Выявлен туберкулез (сельские жители)',
    'Выявлен туберкулез из них у сельских жителей': 'Выявлен туберкулез (сельские жители)',
    'из них сельских жителей': 'сельские жители',
    'Выявлен туберкулез, включая рецидив туберкулеза Всего': 'Выявлен туберкулез (всего)',
    'Выявлен туберкулез, включая рецидив туберкулеза из них у сельских жителей': 'Выявлен туберкулез (сельские жители)',
}

df_all_2513['Категория'] = df_all_2513['Категория'].replace(category_mapping)

# Проверяем результат
print("Уникальные значения после нормализации:")
print(sorted(df_all_2513['Категория'].unique()))

Уникальные значения после нормализации:
['Выявлен туберкулез (всего)', 'Выявлен туберкулез (сельские жители)', 'всего', 'сельские жители']


In [164]:
# Нормализация (если нужно объединить детей)
children_mapping = {
    'из них детей: 1-7 лет включительно': 'дети 0-7 лет',
    'из них детей: 0-7 лет включительно': 'дети 0-7 лет',
    # Не объединяйте, если это разные возрастные группы!
}

df_all_2513['Профилактические осмотры на туберкулез'] = df_all_2513['Профилактические осмотры на туберкулез'].replace(children_mapping)

In [165]:
df_all_2513['Категория'].unique()

array(['всего', 'сельские жители', 'Выявлен туберкулез (всего)',
       'Выявлен туберкулез (сельские жители)'], dtype=object)

In [166]:
df_all_2513['Профилактические осмотры на туберкулез'].unique()

array(['Осмотрено пациентов, всего', 'дети 0-7 лет',
       '8-14 лет включительно', '15-17 лет включительно',
       'Из числа осмотренных (стр.1) обследовано: флюорографически',
       'бактериоскопически',
       'Из числа осмотренных детей (стр. 1.1+1.2+1.3) проведены: иммунодиагностика с применением аллергена бактерий с 2 туберкулиновыми единицами очищенного туберкулина в стандартном разведении',
       'иммунодиагностика с применением аллергена туберкулезного рекомбинантного в стандартном разведении',
       'рентгенологическое (флюорографическое) исследование органов грудной клетки'],
      dtype=object)

In [167]:
print(df_all_2513['Строка'].unique())
print(df_all_2513['Графа'].unique())
print(df_all_2513['Год'].unique())

[1.  1.1 1.2 1.3 2.  3.  4.  5.  6. ]
[3 4 5 6]
[2016 2017 2018 2019 2020 2021 2022 2023 2024]


In [168]:
df_all_2513.head()

,Профилактические осмотры на туберкулез,Категория,Год,Значение,Строка,Графа
0,"Осмотрено пациентов, всего",всего,2016,1892276.0,1.0,3
1,дети 0-7 лет,всего,2016,248618.0,1.1,3
2,8-14 лет включительно,всего,2016,204640.0,1.2,3
3,15-17 лет включительно,всего,2016,77945.0,1.3,3
4,Из числа осмотренных (стр.1) обследовано: флюорографически,всего,2016,1411232.0,2.0,3


### Nan на Ноль


In [169]:
# Список датасетов
datasets = [
    df_all_1000, df_all_2800, df_all_2700,
    df_all_2100, df_all_2200, df_all_2300,
    df_all_2400, df_all_2500, df_all_2600,  df_all_2310, df_all_2513, df_all_1100, df_all_3100
]

# Заменяем NaN на 0 и преобразуем в int во всех датафреймах
for df in datasets:
    df['Значение'] = df['Значение'].fillna(0)#.astype(int)

C:\Users\urize\AppData\Local\Temp\ipykernel_15856\3430257322.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Значение'] = df['Значение'].fillna(0)#.astype(int)


In [170]:

# Соответствующие имена переменных (в том же порядке)
names = [
    'df_all_1000', 'df_all_2800', 'df_all_2700',
    'df_all_2100', 'df_all_2200', 'df_all_2300',
    'df_all_2400', 'df_all_2500', 'df_all_2600', 'df_all_2310', 'df_all_2513', 'df_all_1100', 'df_all_3100'
]

# Добавляем столбец 'Таблица' в каждый датафрейм
for df, name in zip(datasets, names):
    # Извлекаем число из имени (всё после последнего подчёркивания или между '_' и концом)
    # Проще: отрезаем префикс 'df_all_' и получаем номер
    table_number = name.replace('df_all_', '')
    df['Таблица'] = table_number

# Проверка (выводим первые строки каждого датафрейма)
for df, name in zip(datasets, names):
    print(f"{name}: столбец 'Таблица' = {df['Таблица'].iloc[0]}")

df_all_1000: столбец 'Таблица' = 1000
df_all_2800: столбец 'Таблица' = 2800
df_all_2700: столбец 'Таблица' = 2700
df_all_2100: столбец 'Таблица' = 2100
df_all_2200: столбец 'Таблица' = 2200
df_all_2300: столбец 'Таблица' = 2300
df_all_2400: столбец 'Таблица' = 2400
df_all_2500: столбец 'Таблица' = 2500
df_all_2600: столбец 'Таблица' = 2600
df_all_2310: столбец 'Таблица' = 2310
df_all_2513: столбец 'Таблица' = 2513
df_all_1100: столбец 'Таблица' = 1100
df_all_3100: столбец 'Таблица' = 3100


### Сохранение полученных датасетов

### Фильтруем по 2019-2021


In [171]:
# ========== ЧАСТЬ 1: ТОЛЬКО ФИЛЬТРАЦИЯ ==========

years_range = [2019, 2020, 2021]
suffix = '2019_2021'

original_names = [
    'df_all_1000', 'df_all_2800', 'df_all_2700',
    'df_all_2100', 'df_all_2200', 'df_all_2300',
    'df_all_2400', 'df_all_2500', 'df_all_2600', 'df_all_2310', 'df_all_2513', 'df_all_1100', 'df_all_3100'
]

def filter_by_year(df, years):
    if 'год' in df.columns:
        return df[df['год'].isin(years)].copy()
    elif 'Год' in df.columns:
        return df[df['Год'].isin(years)].copy()
    else:
        print(f"Колонка года не найдена в {df}")
        return df.copy()

# Фильтруем и создаём переменные (например, df_1000_2019_2021)
for name in original_names:
    df = globals()[name]
    filtered = filter_by_year(df, years_range)
    base_name = name.replace('_all_', '_')
    new_var_name = f'{base_name}_{suffix}'
    globals()[new_var_name] = filtered
    print(f"Создана переменная {new_var_name}")

print("\nФильтрация завершена.")



Создана переменная df_1000_2019_2021
Создана переменная df_2800_2019_2021
Создана переменная df_2700_2019_2021
Создана переменная df_2100_2019_2021
Создана переменная df_2200_2019_2021
Создана переменная df_2300_2019_2021
Создана переменная df_2400_2019_2021
Создана переменная df_2500_2019_2021
Создана переменная df_2600_2019_2021
Создана переменная df_2310_2019_2021
Создана переменная df_2513_2019_2021
Создана переменная df_1100_2019_2021
Создана переменная df_3100_2019_2021

Фильтрация завершена.


In [172]:
# ========== ЧАСТЬ 2: СОХРАНЕНИЕ В CSV И ССЫЛКИ ==========

def create_download_link(df, filename):
    csv = df.to_csv(index=False, encoding='utf-8-sig')
    b64 = base64.b64encode(csv.encode()).decode()
    display(HTML(f'<a href="data:file/csv;base64,{b64}" download="{filename}">Скачать {filename}</a>'))

# Сохраняем исходные датасеты (на диск + ссылки)
print("Исходные датасеты (сохранены на диск и доступны для скачивания):")
for name in original_names:
    df = globals()[name]
    out_name = f'{name}.csv'
    df.to_csv(out_name, index=False, encoding='utf-8-sig')  # СОХРАНЯЕМ НА ДИСК
    create_download_link(df, out_name)                      # ССЫЛКА ДЛЯ СКАЧИВАНИЯ

# Только ссылки для отфильтрованных датасетов (НЕ сохраняем на диск)
print("\nОтфильтрованные датасеты (только ссылки для скачивания, файлы НЕ сохранены на диск):")
for name in original_names:
    base_name = name.replace('_all_', '_')
    var_name = f'{base_name}_{suffix}'
    df_filtered = globals()[var_name]
    out_name = f'{var_name}.csv'
    create_download_link(df_filtered, out_name)  # ТОЛЬКО ССЫЛКА, БЕЗ to_csv

print("\nГотово! Исходные датасеты сохранены в текущую папку. Отфильтрованные доступны только по ссылкам.")

Исходные датасеты (сохранены на диск и доступны для скачивания):



Отфильтрованные датасеты (только ссылки для скачивания, файлы НЕ сохранены на диск):



Готово! Исходные датасеты сохранены в текущую папку. Отфильтрованные доступны только по ссылкам.
